# Benchmarking German DSOs: continuity of supply and local resilience features

Reproduces Supplementary Tables S1–S12 of Khaustova, Salashenko, Kotliarov, Shulga & Kostenko, *Resilience management of distribution power systems based on continuity of supply: benchmarking evidence from Germany*.

* Input: `../data/DSO_2024_corrected.csv` (Bundesnetzagentur data for 2024; see `../data/README.md`).
* Output: `../results/Table_S*.csv`. The tables reported in the manuscript are in `../reference_results/`.
* Run all cells in order from the `notebooks/` folder. Random procedures use fixed seeds.

In [1]:
# Supplementary Materials, Table S1
# Distribution of DSO indicators (P10, Q1, median, Q3, P90) by distribution power system architecture

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import pandas as pd

DATA_PATH = Path('../data/DSO_2024_corrected.csv')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S1'

ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
STATISTICS = [('P10', 0.10), ('Q1', 0.25), ('Med.', 0.50), ('Q3', 0.75), ('P90', 0.90)]

INDICATOR_METADATA = [(1, 'lv_network_density_km_per_km2', 'LV network density, km/km²', 'km/km²', 1),
 (2, 'mv_network_density_km_per_km2', 'MV network density, km/km²', 'km/km²', 2),
 (3, 'lv_to_mv_network_length_ratio', 'Ratio of LV to MV network length, km/km', 'km/km', 2),
 (4, 'network_length_per_consumption_point_m', 'Network length per consumption point, m/point', 'm/point', 1),
 (5, 'network_length_per_generation_point_m', 'Network length per generation point, m/point', 'm/point', 0),
 (6, 'generation_points_per_1000_consumption_points', 'Generation points per 1,000 consumption points, units', 'units', 0),
 (7, 'vertical_exchange_intensity_kw_per_kw', 'Vertical exchange intensity, kW/kW', 'kW/kW', 2),
 (8, 'lv_gis_coverage_pct', 'GIS coverage of LV network, %', '%', 1),
 (9, 'mv_gis_coverage_pct', 'GIS coverage of MV network, %', '%', 1),
 (10, 'hv_gis_coverage_pct', 'GIS coverage of HV network, %', '%', 1),
 (11, 'non_remote_maximum_demand_meters_share_pct', 'Share of non-remote maximum-demand meters, %', '%', 1),
 (12, 'local_substations_remote_transformer_measurements_share_pct',
  'Share of local substations with remote measurements at the transformer, %', '%', 1),
 (13, 'local_substations_remote_lv_board_measurements_share_pct',
  'Share of local substations with remote measurements in the LV distribution board, %', '%', 1),
 (14, 'lv_lines_centralized_online_monitoring_share_pct', 'Share of LV lines with centralized online monitoring, %', '%', 1),
 (15, 'mv_lines_centralized_online_monitoring_share_pct', 'Share of MV lines with centralized online monitoring, %', '%', 1),
 (16, 'hv_lines_centralized_online_monitoring_share_pct', 'Share of HV lines with centralized online monitoring, %', '%', 1),
 (17, 'mv_lv_transformers_centralized_online_monitoring_share_pct',
  'Share of MV/LV transformers with centralized online monitoring, %', '%', 1),
 (18, 'hv_mv_transformers_centralized_online_monitoring_share_pct',
  'Share of HV/MV transformers with centralized online monitoring, %', '%', 1),
 (19, 'lv_equipment_centralized_control_share_pct', 'Share of LV network equipment with centralized control, %', '%', 0),
 (20, 'mv_equipment_centralized_control_share_pct', 'Share of MV network equipment with centralized control, %', '%', 1),
 (21, 'hv_equipment_centralized_control_share_pct', 'Share of HV network equipment with centralized control, %', '%', 1),
 (22, 'mv_lv_substations_centralized_control_share_pct', 'Share of MV/LV substations with centralized control, %', '%', 1),
 (23, 'hv_mv_substations_centralized_control_share_pct', 'Share of HV/MV substations with centralized control, %', '%', 1),
 (24, 'lv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak LV load, kW/kW', 'kW/kW', 2),
 (25, 'mv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak MV load, kW/kW', 'kW/kW', 2),
 (26, 'hv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak HV load, kW/kW', 'kW/kW', 2),
 (27, 'lv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at LV level, %', '%', 1),
 (28, 'mv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at MV level, %', '%', 1),
 (29, 'hv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at HV level, %', '%', 1),
 (30, 'lv_solar_generation_share_pct', 'Share of solar generation at LV level, %', '%', 1),
 (31, 'lv_wind_generation_share_pct', 'Share of wind generation at LV level, %', '%', 2),
 (32, 'lv_other_res_generation_share_pct', 'Share of other RES at LV level, %', '%', 2),
 (33, 'mv_solar_generation_share_pct', 'Share of solar generation at MV level, %', '%', 1),
 (34, 'mv_wind_generation_share_pct', 'Share of wind generation at MV level, %', '%', 1),
 (35, 'mv_other_res_generation_share_pct', 'Share of other RES at MV level, %', '%', 1),
 (36, 'hv_solar_generation_share_pct', 'Share of solar generation at HV level, %', '%', 1),
 (37, 'hv_wind_generation_share_pct', 'Share of wind generation at HV level, %', '%', 1),
 (38, 'hv_other_res_generation_share_pct', 'Share of other RES at HV level, %', '%', 1),
 (39, 'lv_solar_connection_application_avg_capacity_kw',
  'Average capacity of solar connection applications at LV level, kW/point', 'kW/point', 1),
 (40, 'lv_wind_connection_application_avg_capacity_kw',
  'Average capacity of wind connection applications at LV level, kW/point', 'kW/point', 1),
 (41, 'lv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at LV level, kW/point', 'kW/point', 0),
 (42, 'mv_solar_connection_application_avg_capacity_kw',
  'Average capacity of solar connection applications at MV level, kW/point', 'kW/point', 0),
 (43, 'mv_wind_connection_application_avg_capacity_kw',
  'Average capacity of wind connection applications at MV level, kW/point', 'kW/point', 0),
 (44, 'mv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at MV level, kW/point', 'kW/point', 0),
 (45, 'hv_solar_connection_application_avg_capacity_kw',
  'Average capacity of solar connection applications at HV level, kW/point', 'kW/point', 0),
 (46, 'hv_wind_connection_application_avg_capacity_kw',
  'Average capacity of wind connection applications at HV level, kW/point', 'kW/point', 0),
 (47, 'lv_generation_connection_duration_days', 'Duration of generation connection to LV network, days', 'days', 0),
 (48, 'mv_generation_connection_duration_days', 'Duration of generation connection to MV network, days', 'days', 0),
 (49, 'hv_generation_connection_duration_days', 'Duration of generation connection to HV network, days', 'days', 0),
 (50, 'lv_peak_load_per_consumer_kw', 'Peak LV load per LV consumer, kW/consumer', 'kW/consumer', 2),
 (51, 'mv_peak_load_per_consumer_kw', 'Peak MV load per MV consumer, kW/consumer', 'kW/consumer', 0),
 (52, 'hv_peak_load_per_consumer_kw', 'Peak HV load per HV consumer, kW/consumer', 'kW/consumer', 0),
 (53, 'lv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak LV load, %', '%', 1),
 (54, 'mv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak MV load, %', '%', 1),
 (55, 'lv_heat_pump_capacity_to_peak_load_pct', 'Share of heat pumps in peak LV load, %', '%', 1),
 (56, 'lv_connection_applications_per_1000_consumers',
  'LV connection applications per 1,000 LV consumers, units', 'units', 1),
 (57, 'lv_new_or_upgraded_points_per_1000_existing_points',
  'New/upgraded LV connection points per 1,000 LV points, units', 'units', 1),
 (58, 'lv_consumer_connection_duration_days', 'Duration of consumer connection to LV network, days', 'days', 1),
 (59, 'lv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at LV level, %', '%', 1),
 (60, 'mv_connection_applications_per_1000_consumers',
  'MV connection applications per 1,000 MV consumers, units', 'units', 0),
 (61, 'mv_new_or_upgraded_points_per_1000_existing_points',
  'New/upgraded MV connection points per 1,000 MV points, units', 'units', 1),
 (62, 'mv_consumer_connection_duration_days', 'Duration of consumer connection to MV network, days', 'days', 0),
 (63, 'mv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at MV level, %', '%', 1),
 (64, 'lv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak LV load, kW/MW', 'kW/MW', 0),
 (65, 'mv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak MV load, kW/MW', 'kW/MW', 1),
 (66, 'lv_bess_connection_applications_per_1000_consumers',
  'BESS connection applications per 1,000 LV consumers, units', 'units', 1),
 (67, 'mv_bess_connection_applications_per_1000_consumers',
  'BESS connection applications per 1,000 MV consumers, units', 'units', 1),
 (68, 'lv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at LV level, days', 'days', 1),
 (69, 'mv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at MV level, days', 'days', 1),
 (70, 'prosumers_full_market_mode_per_1000_lv_consumers',
  'Prosumers in full market mode per 1,000 LV consumers, units', 'units', 2),
 (71, 'prosumer_market_mode_switch_duration_days', 'Time to switch prosumers to full market mode, days', 'days', 1),
 (72, 'prosumer_switching_automation_pct', 'Level of automation of prosumer switching, %', '%', 1),
 (73, 'dynamic_tariff_consumers_per_1000_lv_consumers',
  'Consumers with dynamic tariffs per 1,000 LV consumers, units', 'units', 2),
 (74, 'saidi_lv_min_per_year', 'SAIDI, LV networks, min/year', 'min/year', 2),
 (75, 'asidi_mv_min_per_year', 'ASIDI, MV networks, min/year', 'min/year', 1)]

# HV generation, connection and load indicators are outside the LV–MV architecture scope.
HV_ONLY_VARIABLES = {
    'hv_dg_capacity_to_peak_load_kw_per_kw',
    'hv_dg_capacity_utilization_factor_pct',
    'hv_solar_generation_share_pct',
    'hv_wind_generation_share_pct',
    'hv_other_res_generation_share_pct',
    'hv_solar_connection_application_avg_capacity_kw',
    'hv_wind_connection_application_avg_capacity_kw',
    'hv_generation_connection_duration_days',
    'hv_peak_load_per_consumer_kw',
}
OUTCOMES = ['saidi_lv_min_per_year', 'asidi_mv_min_per_year']


def format_number(value, digits):
    """Round half up for display only; unrounded quantiles are kept in the numeric file."""
    if not np.isfinite(value):
        return '–'
    rounded = Decimal(f'{value:.12f}').quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP)
    return f'{rounded:,.{digits}f}'


# 1. Load the corrected source with missing values preserved.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
source['dps_type'] = source['dps_type'].str.strip().str.replace(r'\s*[-−—–]\s*', '–', regex=True)
source['dso_id'] = source['dso_id'].str.strip()
if source['dso_id'].isna().any() or source['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')

variables = [record[1] for record in INDICATOR_METADATA]
missing_columns = sorted(set(variables) - set(source.columns))
if missing_columns:
    raise ValueError(f'Required source columns are missing: {missing_columns}')
for variable in variables:
    source[variable] = pd.to_numeric(source[variable], errors='raise')
    if not np.isfinite(source[variable].dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{variable}: infinite values are not supported.')

# 2. Complete-outcome analytical sample.
if (source[OUTCOMES] < 0).any().any():
    raise ValueError('Interruption durations must be non-negative.')
valid = source['dps_type'].isin(ARCHITECTURES) & source[OUTCOMES].notna().all(axis=1)
table_s1_excluded = source.loc[~valid, ['dso_id', 'dso_name', 'dps_type', *OUTCOMES]].copy()
data = source.loc[valid].copy()
group_sizes = {a: int(data['dps_type'].eq(a).sum()) for a in ARCHITECTURES}
if any(size == 0 for size in group_sizes.values()):
    raise ValueError('Both network architecture groups must contain observations.')

# 3. Raw distribution quantiles within each architecture.
display_rows, numeric_rows, availability_rows = [], [], []
for number, variable, label, unit, digits in INDICATOR_METADATA:
    display_row = {('Indicator', ''): label}
    numeric_row = {'No.': number, 'Variable': variable, 'Indicator': label, 'Unit': unit}
    for architecture in ARCHITECTURES:
        values = data.loc[data['dps_type'].eq(architecture), variable].dropna()
        applicable = not (architecture == 'LV–MV' and variable in HV_ONLY_VARIABLES)
        if applicable and len(values):
            quantiles = values.quantile([q for _, q in STATISTICS], interpolation='linear').to_numpy()
        else:
            quantiles = np.full(len(STATISTICS), np.nan)
        availability_rows.append({
            'No.': number, 'Variable': variable, 'Indicator': label, 'DPS type': architecture,
            'DSOs in group': group_sizes[architecture], 'Applicable': applicable,
            'Reported n': len(values), 'Used n': len(values) if applicable else 0,
            'Missing n': group_sizes[architecture] - len(values),
            'Reported zero n': int(values.eq(0).sum()),
            'Values outside architecture scope': len(values) if not applicable else 0,
        })
        heading = f'{architecture} (n = {group_sizes[architecture]})'
        for (statistic, _), value in zip(STATISTICS, quantiles):
            numeric_row[f'{architecture} {statistic}'] = float(value)
            display_row[(heading, statistic)] = format_number(value, digits)
    numeric_rows.append(numeric_row)
    display_rows.append(display_row)

table_s1 = pd.DataFrame(display_rows)
table_s1.columns = pd.MultiIndex.from_tuples(table_s1.columns)
table_s1_numeric = pd.DataFrame(numeric_rows)
table_s1_availability = pd.DataFrame(availability_rows)

# 4. Save Table S1 
top_header, sub_header, previous = [], [], None
for top, sub in table_s1.columns:
    top_header.append(top if top != previous else '')
    sub_header.append(sub)
    previous = top
with open(OUTPUT_DIRECTORY / f'{TABLE_ID}.csv', 'w', encoding='utf-8-sig', newline='') as file:
    file.write(';'.join(top_header) + '\n')
    file.write(';'.join(sub_header) + '\n')
    table_s1.to_csv(file, sep=';', index=False, header=False)
table_s1_numeric.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_numeric.csv', sep=';', decimal=',',
                        encoding='utf-8-sig', index=False)
table_s1_availability.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_availability.csv', sep=';',
                             encoding='utf-8-sig', index=False)
table_s1_excluded.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_excluded_DSOs.csv', sep=';', decimal=',',
                         encoding='utf-8-sig', index=False)

print('Analytical sample:', ', '.join(f'{a}: {n}' for a, n in group_sizes.items()))
print(f'Excluded DSOs (missing outcomes or other architectures): {len(table_s1_excluded)}')
print(f'Table S1 rows: {len(table_s1)} (73 indicators and 2 outcomes).')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')

try:
    from IPython.display import display
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.max_colwidth', None):
        display(table_s1)
except ImportError:
    print(table_s1.to_string(index=False))

Analytical sample: LV–MV: 669, LV–MV–HV: 97
Excluded DSOs (missing outcomes or other architectures): 11
Table S1 rows: 75 (73 indicators and 2 outcomes).
Saved to: ../results/Table_S1.csv


Indicator  \
                                                                                                   
0                                                                     LV network density, km/km²   
1                                                                     MV network density, km/km²   
2                                                        Ratio of LV to MV network length, km/km   
3                                                  Network length per consumption point, m/point   
4                                                   Network length per generation point, m/point   
5                                          Generation points per 1,000 consumption points, units   
6                                                             Vertical exchange intensity, kW/kW   
7                                                                  GIS coverage of LV network, %   
8                                                                  GIS coverage of MV network, %   
9                                                                  GIS coverage of HV network, %   
10                                                  Share of non-remote maximum-demand meters, %   
11                     Share of local substations with remote measurements at the transformer, %   
12           Share of local substations with remote measurements in the LV distribution board, %   
13                                       Share of LV lines with centralized online monitoring, %   
14                                       Share of MV lines with centralized online monitoring, %   
15                                       Share of HV lines with centralized online monitoring, %   
16                             Share of MV/LV transformers with centralized online monitoring, %   
17                             Share of HV/MV transformers with centralized online monitoring, %   
18                                     Share of LV network equipment with centralized control, %   
19                                     Share of MV network equipment with centralized control, %   
20                                     Share of HV network equipment with centralized control, %   
21                                        Share of MV/LV substations with centralized control, %   
22                                        Share of HV/MV substations with centralized control, %   
23                                                   DG capacity relative to peak LV load, kW/kW   
24                                                   DG capacity relative to peak MV load, kW/kW   
25                                                   DG capacity relative to peak HV load, kW/kW   
26                                           DG capacity utilization factor (CUF) at LV level, %   
27                                           DG capacity utilization factor (CUF) at MV level, %   
28                                           DG capacity utilization factor (CUF) at HV level, %   
29                                                      Share of solar generation at LV level, %   
30                                                       Share of wind generation at LV level, %   
31                                                             Share of other RES at LV level, %   
32                                                      Share of solar generation at MV level, %   
33                                                       Share of wind generation at MV level, %   
34                                                             Share of other RES at MV level, %   
35                                                      Share of solar generation at HV level, %   
36                                                       Share of wind generation at HV level, %   
37                                                             Share of other RES at HV level, %   
38                       Average capacity of solar connection applications at LV level,

In [2]:
# Supplementary Materials, Table S2
# Composite index of continuity of power supply (ICPS) by quartile and network architecture

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import pandas as pd

DATA_PATH = Path('../data/DSO_2024_corrected.csv')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S2'

ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
QUARTILE_ORDER = ['Q4', 'Q3', 'Q2', 'Q1', 'Total']

# Missing values replaced with zeros in the supplied CSV (restored from the original Excel dataset).
MISSING_IDS = {
    'SAIDI': ['10003357', '10001912', '10000211', '10006795', '10001486'],
    'ASIDI': ['10003357', '10001674', '10001057', '10000211', '10001066',
              '10000464', '10001486', '10000731', '10013580'],
}


def fmt(value, digits=2):
    """Round half up for display only."""
    if not np.isfinite(value):
        return '–'
    return str(Decimal(f'{value:.12f}').quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP))


# 1. Load the dataset.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
data = source[['dso_id', 'dso_name', 'dps_type',
               'saidi_lv_min_per_year', 'asidi_mv_min_per_year']].copy().rename(columns={
    'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'})
data['dso_id'] = data['dso_id'].str.strip()
data['dps_type'] = data['dps_type'].str.strip().str.replace(r'\s*[-−—–]\s*', '–', regex=True)
if data['dso_id'].isna().any() or data['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')

# 2. Restore missing values.
for column, ids in MISSING_IDS.items():
    data[column] = pd.to_numeric(data[column], errors='raise')
    mask = data['dso_id'].isin(ids)
    if data.loc[mask, column].dropna().ne(0).any():
        raise ValueError(f'{column}: verify new values for previously missing DSO IDs')
    data.loc[mask, column] = np.nan

valid = data['dps_type'].isin(ARCHITECTURES) & data[['SAIDI', 'ASIDI']].notna().all(axis=1)
table_s2_excluded = data.loc[~valid].copy()
data = data.loc[valid].copy()

# 3. ICPS separately for each network architecture.
rows, parameters, operator_groups = [], [], []
for dps_type in ARCHITECTURES:
    group = data.loc[data['dps_type'].eq(dps_type)].copy()
    n = len(group)
    if n < 4:
        raise ValueError(f'{dps_type}: insufficient observations')

    x = group[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
    if not np.isfinite(x).all() or (x < 0).any():
        raise ValueError(f'{dps_type}: invalid SAIDI or ASIDI values')

    # Winsorization at P10/P90 and inverse min–max normalization
    p10, p90 = np.quantile(x, [0.10, 0.90], axis=0, method='linear')
    if np.any(p90 <= p10):
        raise ValueError(f'{dps_type}: P90 must exceed P10')
    z = (p90 - np.clip(x, p10, p90)) / (p90 - p10)

    # Entropy weights with the convention 0 · ln(0) = 0.
    shares = z / z.sum(axis=0)
    terms = np.zeros_like(shares)
    positive = shares > 0
    terms[positive] = shares[positive] * np.log(shares[positive])
    entropy = -terms.sum(axis=0) / np.log(n)
    differentiation = np.maximum(1 - entropy, 0)
    if differentiation.sum() <= np.finfo(float).eps:
        raise ValueError(f'{dps_type}: entropy weights cannot be determined')
    weights = differentiation / differentiation.sum()

    group['ICPS'] = z @ weights

    # Quartile boundaries: Q1 ≤ P25 < Q2 ≤ P50 < Q3 < P75 ≤ Q4.
    p25, p50, p75 = group['ICPS'].quantile([0.25, 0.50, 0.75])
    if not p25 < p50 < p75:
        raise ValueError(f'{dps_type}: quartile thresholds coincide')
    group['ICPS_quartile'] = np.select(
        [group['ICPS'] <= p25, group['ICPS'] <= p50, group['ICPS'] < p75],
        ['Q1', 'Q2', 'Q3'], default='Q4')
    operator_groups.append(group)

    parameters.append({
        'DPS type': dps_type, 'N': n,
        'SAIDI P10': p10[0], 'SAIDI P90': p90[0],
        'ASIDI P10': p10[1], 'ASIDI P90': p90[1],
        'SAIDI entropy': entropy[0], 'ASIDI entropy': entropy[1],
        'SAIDI weight': weights[0], 'ASIDI weight': weights[1],
        'ICPS P25': p25, 'ICPS P50': p50, 'ICPS P75': p75,
    })

    # Summary statistics for each quartile and the entire group.
    for quartile in QUARTILE_ORDER:
        subset = group if quartile == 'Total' else group.loc[group['ICPS_quartile'].eq(quartile)]
        row = {'DPS type': dps_type, 'ICPS quartile': quartile, 'N': len(subset)}
        for column in ['ICPS', 'SAIDI', 'ASIDI']:
            values = subset[column]
            if column == 'ICPS':
                lower, upper = values.min(), values.max()
                label, unit = 'min–max', ''
            else:
                lower, upper = values.quantile([0.25, 0.75])
                label, unit = 'P25–P75', ', min/year'
            row[f'{column} median ({label}){unit}'] = (
                f'{fmt(values.median())} ({fmt(lower)}–{fmt(upper)})')
            if column != 'ICPS':
                row[f'{column} mean, min/year'] = fmt(values.mean())
        row['DSO with ASIDI = 0'] = int(subset['ASIDI'].eq(0).sum())
        rows.append(row)

# 4. Combine results.
table_s2 = pd.DataFrame(rows)
table_s2_parameters = pd.DataFrame(parameters)
table_s2_operators = pd.concat(operator_groups, ignore_index=True)

# 5. Save Table S2, ICPS parameters, operator-level scores and exclusions.
table_s2.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}.csv', sep=';', encoding='utf-8-sig', index=False)
table_s2_parameters.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_parameters.csv', sep=';', decimal=',',
                           encoding='utf-8-sig', index=False)
table_s2_operators.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_ICPS_by_DSO.csv', sep=';', decimal=',',
                          encoding='utf-8-sig', index=False)
table_s2_excluded.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_excluded_DSOs.csv', sep=';', decimal=',',
                         encoding='utf-8-sig', index=False)

# 6. Display the results.
print(f'Included DSOs: {len(table_s2_operators)}; excluded DSOs: {len(table_s2_excluded)}')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
try:
    from IPython.display import display
    display(table_s2_parameters[['DPS type', 'N', 'SAIDI weight', 'ASIDI weight']])
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
        display(table_s2)
except ImportError:
    print(table_s2_parameters[['DPS type', 'N', 'SAIDI weight', 'ASIDI weight']].to_string(index=False))
    print(table_s2.to_string(index=False))

Included DSOs: 766; excluded DSOs: 11
Saved to: ../results/Table_S2.csv


,DPS type,N,SAIDI weight,ASIDI weight
0,LV–MV,669,0.535160,0.464840
1,LV–MV–HV,97,0.504126,0.495874


,DPS type,ICPS quartile,N,ICPS median (min–max),"SAIDI median (P25–P75), min/year","SAIDI mean, min/year","ASIDI median (P25–P75), min/year","ASIDI mean, min/year",DSO with ASIDI = 0
0,LV–MV,Q4,168,0.98 (0.90–1.00),0.24 (0.03–0.65),0.37,0.00 (0.00–0.00),0.22,133
1,LV–MV,Q3,166,0.83 (0.75–0.90),1.47 (1.07–1.99),1.48,0.57 (0.00–2.57),1.67,63
2,LV–MV,Q2,167,0.65 (0.51–0.75),2.67 (1.72–3.58),2.58,3.49 (0.29–7.77),5.32,36
3,LV–MV,Q1,168,0.40 (0.00–0.51),4.91 (2.48–7.01),5.67,11.14 (2.08–22.41),26.88,27
4,LV–MV,Total,669,0.75 (0.00–1.00),1.69 (0.60–3.33),2.53,1.06 (0.00–6.18),8.55,259
5,LV–MV–HV,Q4,25,0.85 (0.76–1.00),1.99 (0.37–2.38),1.66,1.72 (1.05–3.30),2.08,3
6,LV–MV–HV,Q3,23,0.65 (0.60–0.76),3.42 (2.24–4.29),3.28,4.73 (2.74–6.16),4.82,0
7,LV–MV–HV,Q2,24,0.50 (0.40–0.57),5.15 (2.77–6.50),17.03,5.93 (3.23–9.10),6.32,2
8,LV–MV–HV,Q1,25,0.28 (0.00–0.40),6.38 (4.39–7.50),6.04,11.40 (6.75–14.02),13.42,0
9,LV–MV–HV,Total,97,0.57 (0.00–1.00),3.54 (2.14–5.57),6.98,4.73 (2.78–9.18),6.70,5


In [3]:
# Supplementary Materials, Table S3
# Spearman correlations between DSO profile indicators and continuity outcomes (ICPS, SAIDI, ASIDI)
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm

DATA_PATH = Path('../data/DSO_2024_corrected.csv')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S3'
ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
OUTCOMES = ['ICPS', 'SAIDI', 'ASIDI']
MIN_N = 10
FDR_THRESHOLD = 0.10
MIN_ABS_RHO = 0.10
EXPECTED_GROUP_SIZES = {'LV–MV': 669, 'LV–MV–HV': 97}
EXPECTED_TEST_COUNTS = {'LV–MV': 192, 'LV–MV–HV': 219}

# Source variables and English labels for the 73 manuscript profile indicators.
# Eligibility is determined from the data, before any significance filtering.
INDICATOR_METADATA = [('lv_network_density_km_per_km2', 'LV network density'),
 ('mv_network_density_km_per_km2', 'MV network density'),
 ('lv_to_mv_network_length_ratio', 'Ratio of LV to MV network length'),
 ('network_length_per_consumption_point_m', 'Network length per consumption point'),
 ('network_length_per_generation_point_m', 'Network length per generation point'),
 ('generation_points_per_1000_consumption_points', 'Generation points per 1,000 consumption points'),
 ('vertical_exchange_intensity_kw_per_kw', 'Vertical exchange intensity'),
 ('lv_gis_coverage_pct', 'GIS coverage of LV network'),
 ('mv_gis_coverage_pct', 'GIS coverage of MV network'),
 ('hv_gis_coverage_pct', 'GIS coverage of HV network'),
 ('non_remote_maximum_demand_meters_share_pct', 'Share of non-remote maximum-demand meters'),
 ('local_substations_remote_transformer_measurements_share_pct',
  'Share of local substations with remote measurements at the transformer'),
 ('local_substations_remote_lv_board_measurements_share_pct',
  'Share of local substations with remote measurements in the LV distribution board'),
 ('lv_lines_centralized_online_monitoring_share_pct', 'Share of LV lines with centralized online monitoring'),
 ('mv_lines_centralized_online_monitoring_share_pct', 'Share of MV lines with centralized online monitoring'),
 ('hv_lines_centralized_online_monitoring_share_pct', 'Share of HV lines with centralized online monitoring'),
 ('mv_lv_transformers_centralized_online_monitoring_share_pct',
  'Share of MV/LV transformers with centralized online monitoring'),
 ('hv_mv_transformers_centralized_online_monitoring_share_pct',
  'Share of HV/MV transformers with centralized online monitoring'),
 ('lv_equipment_centralized_control_share_pct', 'Share of LV network equipment with centralized control'),
 ('mv_equipment_centralized_control_share_pct', 'Share of MV network equipment with centralized control'),
 ('hv_equipment_centralized_control_share_pct', 'Share of HV network equipment with centralized control'),
 ('mv_lv_substations_centralized_control_share_pct', 'Share of MV/LV substations with centralized control'),
 ('hv_mv_substations_centralized_control_share_pct', 'Share of HV/MV substations with centralized control'),
 ('lv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak LV load'),
 ('mv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak MV load'),
 ('hv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak HV load'),
 ('lv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at LV level'),
 ('mv_dg_capacity_utilization_factor_pct', 'DG CUF at MV level'),
 ('hv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at HV level'),
 ('lv_solar_generation_share_pct', 'Share of solar generation at LV level'),
 ('lv_wind_generation_share_pct', 'Share of wind generation at LV level'),
 ('lv_other_res_generation_share_pct', 'Share of other RES at LV level'),
 ('mv_solar_generation_share_pct', 'Share of solar generation at MV level'),
 ('mv_wind_generation_share_pct', 'Share of wind generation at MV level'),
 ('mv_other_res_generation_share_pct', 'Share of other RES at MV level'),
 ('hv_solar_generation_share_pct', 'Share of solar generation at HV level'),
 ('hv_wind_generation_share_pct', 'Share of wind generation at HV level'),
 ('hv_other_res_generation_share_pct', 'Share of other RES at HV level'),
 ('lv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at LV level'),
 ('lv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at LV level'),
 ('lv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at LV level'),
 ('mv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at MV level'),
 ('mv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at MV level'),
 ('mv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at MV level'),
 ('hv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at HV level'),
 ('hv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at HV level'),
 ('lv_generation_connection_duration_days', 'Duration of generation connection to LV network'),
 ('mv_generation_connection_duration_days', 'Duration of generation connection to MV network'),
 ('hv_generation_connection_duration_days', 'Duration of generation connection to HV network'),
 ('lv_peak_load_per_consumer_kw', 'Peak LV load per LV consumer'),
 ('mv_peak_load_per_consumer_kw', 'Peak MV load per MV consumer'),
 ('hv_peak_load_per_consumer_kw', 'Peak HV load per HV consumer'),
 ('lv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak LV load'),
 ('mv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak MV load'),
 ('lv_heat_pump_capacity_to_peak_load_pct', 'Share of heat pumps in peak LV load'),
 ('lv_connection_applications_per_1000_consumers', 'LV connection applications per 1,000 LV consumers'),
 ('lv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded LV connection points per 1,000 LV points'),
 ('lv_consumer_connection_duration_days', 'Duration of consumer connection to LV network'),
 ('lv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at LV level'),
 ('mv_connection_applications_per_1000_consumers', 'MV connection applications per 1,000 MV consumers'),
 ('mv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded MV connection points per 1,000 MV points'),
 ('mv_consumer_connection_duration_days', 'Duration of consumer connection to MV network'),
 ('mv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at MV level'),
 ('lv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak LV load'),
 ('mv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak MV load'),
 ('lv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 LV consumers'),
 ('mv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 MV consumers'),
 ('lv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at LV level'),
 ('mv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at MV level'),
 ('prosumers_full_market_mode_per_1000_lv_consumers', 'Prosumers in full market mode per 1,000 LV consumers'),
 ('prosumer_market_mode_switch_duration_days', 'Time to switch prosumers to full market mode'),
 ('prosumer_switching_automation_pct', 'Level of automation of prosumer switching'),
 ('dynamic_tariff_consumers_per_1000_lv_consumers', 'Consumers with dynamic tariffs per 1,000 LV consumers')]


def bh_adjust(p_values):
    """Benjamini–Hochberg adjustment over one complete architecture test family."""
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def spearman_ci(rho, n):
    """Pointwise approximate 95% Fisher-z CI with Bonett–Wright variance.

    The CI is not adjusted for multiple testing; the q-value is BH-adjusted.
    """
    if n <= 3 or not np.isfinite(rho) or abs(rho) > 1:
        raise ValueError('The Spearman CI requires n > 3 and a valid rho.')
    if np.isclose(abs(rho), 1.0, rtol=0, atol=1e-14):
        return float(rho), float(rho)
    z = np.arctanh(rho)
    standard_error = np.sqrt((1 + rho**2 / 2) / (n - 3))
    bounds = np.tanh(z + np.array([-1, 1]) * norm.ppf(0.975) * standard_error)
    return float(bounds[0]), float(bounds[1])


def calculate_icps(group):
    """Calculate architecture-specific ICPS using the same rules as Table S2."""
    group = group.copy()
    x = group[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
    if len(group) < 4 or not np.isfinite(x).all() or (x < 0).any():
        raise ValueError('ICPS requires at least four valid complete outcome observations.')
    p10, p90 = np.quantile(x, [0.10, 0.90], axis=0, method='linear')
    if np.any(p90 <= p10):
        raise ValueError('P90 must exceed P10 for both continuity components.')
    normalized = (p90 - np.clip(x, p10, p90)) / (p90 - p10)
    shares = normalized / normalized.sum(axis=0)
    terms = np.zeros_like(shares)
    positive = shares > 0
    terms[positive] = shares[positive] * np.log(shares[positive])
    entropy = -terms.sum(axis=0) / np.log(len(group))
    differentiation = np.maximum(1 - entropy, 0)
    if differentiation.sum() <= np.finfo(float).eps:
        raise ValueError('Entropy weights cannot be determined.')
    weights = differentiation / differentiation.sum()
    group['ICPS'] = normalized @ weights
    parameters = {
        'SAIDI P10': p10[0], 'SAIDI P90': p90[0],
        'ASIDI P10': p10[1], 'ASIDI P90': p90[1],
        'SAIDI weight': weights[0], 'ASIDI weight': weights[1]
    }
    return group, parameters


# 1. Load the corrected source and validate the required columns.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
source_outcomes = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}
indicator_columns = [variable for variable, _ in INDICATOR_METADATA]
required_columns = {'dso_id', 'dso_name', 'dps_type', *source_outcomes, *indicator_columns}
missing_columns = sorted(required_columns - set(source.columns))
if missing_columns:
    raise ValueError(f'Required source columns are missing: {missing_columns}')
source['dso_id'] = source['dso_id'].str.strip()
source['dps_type'] = (source['dps_type'].str.strip()
    .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
if source['dso_id'].isna().any() or source['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')
for column in [*indicator_columns, *source_outcomes]:
    source[column] = pd.to_numeric(source[column], errors='raise')
    if not np.isfinite(source[column].dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{column}: infinite values are not supported.')
source = source.rename(columns=source_outcomes)
valid = source['dps_type'].isin(ARCHITECTURES) & source[['SAIDI', 'ASIDI']].notna().all(axis=1)
table_s3_excluded = source.loc[~valid, ['dso_id', 'dso_name', 'dps_type', 'SAIDI', 'ASIDI']].copy()
data = source.loc[valid].copy()

# 2. Recalculate ICPS and run all eligible tests within each architecture.
# Zeros are retained. Missing predictor values are excluded pairwise, not globally.
correlation_groups, eligibility_rows, summary_rows, parameter_rows = [], [], [], []
operator_groups = []
for architecture in ARCHITECTURES:
    group = data.loc[data['dps_type'].eq(architecture)].copy()
    if len(group) != EXPECTED_GROUP_SIZES[architecture]:
        raise ValueError(
            f'{architecture}: expected {EXPECTED_GROUP_SIZES[architecture]} DSOs, '
            f'found {len(group)}. Verify the corrected CSV and missing outcome values.')
    group, parameters = calculate_icps(group)
    operator_groups.append(group)
    parameter_rows.append({'DPS type': architecture, 'N': len(group), **parameters})
    tests = []
    for variable, label in INDICATOR_METADATA:
        observed = group[variable].dropna()
        eligible = len(observed) >= MIN_N and observed.nunique() >= 2
        eligibility_rows.append({
            'DPS type': architecture, 'Variable': variable, 'Indicator': label,
            'Observed n': len(observed), 'Missing n': len(group) - len(observed),
            'Zero n': int(observed.eq(0).sum()), 'Unique values': observed.nunique(),
            'Eligible': eligible,
            'Reason': 'Eligible' if eligible else (
                'Fewer than 10 observations' if len(observed) < MIN_N else 'Zero variance')
        })
        if not eligible:
            continue
        for outcome in OUTCOMES:
            pair = group[[variable, outcome]].dropna()
            if len(pair) < MIN_N or pair[outcome].nunique() < 2:
                raise ValueError(f'{architecture}: invalid pair {variable} / {outcome}.')
            rho, p_value = spearmanr(pair[variable], pair[outcome])
            lower, upper = spearman_ci(float(rho), len(pair))
            tests.append({
                'DPS type': architecture, 'Variable': variable, 'Indicator': label,
                'Outcome': outcome, 'n': len(pair), 'rho': float(rho),
                '95% CI lower': lower, '95% CI upper': upper, 'p': float(p_value)
            })
    results = pd.DataFrame(tests)
    if len(results) != EXPECTED_TEST_COUNTS[architecture]:
        raise ValueError(
            f'{architecture}: expected {EXPECTED_TEST_COUNTS[architecture]} tests, '
            f'found {len(results)}. Verify missing values and indicator eligibility.')
    # Apply BH once to the complete family for THIS architecture, before selection.
    results['q'] = bh_adjust(results['p'].to_numpy())
    results['Selection'] = ((results['q'] < FDR_THRESHOLD)
                            & (results['rho'].abs() >= MIN_ABS_RHO))
    correlation_groups.append(results)
    summary_rows.append({
        'DPS type': architecture, 'N': len(group),
        'Eligible indicators': results['Variable'].nunique(), 'Tests': len(results),
        'Selected relationships': int(results['Selection'].sum()),
        'Selected indicators': results.loc[results['Selection'], 'Variable'].nunique()
    })

# 3. Keep all tests at full precision and create the manuscript display table.
table_s3_numeric = pd.concat(correlation_groups, ignore_index=True)
table_s3_summary = pd.DataFrame(summary_rows)
table_s3_eligibility = pd.DataFrame(eligibility_rows)
icps_parameters = pd.DataFrame(parameter_rows)
table_s3_operators = pd.concat(operator_groups, ignore_index=True)
table_s3 = table_s3_numeric.drop(columns='Variable').copy()
for column in ['rho', '95% CI lower', '95% CI upper']:
    table_s3[column] = table_s3_numeric[column].map(lambda value: f'{value:.3f}')
for column in ['p', 'q']:
    table_s3[column] = table_s3_numeric[column].map(
        lambda value: '<0.001' if value < 0.001 else f'{value:.3f}')
table_s3['Selection'] = table_s3_numeric['Selection'].map({True: 'TRUE', False: 'FALSE'})

# 4. Export Table S3, unrounded results, summary, eligibility audit, ICPS parameters and exclusions.
table_s3.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}.csv', sep=';', encoding='utf-8-sig', index=False)
table_s3_numeric.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_numeric.csv', sep=';', decimal=',',
                        encoding='utf-8-sig', index=False)
table_s3_summary.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_summary.csv', sep=';',
                        encoding='utf-8-sig', index=False)
table_s3_eligibility.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_eligibility.csv', sep=';',
                            encoding='utf-8-sig', index=False)
icps_parameters.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_ICPS_parameters.csv', sep=';', decimal=',',
                       encoding='utf-8-sig', index=False)
table_s3_excluded.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_excluded_DSOs.csv', sep=';', decimal=',',
                         encoding='utf-8-sig', index=False)

print(f'Total Table S3 tests: {len(table_s3)}')
print(f'Excluded DSOs: {len(table_s3_excluded)}')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
print('BH correction is applied separately to each complete architecture test family.')
print('Selection uses unrounded q < 0.10 and absolute rho >= 0.10.')
print('Actual correlation signs are retained; confidence intervals are pointwise 95% intervals.')
try:
    from IPython.display import display
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.max_colwidth', None):
        display(table_s3_summary)
        display(table_s3)
except ImportError:
    print(table_s3_summary.to_string(index=False))
    print(table_s3.to_string(index=False))

Total Table S3 tests: 411
Excluded DSOs: 11
Saved to: ../results/Table_S3.csv
BH correction is applied separately to each complete architecture test family.
Selection uses unrounded q < 0.10 and absolute rho >= 0.10.
Actual correlation signs are retained; confidence intervals are pointwise 95% intervals.


,DPS type,N,Eligible indicators,Tests,Selected relationships,Selected indicators
0,LV–MV,669,64,192,59,30
1,LV–MV–HV,97,73,219,22,16


,DPS type,Indicator,Outcome,n,rho,95% CI lower,95% CI upper,p,q,Selection
0,LV–MV,LV network density,ICPS,669,-0.069,-0.145,0.007,0.073,0.175,FALSE
1,LV–MV,LV network density,SAIDI,669,0.055,-0.021,0.131,0.154,0.281,FALSE
2,LV–MV,LV network density,ASIDI,669,0.046,-0.030,0.122,0.232,0.377,FALSE
3,LV–MV,MV network density,ICPS,663,-0.009,-0.085,0.067,0.823,0.883,FALSE
4,LV–MV,MV network density,SAIDI,663,0.073,-0.003,0.148,0.061,0.155,FALSE
5,LV–MV,MV network density,ASIDI,663,-0.095,-0.170,-0.019,0.015,0.049,FALSE
6,LV–MV,Ratio of LV to MV network length,ICPS,669,0.108,0.032,0.183,0.005,0.023,TRUE
7,LV–MV,Ratio of LV to MV network length,SAIDI,669,-0.054,-0.129,0.022,0.164,0.292,FALSE
8,LV–MV,Ratio of LV to MV network length,ASIDI,669,-0.191,-0.264,-0.116,<0.001,<0.001,TRUE
9,LV–MV,Network length per consumption point,ICPS,669,-0.007,-0.082,0.069,0.863,0.906,FALSE


In [4]:
# Supplementary Materials, Table S4
# Selected indicators associated with continuity of power supply and their benchmarks by ICPS quartile

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm, spearmanr

DEFAULT_DATA_PATH = Path('../data/DSO_2024_corrected.csv')
# In Jupyter, sys.argv contains kernel options (e.g. '-f ...json'), so a
# command-line argument is used only if it is an existing .csv file.
_cli_paths = [Path(a) for a in sys.argv[1:] if a.lower().endswith('.csv')]
DATA_PATH = _cli_paths[0] if _cli_paths else DEFAULT_DATA_PATH
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Data file not found: {DATA_PATH}')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
MIN_N = 10            # minimum number of non-missing observations per indicator
MIN_NONZERO_N = 5     # minimum number of non-zero values to report a median
FDR_THRESHOLD = 0.10
MIN_ABS_RHO = 0.10
MINUS = '\u2212'
INCLUDE_NONZERO_SHARE = True   # False -> table without the non-zero share column
DPS_TYPE = 'LV–MV'             # network architecture reported in Table S4
TABLE_ID = 'Table_S4'
TABLE_PATH = None              # None -> Table_S4.csv next to the data file

# Display metadata only: all eligible indicators are tested independently of this map.
# (label, dimension, decimals for the medians)
DISPLAY_METADATA = {
    # Physical network architecture
    'mv_network_density_km_per_km2': ('MV network density, km/km²', 'Physical network architecture', 2),
    'lv_to_mv_network_length_ratio': ('Ratio of LV to MV network length, km/km', 'Physical network architecture', 2),
    'network_length_per_generation_point_m': ('Network length per generation point, m/point', 'Physical network architecture', 0),
    'generation_points_per_1000_consumption_points': ('Generation points per 1,000 consumption points', 'Physical network architecture', 0),
    'vertical_exchange_intensity_kw_per_kw': ('Vertical exchange intensity, kW/kW', 'Physical network architecture', 2),
    # Operational controllability
    'lv_gis_coverage_pct': ('GIS coverage of LV network, %', 'Operational controllability', 0),
    'mv_gis_coverage_pct': ('GIS coverage of MV network, %', 'Operational controllability', 0),
    'non_remote_maximum_demand_meters_share_pct': ('Share of non-remote maximum-demand meters, %', 'Operational controllability', 0),
    'hv_lines_centralized_online_monitoring_share_pct': ('Share of HV lines with centralized online monitoring, %', 'Operational controllability', 0),
    'hv_mv_transformers_centralized_online_monitoring_share_pct': ('Share of HV/MV transformers with centralized online monitoring, %', 'Operational controllability', 0),
    'mv_equipment_centralized_control_share_pct': ('Share of MV network equipment with centralized control, %', 'Operational controllability', 0),
    'mv_lv_substations_centralized_control_share_pct': ('Share of MV/LV substations with centralized control, %', 'Operational controllability', 0),
    'hv_mv_substations_centralized_control_share_pct': ('Share of HV/MV substations with centralized control, %', 'Operational controllability', 0),
    # Generation capacity
    'lv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak LV load, kW/kW', 'Generation capacity', 2),
    'mv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak MV load, kW/kW', 'Generation capacity', 2),
    'hv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak HV load, kW/kW', 'Generation capacity', 2),
    'lv_dg_capacity_utilization_factor_pct': ('DG capacity utilization factor (CUF) at LV level, %', 'Generation capacity', 2),
    'mv_dg_capacity_utilization_factor_pct': ('DG capacity utilization factor (CUF) at MV level, %', 'Generation capacity', 1),
    'lv_solar_generation_share_pct': ('Share of solar generation at LV level, %', 'Generation capacity', 0),
    'lv_wind_generation_share_pct': ('Share of wind generation at LV level, %', 'Generation capacity', 3),
    'mv_solar_generation_share_pct': ('Share of solar generation at MV level, %', 'Generation capacity', 1),
    'mv_wind_generation_share_pct': ('Share of wind generation at MV level, %', 'Generation capacity', 1),
    'lv_other_res_connection_application_avg_capacity_kw': ('Average capacity of other RES connection applications at LV level, kW/point', 'Generation capacity', 1),
    'mv_solar_connection_application_avg_capacity_kw': ('Average capacity of solar connection applications at MV level, kW/point', 'Generation capacity', 0),
    'hv_solar_connection_application_avg_capacity_kw': ('Average capacity of solar connection applications at HV level, kW/point', 'Generation capacity', 0),
    # Consumer load
    'lv_peak_load_per_consumer_kw': ('Peak LV load per LV consumer, kW/consumer', 'Consumer load', 2),
    'mv_peak_load_per_consumer_kw': ('Peak MV load per MV consumer, kW/consumer', 'Consumer load', 0),
    'mv_ev_charging_capacity_to_peak_load_pct': ('Share of EV charging stations in peak MV load, %', 'Consumer load', 2),
    'lv_connection_applications_per_1000_consumers': ('LV connection applications per 1,000 LV consumers', 'Consumer load', 1),
    'mv_connection_applications_per_1000_consumers': ('MV connection applications per 1,000 MV consumers', 'Consumer load', 1),
    'mv_new_or_upgraded_points_per_1000_existing_points': ('New/upgraded MV connection points per 1,000 MV points', 'Consumer load', 1),
    'mv_consumer_connection_duration_days': ('Duration of consumer connection to MV network, days', 'Consumer load', 0),
    # Flexibility
    'lv_bess_capacity_to_peak_load_kw_per_mw': ('BESS capacity relative to peak LV load, kW/MW', 'Flexibility', 0),
    'mv_bess_capacity_to_peak_load_kw_per_mw': ('BESS capacity relative to peak MV load, kW/MW', 'Flexibility', 2),
    'lv_bess_connection_applications_per_1000_consumers': ('BESS connection applications per 1,000 LV consumers', 'Flexibility', 2),
    'mv_bess_connection_applications_per_1000_consumers': ('BESS connection applications per 1,000 MV consumers', 'Flexibility', 1),
    'mv_bess_application_processing_time_days': ('Processing time of connection requests for consumption devices and storage at MV level, days', 'Flexibility', 1),
    'prosumers_full_market_mode_per_1000_lv_consumers': ('Prosumers in full market mode per 1,000 LV consumers', 'Flexibility', 2),
}
DIMENSION_ORDER = ['Physical network architecture', 'Operational controllability',
                   'Generation capacity', 'Consumer load', 'Flexibility', 'Other']

def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def spearman_ci(rho, n):
    """95% CI for Spearman's rho: Fisher z with the Bonett–Wright variance."""
    if np.isclose(abs(rho), 1.0, rtol=0, atol=1e-14):
        return rho, rho
    z = np.arctanh(rho)
    standard_error = np.sqrt((1 + rho**2 / 2) / (n - 3))
    lower, upper = np.tanh(z + np.array([-1, 1]) * norm.ppf(0.975) * standard_error)
    return float(lower), float(upper)


def fmt(value, digits):
    """Number with a thousands separator and a typographic minus."""
    if not np.isfinite(value):
        return '–'
    text = f'{value:,.{digits}f}'
    if text.lstrip('-').strip('0.,') == '':
        text = text.lstrip('-')          # avoid "−0.00"
    return text.replace('-', MINUS)


def load_source(path):
    source = pd.read_csv(path, sep=';', decimal=',', encoding='utf-8-sig',
                         dtype={'dso_id': 'string'}, float_precision='round_trip')
    source['dps_type'] = (source['dps_type'].str.strip()
                          .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
    source['dso_id'] = source['dso_id'].str.strip()
    if source['dso_id'].isna().any() or source['dso_id'].duplicated().any():
        raise ValueError('DSO IDs must be complete and unique.')
    return source


def build_table(source, dps_type):
    outcome_columns = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}
    excluded_columns = {'dso_id', 'dso_name', 'dps_type', 'profile_group', *outcome_columns}
    indicator_columns = [c for c in source.columns if c not in excluded_columns]

    # 1. Operators of the given type with both continuity indicators (empty cells = missing).
    group = source.loc[source['dps_type'].eq(dps_type)].copy()
    for column in [*indicator_columns, *outcome_columns]:
        group[column] = pd.to_numeric(group[column], errors='raise')
        if np.isinf(group[column].dropna().to_numpy(dtype=float)).any():
            raise ValueError(f'{column}: infinite values are not supported.')
    group = group.rename(columns=outcome_columns).dropna(subset=['SAIDI', 'ASIDI'])
    if len(group) < MIN_N or (group[['SAIDI', 'ASIDI']] < 0).any().any():
        raise ValueError('Insufficient observations or negative interruption durations.')

    # 2. ICPS and its quartiles (same rules as Table S2).
    x = group[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
    p10, p90 = np.quantile(x, [0.10, 0.90], axis=0, method='linear')
    if np.any(p90 <= p10):
        raise ValueError('P90 must exceed P10 for both continuity components.')
    normalized = (p90 - np.clip(x, p10, p90)) / (p90 - p10)
    shares = normalized / normalized.sum(axis=0)
    terms = np.zeros_like(shares)
    positive = shares > 0
    terms[positive] = shares[positive] * np.log(shares[positive])
    entropy = -terms.sum(axis=0) / np.log(len(group))
    differentiation = np.maximum(1 - entropy, 0)
    if differentiation.sum() <= np.finfo(float).eps:
        raise ValueError('Entropy weights cannot be determined.')
    weights = differentiation / differentiation.sum()
    group['ICPS'] = normalized @ weights
    p25, p50, p75 = group['ICPS'].quantile([0.25, 0.50, 0.75])
    if not p25 < p50 < p75:
        raise ValueError('ICPS quartile thresholds coincide.')
    group['ICPS_quartile'] = np.select(
        [group['ICPS'] <= p25, group['ICPS'] <= p50, group['ICPS'] < p75],
        ['Q1', 'Q2', 'Q3'], default='Q4')

    # 3. Test every eligible indicator against all three outcomes.
    tests = []
    for indicator in indicator_columns:
        observed = group[indicator].dropna()
        if len(observed) < MIN_N or observed.nunique() < 2:
            continue
        for outcome in ['ICPS', 'SAIDI', 'ASIDI']:
            pair = group[[indicator, outcome]].dropna()
            if len(pair) < MIN_N or pair[outcome].nunique() < 2:
                raise ValueError(f'{indicator} / {outcome}: insufficient pairwise variation.')
            rho, p_value = spearmanr(pair[indicator], pair[outcome])
            lower, upper = spearman_ci(float(rho), len(pair))
            tests.append({'Variable': indicator, 'Outcome': outcome, 'n': len(pair),
                          'rho': float(rho), 'CI lower': lower, 'CI upper': upper,
                          'p': float(p_value)})
    all_tests = pd.DataFrame(tests)
    # BH applied ONCE across all indicator–outcome tests of this DPS type.
    all_tests['q'] = bh_adjust(all_tests['p'])
    all_tests['Selected'] = (all_tests['q'] < FDR_THRESHOLD) & (all_tests['rho'].abs() >= MIN_ABS_RHO)

    # 4. One relationship per feature: the largest absolute selected rho.
    selected = all_tests.loc[all_tests['Selected']].copy()
    selected['Absolute rho'] = selected['rho'].abs()
    representatives = (selected.sort_values(['Absolute rho', 'q'], ascending=[False, True], kind='stable')
                       .drop_duplicates('Variable').set_index('Variable'))

    # 5. Benchmarks: share of non-zero values and medians of NON-ZERO values by ICPS quartile.
    display_rows = []
    for indicator in indicator_columns:
        if indicator not in representatives.index:
            continue
        result = representatives.loc[indicator]
        label, dimension, digits = DISPLAY_METADATA.get(indicator, (indicator, 'Other', 3))
        samples = [group.loc[group['ICPS_quartile'].eq('Q4'), indicator].dropna(),
                   group[indicator].dropna(),
                   group.loc[group['ICPS_quartile'].eq('Q1'), indicator].dropna()]
        nonzero = [s.loc[s.ne(0)] for s in samples]
        share_nonzero = [100 * len(nz) / len(s) if len(s) else np.nan for nz, s in zip(nonzero, samples)]
        medians = [float(nz.median()) if len(nz) >= MIN_NONZERO_N else np.nan for nz in nonzero]
        shown = [fmt(m, digits) for m in medians]
        identical = shown[0] == shown[2] and shown[0] != '–'
        display_rows.append({
            'Dimension': dimension,
            'Indicator': label + (' *' if identical else ''),
            'Outcome': result['Outcome'], 'n': int(result['n']),
            'rho [95% CI]': f"{fmt(result['rho'], 2)} [{fmt(result['CI lower'], 2)}; {fmt(result['CI upper'], 2)}]",
            'q': '<0.001' if result['q'] < 0.001 else f"{result['q']:.4f}",
            **({'Share of DSOs with non-zero values, %: Q4 / All / Q1':
                ' / '.join(fmt(v, 0) for v in share_nonzero)} if INCLUDE_NONZERO_SHARE else {}),
            'Best quartile': shown[0], 'All DSOs': shown[1], 'Worst quartile': shown[2]})

    order = {d: i for i, d in enumerate(DIMENSION_ORDER)}
    display = pd.DataFrame(display_rows).sort_values('Dimension', key=lambda s: s.map(order), kind='stable')

    return display


source = load_source(DATA_PATH)
table_s4 = build_table(source, DPS_TYPE).drop(columns='Dimension').reset_index(drop=True)

output_path = Path(TABLE_PATH) if TABLE_PATH else OUTPUT_DIRECTORY / f'{TABLE_ID}.csv'
table_s4.to_csv(output_path, sep=';', index=False, encoding='utf-8-sig')
print(f'{TABLE_ID} ({DPS_TYPE}): {len(table_s4)} indicators')
print(f'Saved to: {output_path}')

try:
    from IPython.display import display
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
        display(table_s4)
except ImportError:
    print(table_s4.to_string(index=False))

Table_S4 (LV–MV): 30 indicators
Saved to: ../results/Table_S4.csv


,Indicator,Outcome,n,rho [95% CI],q,"Share of DSOs with non-zero values, %: Q4 / All / Q1",Best quartile,All DSOs,Worst quartile
0,"Ratio of LV to MV network length, km/km",ASIDI,669,−0.19 [−0.26; −0.12],<0.001,100 / 100 / 100,2.66,2.50,2.45
1,"Network length per generation point, m/point",ICPS,667,−0.11 [−0.18; −0.03],0.0229,100 / 100 / 100,335,342,351
2,"Vertical exchange intensity, kW/kW",ASIDI,636,−0.13 [−0.21; −0.05],0.0063,96 / 98 / 99,1.00,0.99,0.98
3,"GIS coverage of LV network, % *",ICPS,667,−0.20 [−0.28; −0.13],<0.001,85 / 89 / 93,100,100,100
4,"GIS coverage of MV network, % *",SAIDI,667,0.19 [0.12; 0.27],<0.001,89 / 93 / 95,100,100,100
5,"Share of non-remote maximum-demand meters, % *",ICPS,661,−0.19 [−0.26; −0.11],<0.001,78 / 87 / 92,90,90,90
6,"Share of HV/MV transformers with centralized online monitoring, % *",ASIDI,386,0.32 [0.22; 0.41],<0.001,15 / 38 / 46,100,100,100
7,"Share of MV network equipment with centralized control, % *",SAIDI,664,0.16 [0.09; 0.24],<0.001,51 / 67 / 74,20,20,20
8,"Share of MV/LV substations with centralized control, % *",ASIDI,657,0.12 [0.05; 0.20],0.0077,15 / 18 / 22,10,10,10
9,"Share of HV/MV substations with centralized control, % *",SAIDI,386,0.30 [0.20; 0.39],<0.001,12 / 33 / 43,100,100,100


In [5]:
# Supplementary Materials, Table S5
# Selected indicators associated with continuity of power supply and their benchmarks by ICPS quartile (LV–MV–HV)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm, spearmanr

DEFAULT_DATA_PATH = Path('../data/DSO_2024_corrected.csv')

_cli_paths = [Path(a) for a in sys.argv[1:] if a.lower().endswith('.csv')]
DATA_PATH = _cli_paths[0] if _cli_paths else DEFAULT_DATA_PATH
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Data file not found: {DATA_PATH}')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
MIN_N = 10         
MIN_NONZERO_N = 5     
FDR_THRESHOLD = 0.10
MIN_ABS_RHO = 0.10
MINUS = '\u2212'
INCLUDE_NONZERO_SHARE = True   
DPS_TYPE = 'LV–MV–HV'          # network architecture reported in Table S5
TABLE_ID = 'Table_S5'
TABLE_PATH = None              # None -> Table_S5.csv next to the data file

DISPLAY_METADATA = {
    # Physical network architecture
    'mv_network_density_km_per_km2': ('MV network density, km/km²', 'Physical network architecture', 2),
    'lv_to_mv_network_length_ratio': ('Ratio of LV to MV network length, km/km', 'Physical network architecture', 2),
    'network_length_per_generation_point_m': ('Network length per generation point, m/point', 'Physical network architecture', 0),
    'generation_points_per_1000_consumption_points': ('Generation points per 1,000 consumption points', 'Physical network architecture', 0),
    'vertical_exchange_intensity_kw_per_kw': ('Vertical exchange intensity, kW/kW', 'Physical network architecture', 2),
    # Operational controllability
    'lv_gis_coverage_pct': ('GIS coverage of LV network, %', 'Operational controllability', 0),
    'mv_gis_coverage_pct': ('GIS coverage of MV network, %', 'Operational controllability', 0),
    'non_remote_maximum_demand_meters_share_pct': ('Share of non-remote maximum-demand meters, %', 'Operational controllability', 0),
    'hv_lines_centralized_online_monitoring_share_pct': ('Share of HV lines with centralized online monitoring, %', 'Operational controllability', 0),
    'hv_mv_transformers_centralized_online_monitoring_share_pct': ('Share of HV/MV transformers with centralized online monitoring, %', 'Operational controllability', 0),
    'mv_equipment_centralized_control_share_pct': ('Share of MV network equipment with centralized control, %', 'Operational controllability', 0),
    'mv_lv_substations_centralized_control_share_pct': ('Share of MV/LV substations with centralized control, %', 'Operational controllability', 0),
    'hv_mv_substations_centralized_control_share_pct': ('Share of HV/MV substations with centralized control, %', 'Operational controllability', 0),
    # Generation capacity
    'lv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak LV load, kW/kW', 'Generation capacity', 2),
    'mv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak MV load, kW/kW', 'Generation capacity', 2),
    'hv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak HV load, kW/kW', 'Generation capacity', 2),
    'lv_dg_capacity_utilization_factor_pct': ('DG capacity utilization factor (CUF) at LV level, %', 'Generation capacity', 2),
    'mv_dg_capacity_utilization_factor_pct': ('DG capacity utilization factor (CUF) at MV level, %', 'Generation capacity', 1),
    'lv_solar_generation_share_pct': ('Share of solar generation at LV level, %', 'Generation capacity', 0),
    'lv_wind_generation_share_pct': ('Share of wind generation at LV level, %', 'Generation capacity', 3),
    'mv_solar_generation_share_pct': ('Share of solar generation at MV level, %', 'Generation capacity', 1),
    'mv_wind_generation_share_pct': ('Share of wind generation at MV level, %', 'Generation capacity', 1),
    'lv_other_res_connection_application_avg_capacity_kw': ('Average capacity of other RES connection applications at LV level, kW/point', 'Generation capacity', 1),
    'mv_solar_connection_application_avg_capacity_kw': ('Average capacity of solar connection applications at MV level, kW/point', 'Generation capacity', 0),
    'hv_solar_connection_application_avg_capacity_kw': ('Average capacity of solar connection applications at HV level, kW/point', 'Generation capacity', 0),
    # Consumer load
    'lv_peak_load_per_consumer_kw': ('Peak LV load per LV consumer, kW/consumer', 'Consumer load', 2),
    'mv_peak_load_per_consumer_kw': ('Peak MV load per MV consumer, kW/consumer', 'Consumer load', 0),
    'mv_ev_charging_capacity_to_peak_load_pct': ('Share of EV charging stations in peak MV load, %', 'Consumer load', 2),
    'lv_connection_applications_per_1000_consumers': ('LV connection applications per 1,000 LV consumers', 'Consumer load', 1),
    'mv_connection_applications_per_1000_consumers': ('MV connection applications per 1,000 MV consumers', 'Consumer load', 1),
    'mv_new_or_upgraded_points_per_1000_existing_points': ('New/upgraded MV connection points per 1,000 MV points', 'Consumer load', 1),
    'mv_consumer_connection_duration_days': ('Duration of consumer connection to MV network, days', 'Consumer load', 0),
    # Flexibility
    'lv_bess_capacity_to_peak_load_kw_per_mw': ('BESS capacity relative to peak LV load, kW/MW', 'Flexibility', 0),
    'mv_bess_capacity_to_peak_load_kw_per_mw': ('BESS capacity relative to peak MV load, kW/MW', 'Flexibility', 2),
    'lv_bess_connection_applications_per_1000_consumers': ('BESS connection applications per 1,000 LV consumers', 'Flexibility', 2),
    'mv_bess_connection_applications_per_1000_consumers': ('BESS connection applications per 1,000 MV consumers', 'Flexibility', 1),
    'mv_bess_application_processing_time_days': ('Processing time of connection requests for consumption devices and storage at MV level, days', 'Flexibility', 1),
    'prosumers_full_market_mode_per_1000_lv_consumers': ('Prosumers in full market mode per 1,000 LV consumers', 'Flexibility', 2),
}
DIMENSION_ORDER = ['Physical network architecture', 'Operational controllability',
                   'Generation capacity', 'Consumer load', 'Flexibility', 'Other']

def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def spearman_ci(rho, n):
    """95% CI for Spearman's rho: Fisher z with the Bonett–Wright variance."""
    if np.isclose(abs(rho), 1.0, rtol=0, atol=1e-14):
        return rho, rho
    z = np.arctanh(rho)
    standard_error = np.sqrt((1 + rho**2 / 2) / (n - 3))
    lower, upper = np.tanh(z + np.array([-1, 1]) * norm.ppf(0.975) * standard_error)
    return float(lower), float(upper)


def fmt(value, digits):
    """Number with a thousands separator and a typographic minus."""
    if not np.isfinite(value):
        return '–'
    text = f'{value:,.{digits}f}'
    if text.lstrip('-').strip('0.,') == '':
        text = text.lstrip('-')          # avoid "−0.00"
    return text.replace('-', MINUS)


def load_source(path):
    source = pd.read_csv(path, sep=';', decimal=',', encoding='utf-8-sig',
                         dtype={'dso_id': 'string'}, float_precision='round_trip')
    source['dps_type'] = (source['dps_type'].str.strip()
                          .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
    source['dso_id'] = source['dso_id'].str.strip()
    if source['dso_id'].isna().any() or source['dso_id'].duplicated().any():
        raise ValueError('DSO IDs must be complete and unique.')
    return source


def build_table(source, dps_type):
    outcome_columns = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}
    excluded_columns = {'dso_id', 'dso_name', 'dps_type', 'profile_group', *outcome_columns}
    indicator_columns = [c for c in source.columns if c not in excluded_columns]

    # 1. Operators of the given type with both continuity indicators (empty cells = missing).
    group = source.loc[source['dps_type'].eq(dps_type)].copy()
    for column in [*indicator_columns, *outcome_columns]:
        group[column] = pd.to_numeric(group[column], errors='raise')
        if np.isinf(group[column].dropna().to_numpy(dtype=float)).any():
            raise ValueError(f'{column}: infinite values are not supported.')
    group = group.rename(columns=outcome_columns).dropna(subset=['SAIDI', 'ASIDI'])
    if len(group) < MIN_N or (group[['SAIDI', 'ASIDI']] < 0).any().any():
        raise ValueError('Insufficient observations or negative interruption durations.')

    # 2. ICPS and its quartiles (same rules as Table S2)
    x = group[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
    p10, p90 = np.quantile(x, [0.10, 0.90], axis=0, method='linear')
    if np.any(p90 <= p10):
        raise ValueError('P90 must exceed P10 for both continuity components.')
    normalized = (p90 - np.clip(x, p10, p90)) / (p90 - p10)
    shares = normalized / normalized.sum(axis=0)
    terms = np.zeros_like(shares)
    positive = shares > 0
    terms[positive] = shares[positive] * np.log(shares[positive])
    entropy = -terms.sum(axis=0) / np.log(len(group))
    differentiation = np.maximum(1 - entropy, 0)
    if differentiation.sum() <= np.finfo(float).eps:
        raise ValueError('Entropy weights cannot be determined.')
    weights = differentiation / differentiation.sum()
    group['ICPS'] = normalized @ weights
    p25, p50, p75 = group['ICPS'].quantile([0.25, 0.50, 0.75])
    if not p25 < p50 < p75:
        raise ValueError('ICPS quartile thresholds coincide.')
    group['ICPS_quartile'] = np.select(
        [group['ICPS'] <= p25, group['ICPS'] <= p50, group['ICPS'] < p75],
        ['Q1', 'Q2', 'Q3'], default='Q4')

    # 3. Test every eligible indicator against all three outcomes.
    tests = []
    for indicator in indicator_columns:
        observed = group[indicator].dropna()
        if len(observed) < MIN_N or observed.nunique() < 2:
            continue
        for outcome in ['ICPS', 'SAIDI', 'ASIDI']:
            pair = group[[indicator, outcome]].dropna()
            if len(pair) < MIN_N or pair[outcome].nunique() < 2:
                raise ValueError(f'{indicator} / {outcome}: insufficient pairwise variation.')
            rho, p_value = spearmanr(pair[indicator], pair[outcome])
            lower, upper = spearman_ci(float(rho), len(pair))
            tests.append({'Variable': indicator, 'Outcome': outcome, 'n': len(pair),
                          'rho': float(rho), 'CI lower': lower, 'CI upper': upper,
                          'p': float(p_value)})
    all_tests = pd.DataFrame(tests)
    # BH applied ONCE across all indicator–outcome tests of this DPS type.
    all_tests['q'] = bh_adjust(all_tests['p'])
    all_tests['Selected'] = (all_tests['q'] < FDR_THRESHOLD) & (all_tests['rho'].abs() >= MIN_ABS_RHO)

    # 4. One relationship per feature: the largest absolute selected rho.
    selected = all_tests.loc[all_tests['Selected']].copy()
    selected['Absolute rho'] = selected['rho'].abs()
    representatives = (selected.sort_values(['Absolute rho', 'q'], ascending=[False, True], kind='stable')
                       .drop_duplicates('Variable').set_index('Variable'))

    # 5. Benchmarks: share of non-zero values and medians of NON-ZERO values by ICPS quartile.
    display_rows = []
    for indicator in indicator_columns:
        if indicator not in representatives.index:
            continue
        result = representatives.loc[indicator]
        label, dimension, digits = DISPLAY_METADATA.get(indicator, (indicator, 'Other', 3))
        samples = [group.loc[group['ICPS_quartile'].eq('Q4'), indicator].dropna(),
                   group[indicator].dropna(),
                   group.loc[group['ICPS_quartile'].eq('Q1'), indicator].dropna()]
        nonzero = [s.loc[s.ne(0)] for s in samples]
        share_nonzero = [100 * len(nz) / len(s) if len(s) else np.nan for nz, s in zip(nonzero, samples)]
        medians = [float(nz.median()) if len(nz) >= MIN_NONZERO_N else np.nan for nz in nonzero]
        shown = [fmt(m, digits) for m in medians]
        identical = shown[0] == shown[2] and shown[0] != '–'
        display_rows.append({
            'Dimension': dimension,
            'Indicator': label + (' *' if identical else ''),
            'Outcome': result['Outcome'], 'n': int(result['n']),
            'rho [95% CI]': f"{fmt(result['rho'], 2)} [{fmt(result['CI lower'], 2)}; {fmt(result['CI upper'], 2)}]",
            'q': '<0.001' if result['q'] < 0.001 else f"{result['q']:.4f}",
            **({'Share of DSOs with non-zero values, %: Q4 / All / Q1':
                ' / '.join(fmt(v, 0) for v in share_nonzero)} if INCLUDE_NONZERO_SHARE else {}),
            'Best quartile': shown[0], 'All DSOs': shown[1], 'Worst quartile': shown[2]})

    order = {d: i for i, d in enumerate(DIMENSION_ORDER)}
    display = pd.DataFrame(display_rows).sort_values('Dimension', key=lambda s: s.map(order), kind='stable')

    return display


source = load_source(DATA_PATH)
table_s5 = build_table(source, DPS_TYPE).drop(columns='Dimension').reset_index(drop=True)

output_path = Path(TABLE_PATH) if TABLE_PATH else OUTPUT_DIRECTORY / f'{TABLE_ID}.csv'
table_s5.to_csv(output_path, sep=';', index=False, encoding='utf-8-sig')
print(f'{TABLE_ID} ({DPS_TYPE}): {len(table_s5)} indicators')
print(f'Saved to: {output_path}')

try:
    from IPython.display import display
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
        display(table_s5)
except ImportError:
    print(table_s5.to_string(index=False))

Table_S5 (LV–MV–HV): 16 indicators
Saved to: ../results/Table_S5.csv


,Indicator,Outcome,n,rho [95% CI],q,"Share of DSOs with non-zero values, %: Q4 / All / Q1",Best quartile,All DSOs,Worst quartile
0,"MV network density, km/km²",ASIDI,97,−0.32 [−0.49; −0.12],0.0457,100 / 100 / 100,2.75,2.80,1.27
1,"Network length per generation point, m/point",SAIDI,97,0.27 [0.07; 0.45],0.0807,100 / 100 / 100,427,459,452
2,"Generation points per 1,000 consumption points",SAIDI,97,−0.32 [−0.49; −0.13],0.0457,100 / 100 / 100,154,134,138
3,"GIS coverage of LV network, % *",SAIDI,97,0.30 [0.11; 0.48],0.0557,92 / 97 / 100,100,100,100
4,"GIS coverage of MV network, % *",ICPS,97,−0.28 [−0.46; −0.09],0.0623,92 / 98 / 100,100,100,100
5,"Share of non-remote maximum-demand meters, %",ICPS,94,−0.31 [−0.48; −0.11],0.0557,79 / 91 / 100,90,100,100
6,"Share of HV lines with centralized online monitoring, % *",ASIDI,96,0.29 [0.09; 0.47],0.0623,88 / 96 / 100,100,100,100
7,"Share of HV/MV transformers with centralized online monitoring, % *",ICPS,95,−0.28 [−0.46; −0.08],0.0651,91 / 97 / 100,100,100,100
8,"DG capacity relative to peak MV load, kW/kW",ASIDI,97,0.26 [0.06; 0.44],0.0924,100 / 98 / 96,0.42,0.39,1.07
9,"DG capacity relative to peak HV load, kW/kW",ASIDI,92,0.54 [0.37; 0.68],<0.001,5 / 39 / 80,–,0.10,0.20


In [6]:
# Supplementary Materials, Table S6
# Mann–Whitney comparison of DSO profile indicators and continuity outcomes between LV–MV and LV–MV–HV systems

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DATA_PATH = Path('../data/DSO_2024_corrected.csv')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S6'
ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
EXPECTED_GROUP_SIZES = {'LV–MV': 669, 'LV–MV–HV': 97}
MIN_N = 10
FDR_THRESHOLD = 0.10
OUTCOME_VARIABLES = {'saidi_lv_min_per_year', 'asidi_mv_min_per_year'}
EXPECTED_LOCAL_TESTS = 64
EXPECTED_TOTAL_TESTS = 66
MINUS = '\u2212'
# Indicators defined at the HV level: by construction absent (zero) in LV–MV systems.
HV_LEVEL_VARIABLES = {'hv_gis_coverage_pct', 'hv_lines_centralized_online_monitoring_share_pct',
                      'hv_mv_transformers_centralized_online_monitoring_share_pct',
                      'hv_equipment_centralized_control_share_pct',
                      'hv_mv_substations_centralized_control_share_pct'}

# labels for all manuscript profile indicators and continuity outcomes.
INDICATOR_METADATA = [('lv_network_density_km_per_km2', 'LV network density'),
 ('mv_network_density_km_per_km2', 'MV network density'),
 ('lv_to_mv_network_length_ratio', 'Ratio of LV to MV network length'),
 ('network_length_per_consumption_point_m', 'Network length per consumption point'),
 ('network_length_per_generation_point_m', 'Network length per generation point'),
 ('generation_points_per_1000_consumption_points', 'Generation points per 1,000 consumption points'),
 ('vertical_exchange_intensity_kw_per_kw', 'Vertical exchange intensity'),
 ('lv_gis_coverage_pct', 'GIS coverage of LV network'),
 ('mv_gis_coverage_pct', 'GIS coverage of MV network'),
 ('hv_gis_coverage_pct', 'GIS coverage of HV network'),
 ('non_remote_maximum_demand_meters_share_pct', 'Share of non-remote maximum-demand meters'),
 ('local_substations_remote_transformer_measurements_share_pct',
  'Share of local substations with remote measurements at the transformer'),
 ('local_substations_remote_lv_board_measurements_share_pct',
  'Share of local substations with remote measurements in the LV distribution board'),
 ('lv_lines_centralized_online_monitoring_share_pct', 'Share of LV lines with centralized online monitoring'),
 ('mv_lines_centralized_online_monitoring_share_pct', 'Share of MV lines with centralized online monitoring'),
 ('hv_lines_centralized_online_monitoring_share_pct', 'Share of HV lines with centralized online monitoring'),
 ('mv_lv_transformers_centralized_online_monitoring_share_pct',
  'Share of MV/LV transformers with centralized online monitoring'),
 ('hv_mv_transformers_centralized_online_monitoring_share_pct',
  'Share of HV/MV transformers with centralized online monitoring'),
 ('lv_equipment_centralized_control_share_pct', 'Share of LV network equipment with centralized control'),
 ('mv_equipment_centralized_control_share_pct', 'Share of MV network equipment with centralized control'),
 ('hv_equipment_centralized_control_share_pct', 'Share of HV network equipment with centralized control'),
 ('mv_lv_substations_centralized_control_share_pct', 'Share of MV/LV substations with centralized control'),
 ('hv_mv_substations_centralized_control_share_pct', 'Share of HV/MV substations with centralized control'),
 ('lv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak LV load'),
 ('mv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak MV load'),
 ('hv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak HV load'),
 ('lv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at LV level'),
 ('mv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at MV level'),
 ('hv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at HV level'),
 ('lv_solar_generation_share_pct', 'Share of solar generation at LV level'),
 ('lv_wind_generation_share_pct', 'Share of wind generation at LV level'),
 ('lv_other_res_generation_share_pct', 'Share of other RES at LV level'),
 ('mv_solar_generation_share_pct', 'Share of solar generation at MV level'),
 ('mv_wind_generation_share_pct', 'Share of wind generation at MV level'),
 ('mv_other_res_generation_share_pct', 'Share of other RES at MV level'),
 ('hv_solar_generation_share_pct', 'Share of solar generation at HV level'),
 ('hv_wind_generation_share_pct', 'Share of wind generation at HV level'),
 ('hv_other_res_generation_share_pct', 'Share of other RES at HV level'),
 ('lv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at LV level'),
 ('lv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at LV level'),
 ('lv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at LV level'),
 ('mv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at MV level'),
 ('mv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at MV level'),
 ('mv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at MV level'),
 ('hv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at HV level'),
 ('hv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at HV level'),
 ('lv_generation_connection_duration_days', 'Duration of generation connection to LV network'),
 ('mv_generation_connection_duration_days', 'Duration of generation connection to MV network'),
 ('hv_generation_connection_duration_days', 'Duration of generation connection to HV network'),
 ('lv_peak_load_per_consumer_kw', 'Peak LV load per LV consumer'),
 ('mv_peak_load_per_consumer_kw', 'Peak MV load per MV consumer'),
 ('hv_peak_load_per_consumer_kw', 'Peak HV load per HV consumer'),
 ('lv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak LV load'),
 ('mv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak MV load'),
 ('lv_heat_pump_capacity_to_peak_load_pct', 'Share of heat pumps in peak LV load'),
 ('lv_connection_applications_per_1000_consumers', 'LV connection applications per 1,000 LV consumers'),
 ('lv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded LV connection points per 1,000 LV points'),
 ('lv_consumer_connection_duration_days', 'Duration of consumer connection to LV network'),
 ('lv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at LV level'),
 ('mv_connection_applications_per_1000_consumers', 'MV connection applications per 1,000 MV consumers'),
 ('mv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded MV connection points per 1,000 MV points'),
 ('mv_consumer_connection_duration_days', 'Duration of consumer connection to MV network'),
 ('mv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at MV level'),
 ('lv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak LV load'),
 ('mv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak MV load'),
 ('lv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 LV consumers'),
 ('mv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 MV consumers'),
 ('lv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at LV level'),
 ('mv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at MV level'),
 ('prosumers_full_market_mode_per_1000_lv_consumers', 'Prosumers in full market mode per 1,000 LV consumers'),
 ('prosumer_market_mode_switch_duration_days', 'Time to switch prosumers to full market mode'),
 ('prosumer_switching_automation_pct', 'Level of automation of prosumer switching'),
 ('dynamic_tariff_consumers_per_1000_lv_consumers', 'Consumers with dynamic tariffs per 1,000 LV consumers'),
 ('saidi_lv_min_per_year', 'SAIDI (LV), min/year'),
 ('asidi_mv_min_per_year', 'ASIDI (MV), min/year')]


def bh_adjust(p_values):
    """Apply Benjamini–Hochberg correction to the complete family of 66 tests."""
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


# 1. Load and validate the corrected source.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
labels = dict(INDICATOR_METADATA)
required_columns = {'dso_id', 'dso_name', 'dps_type', *labels}
missing_columns = sorted(required_columns - set(source.columns))
if missing_columns:
    raise ValueError(f'Required source columns are missing: {missing_columns}')
source['dso_id'] = source['dso_id'].str.strip()
source['dps_type'] = (source['dps_type'].str.strip()
    .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
if source['dso_id'].isna().any() or source['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')
for variable in labels:
    source[variable] = pd.to_numeric(source[variable], errors='raise')
    if not np.isfinite(source[variable].dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{variable}: infinite values are not supported.')
if (source[list(OUTCOME_VARIABLES)] < 0).any().any():
    raise ValueError('Interruption durations must be non-negative.')

# 2.  complete-outcome analytical sample 
valid = (source['dps_type'].isin(ARCHITECTURES)
         & source[list(OUTCOME_VARIABLES)].notna().all(axis=1))
table_s6_excluded = source.loc[~valid, ['dso_id', 'dso_name', 'dps_type',
                                         'saidi_lv_min_per_year', 'asidi_mv_min_per_year']].copy()
data = source.loc[valid].copy()
groups = {a: data.loc[data['dps_type'].eq(a)].copy() for a in ARCHITECTURES}
for architecture, group in groups.items():
    if len(group) != EXPECTED_GROUP_SIZES[architecture]:
        raise ValueError(
            f'{architecture}: expected {EXPECTED_GROUP_SIZES[architecture]} DSOs, '
            f'found {len(group)}. Verify the corrected CSV and missing outcome values.')

# 3. Determine eligibility in BOTH architectures and perform two-sided tests.

rows, eligibility_rows = [], []
candidates = [column for column in source.columns if column in labels]
for variable in candidates:
    x = groups['LV–MV'][variable].dropna().to_numpy(dtype=float)
    y = groups['LV–MV–HV'][variable].dropna().to_numpy(dtype=float)
    eligible_x = len(x) >= MIN_N and len(np.unique(x)) >= 2
    eligible_y = len(y) >= MIN_N and len(np.unique(y)) >= 2
    common = eligible_x and eligible_y
    eligibility_rows.append({
        'Variable': variable, 'Indicator': labels[variable],
        'n LV–MV': len(x), 'n LV–MV–HV': len(y),
        'Unique LV–MV': len(np.unique(x)), 'Unique LV–MV–HV': len(np.unique(y)),
        'Eligible LV–MV': eligible_x, 'Eligible LV–MV–HV': eligible_y,
        'Common eligible indicator': common
    })
    if not common:
        continue

    # The returned U statistic corresponds to x, the LV–MV group.

    test = mannwhitneyu(x, y, alternative='two-sided',
                        method='asymptotic', use_continuity=True)
    u_lv_mv = float(test.statistic)
    u_lv_mv_hv = len(x) * len(y) - u_lv_mv

    # Orientation used in Table S6:
    # positive r means LV–MV–HV values tend to be higher;
    # negative r means LV–MV values tend to be higher.
    # Ties receive half weight in the underlying pairwise probability.
    rank_biserial = 1 - 2 * u_lv_mv / (len(x) * len(y))
    rows.append({
        'Variable': variable, 'Indicator': labels[variable],
        'n LV–MV': len(x), 'n LV–MV–HV': len(y),
        'Median LV–MV': float(np.median(x)), 'Median LV–MV–HV': float(np.median(y)),
        'U': u_lv_mv, 'U LV–MV–HV': u_lv_mv_hv,
        'Rank-biserial r': float(rank_biserial), 'p': float(test.pvalue),
        'Is outcome': variable in OUTCOME_VARIABLES,
        'HV-level indicator': variable in HV_LEVEL_VARIABLES
    })

table_s6_numeric = pd.DataFrame(rows)
table_s6_eligibility = pd.DataFrame(eligibility_rows)
local_test_count = int((~table_s6_numeric['Is outcome']).sum())
if len(table_s6_numeric) != EXPECTED_TOTAL_TESTS or local_test_count != EXPECTED_LOCAL_TESTS:
    raise ValueError(
        f'Expected 64 local indicator tests and 2 outcome tests; '
        f'found {local_test_count} local tests and {len(table_s6_numeric)} total tests. '
        'Verify source missing values and common-indicator eligibility.')

# 4. Adjust ALL 66 p-values together, before selecting significant differences.
table_s6_numeric['q'] = bh_adjust(table_s6_numeric['p'].to_numpy())
table_s6_numeric['P(LV–MV–HV > LV–MV), %'] = (
    100 * table_s6_numeric['U LV–MV–HV']
    / (table_s6_numeric['n LV–MV'] * table_s6_numeric['n LV–MV–HV']))
# Consistency check: r = 2P − 1.
if not np.allclose(2 * table_s6_numeric['P(LV–MV–HV > LV–MV), %'] / 100 - 1,
                   table_s6_numeric['Rank-biserial r'], atol=1e-12):
    raise ValueError('Probability of superiority is inconsistent with rank-biserial r.')
table_s6_numeric['Differs (q < 0.10)'] = table_s6_numeric['q'] < FDR_THRESHOLD

# 5. Create the manuscript display table.
def num(value, digits):
    text = f'{value:,.{digits}f}'
    if text.lstrip('-').strip('0.,') == '':
        text = text.lstrip('-')
    return text.replace('-', MINUS)

table_s6 = pd.DataFrame({
    'Indicator': table_s6_numeric['Indicator']
        + np.where(table_s6_numeric['HV-level indicator'], ' †', ''),
    'n LV–MV': table_s6_numeric['n LV–MV'],
    'n LV–MV–HV': table_s6_numeric['n LV–MV–HV'],
    'Median LV–MV': table_s6_numeric['Median LV–MV'].map(lambda v: num(v, 3)),
    'Median LV–MV–HV': table_s6_numeric['Median LV–MV–HV'].map(lambda v: num(v, 3)),
    # Probability of superiority: share of all operator pairs in which the LV–MV–HV
    # value is higher (ties counted as one half); equals (1 + r) / 2.
    'P(LV–MV–HV > LV–MV), %': table_s6_numeric['P(LV–MV–HV > LV–MV), %'].map(lambda v: f'{v:.1f}'),
    'Rank-biserial r': table_s6_numeric['Rank-biserial r'].map(lambda v: num(v, 3)),
    'p': table_s6_numeric['p'].map(lambda v: '<0.0001' if v < 0.0001 else f'{v:.4f}'),
    'q': table_s6_numeric['q'].map(lambda v: '<0.0001' if v < 0.0001 else f'{v:.4f}'),
    'Differs (q < 0.10)': table_s6_numeric['Differs (q < 0.10)'].map({True: 'TRUE', False: 'FALSE'}),
})
table_s6_significant = table_s6.loc[
    table_s6_numeric['Differs (q < 0.10)']].reset_index(drop=True)

summary_rows = []
for category, mask in [
    ('Local profile indicators', ~table_s6_numeric['Is outcome']),
    ('Local profile indicators without HV-level indicators',
     ~table_s6_numeric['Is outcome'] & ~table_s6_numeric['HV-level indicator']),
    ('Continuity outcomes', table_s6_numeric['Is outcome']),
    ('All tests', pd.Series(True, index=table_s6_numeric.index))
]:
    subset = table_s6_numeric.loc[mask]
    summary_rows.append({'Category': category, 'Tests': len(subset),
                         'Significant differences': int(subset['Differs (q < 0.10)'].sum())})
table_s6_summary = pd.DataFrame(summary_rows)

# 6. Export Table S6, significant differences, full-precision results, summary,
#    eligibility audit and excluded DSOs.
table_s6.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}.csv', sep=';', encoding='utf-8-sig', index=False)
table_s6_significant.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_significant.csv', sep=';',
                            encoding='utf-8-sig', index=False)
table_s6_numeric.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_numeric.csv', sep=';', decimal=',',
                        encoding='utf-8-sig', index=False)
table_s6_summary.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_summary.csv', sep=';',
                        encoding='utf-8-sig', index=False)
table_s6_eligibility.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_eligibility.csv', sep=';',
                            encoding='utf-8-sig', index=False)
table_s6_excluded.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_excluded_DSOs.csv', sep=';', decimal=',',
                         encoding='utf-8-sig', index=False)

print('Analytical sample: ' + '; '.join(f'{a} = {len(g)} DSOs' for a, g in groups.items()) + '.')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
print(f'Total tests: {len(table_s6_numeric)}; BH family includes SAIDI and ASIDI.')
print('Mann–Whitney U: two-sided, asymptotic, with tie and continuity corrections.')
print('Rank-biserial r: positive = higher values in LV–MV–HV; negative = higher values in LV–MV.')
print('P(LV–MV–HV > LV–MV): share of operator pairs with a higher LV–MV–HV value (ties = 1/2); r = 2P − 1.')
print('Medians are descriptive; significance concerns the underlying distributions.')
print('† HV-level indicators: zero in LV–MV systems by construction.')
try:
    from IPython.display import display
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.max_colwidth', None):
        display(table_s6_summary)
        display(table_s6)
except ImportError:
    print(table_s6_summary.to_string(index=False))
    print(table_s6.to_string(index=False))

Analytical sample: LV–MV = 669 DSOs; LV–MV–HV = 97 DSOs.
Saved to: ../results/Table_S6.csv
Total tests: 66; BH family includes SAIDI and ASIDI.
Mann–Whitney U: two-sided, asymptotic, with tie and continuity corrections.
Rank-biserial r: positive = higher values in LV–MV–HV; negative = higher values in LV–MV.
P(LV–MV–HV > LV–MV): share of operator pairs with a higher LV–MV–HV value (ties = 1/2); r = 2P − 1.
Medians are descriptive; significance concerns the underlying distributions.
† HV-level indicators: zero in LV–MV systems by construction.


,Category,Tests,Significant differences
0,Local profile indicators,64,39
1,Local profile indicators without HV-level indicators,59,34
2,Continuity outcomes,2,2
3,All tests,66,41


,Indicator,n LV–MV,n LV–MV–HV,Median LV–MV,Median LV–MV–HV,"P(LV–MV–HV > LV–MV), %",Rank-biserial r,p,q,Differs (q < 0.10)
0,LV network density,669,97,21.441,24.485,57.6,0.151,0.0158,0.0282,TRUE
1,MV network density,663,97,2.347,2.795,51.4,0.027,0.6666,0.7457,FALSE
2,Ratio of LV to MV network length,669,97,2.495,1.902,29.7,−0.406,<0.0001,<0.0001,TRUE
3,Network length per consumption point,669,97,53.406,55.399,55.3,0.105,0.0941,0.1478,FALSE
4,Network length per generation point,667,97,342.000,458.505,66.1,0.321,<0.0001,<0.0001,TRUE
5,"Generation points per 1,000 consumption points",669,97,151.291,133.608,39.4,−0.211,0.0008,0.0019,TRUE
6,Vertical exchange intensity,636,91,0.992,1.000,48.8,−0.023,0.7203,0.7668,FALSE
7,GIS coverage of LV network,667,97,100.000,100.000,61.0,0.221,<0.0001,<0.0001,TRUE
8,GIS coverage of MV network,667,97,100.000,100.000,57.2,0.144,0.0011,0.0026,TRUE
9,GIS coverage of HV network †,285,96,0.000,100.000,97.4,0.947,<0.0001,<0.0001,TRUE


In [7]:
# Supplementary Materials, Table S7
# Architecture differences in Spearman correlations between DSO profile indicators and
# continuity outcomes (Fisher z with the Bonett–Wright variance; BH within each outcome, 64 tests).

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm

DATA_PATH = Path('../data/DSO_2024_corrected.csv')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S7'
ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
OUTCOMES = ['ICPS', 'SAIDI', 'ASIDI']
EXPECTED_GROUP_SIZES = {'LV–MV': 669, 'LV–MV–HV': 97}
EXPECTED_COMMON_INDICATORS = 64
EXPECTED_TOTAL_TESTS = 192
MIN_N = 10
FDR_THRESHOLD = 0.10
MINUS = '\u2212'
# Indicators defined at the HV level: by construction absent (zero) in LV–MV systems.
HV_LEVEL_VARIABLES = {'hv_gis_coverage_pct', 'hv_lines_centralized_online_monitoring_share_pct',
                      'hv_mv_transformers_centralized_online_monitoring_share_pct',
                      'hv_equipment_centralized_control_share_pct',
                      'hv_mv_substations_centralized_control_share_pct'}

# Source variables and English labels for the 73 local manuscript indicators.
# Common eligibility is assessed from observations and variation in both groups.
INDICATOR_METADATA = [('lv_network_density_km_per_km2', 'LV network density'),
 ('mv_network_density_km_per_km2', 'MV network density'),
 ('lv_to_mv_network_length_ratio', 'Ratio of LV to MV network length'),
 ('network_length_per_consumption_point_m', 'Network length per consumption point'),
 ('network_length_per_generation_point_m', 'Network length per generation point'),
 ('generation_points_per_1000_consumption_points', 'Generation points per 1,000 consumption points'),
 ('vertical_exchange_intensity_kw_per_kw', 'Vertical exchange intensity'),
 ('lv_gis_coverage_pct', 'GIS coverage of LV network'),
 ('mv_gis_coverage_pct', 'GIS coverage of MV network'),
 ('hv_gis_coverage_pct', 'GIS coverage of HV network'),
 ('non_remote_maximum_demand_meters_share_pct', 'Share of non-remote maximum-demand meters'),
 ('local_substations_remote_transformer_measurements_share_pct',
  'Share of local substations with remote measurements at the transformer'),
 ('local_substations_remote_lv_board_measurements_share_pct',
  'Share of local substations with remote measurements in the LV distribution board'),
 ('lv_lines_centralized_online_monitoring_share_pct', 'Share of LV lines with centralized online monitoring'),
 ('mv_lines_centralized_online_monitoring_share_pct', 'Share of MV lines with centralized online monitoring'),
 ('hv_lines_centralized_online_monitoring_share_pct', 'Share of HV lines with centralized online monitoring'),
 ('mv_lv_transformers_centralized_online_monitoring_share_pct',
  'Share of MV/LV transformers with centralized online monitoring'),
 ('hv_mv_transformers_centralized_online_monitoring_share_pct',
  'Share of HV/MV transformers with centralized online monitoring'),
 ('lv_equipment_centralized_control_share_pct', 'Share of LV network equipment with centralized control'),
 ('mv_equipment_centralized_control_share_pct', 'Share of MV network equipment with centralized control'),
 ('hv_equipment_centralized_control_share_pct', 'Share of HV network equipment with centralized control'),
 ('mv_lv_substations_centralized_control_share_pct', 'Share of MV/LV substations with centralized control'),
 ('hv_mv_substations_centralized_control_share_pct', 'Share of HV/MV substations with centralized control'),
 ('lv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak LV load'),
 ('mv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak MV load'),
 ('hv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak HV load'),
 ('lv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at LV level'),
 ('mv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at MV level'),
 ('hv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at HV level'),
 ('lv_solar_generation_share_pct', 'Share of solar generation at LV level'),
 ('lv_wind_generation_share_pct', 'Share of wind generation at LV level'),
 ('lv_other_res_generation_share_pct', 'Share of other RES at LV level'),
 ('mv_solar_generation_share_pct', 'Share of solar generation at MV level'),
 ('mv_wind_generation_share_pct', 'Share of wind generation at MV level'),
 ('mv_other_res_generation_share_pct', 'Share of other RES at MV level'),
 ('hv_solar_generation_share_pct', 'Share of solar generation at HV level'),
 ('hv_wind_generation_share_pct', 'Share of wind generation at HV level'),
 ('hv_other_res_generation_share_pct', 'Share of other RES at HV level'),
 ('lv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at LV level'),
 ('lv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at LV level'),
 ('lv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at LV level'),
 ('mv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at MV level'),
 ('mv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at MV level'),
 ('mv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at MV level'),
 ('hv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at HV level'),
 ('hv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at HV level'),
 ('lv_generation_connection_duration_days', 'Duration of generation connection to LV network'),
 ('mv_generation_connection_duration_days', 'Duration of generation connection to MV network'),
 ('hv_generation_connection_duration_days', 'Duration of generation connection to HV network'),
 ('lv_peak_load_per_consumer_kw', 'Peak LV load per LV consumer'),
 ('mv_peak_load_per_consumer_kw', 'Peak MV load per MV consumer'),
 ('hv_peak_load_per_consumer_kw', 'Peak HV load per HV consumer'),
 ('lv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak LV load'),
 ('mv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak MV load'),
 ('lv_heat_pump_capacity_to_peak_load_pct', 'Share of heat pumps in peak LV load'),
 ('lv_connection_applications_per_1000_consumers', 'LV connection applications per 1,000 LV consumers'),
 ('lv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded LV connection points per 1,000 LV points'),
 ('lv_consumer_connection_duration_days', 'Duration of consumer connection to LV network'),
 ('lv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at LV level'),
 ('mv_connection_applications_per_1000_consumers', 'MV connection applications per 1,000 MV consumers'),
 ('mv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded MV connection points per 1,000 MV points'),
 ('mv_consumer_connection_duration_days', 'Duration of consumer connection to MV network'),
 ('mv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at MV level'),
 ('lv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak LV load'),
 ('mv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak MV load'),
 ('lv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 LV consumers'),
 ('mv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 MV consumers'),
 ('lv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at LV level'),
 ('mv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at MV level'),
 ('prosumers_full_market_mode_per_1000_lv_consumers', 'Prosumers in full market mode per 1,000 LV consumers'),
 ('prosumer_market_mode_switch_duration_days', 'Time to switch prosumers to full market mode'),
 ('prosumer_switching_automation_pct', 'Level of automation of prosumer switching'),
 ('dynamic_tariff_consumers_per_1000_lv_consumers', 'Consumers with dynamic tariffs per 1,000 LV consumers')]


def bh_adjust(p_values):
    """Apply BH to the entire supplied family, using unrounded p-values."""
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def calculate_icps(group):
    """Recalculate the architecture-specific ICPS used in Tables S2–S5."""
    group = group.copy()
    x = group[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
    if len(group) < 4 or not np.isfinite(x).all() or (x < 0).any():
        raise ValueError('ICPS requires valid complete outcome observations.')
    p10, p90 = np.quantile(x, [0.10, 0.90], axis=0, method='linear')
    if np.any(p90 <= p10):
        raise ValueError('P90 must exceed P10 for both continuity components.')
    normalized = (p90 - np.clip(x, p10, p90)) / (p90 - p10)
    shares = normalized / normalized.sum(axis=0)
    terms = np.zeros_like(shares)
    positive = shares > 0
    terms[positive] = shares[positive] * np.log(shares[positive])
    entropy = -terms.sum(axis=0) / np.log(len(group))
    differentiation = np.maximum(1 - entropy, 0)
    if differentiation.sum() <= np.finfo(float).eps:
        raise ValueError('Entropy weights cannot be determined.')
    weights = differentiation / differentiation.sum()
    group['ICPS'] = normalized @ weights
    return group, {'SAIDI weight': weights[0], 'ASIDI weight': weights[1],
                   'SAIDI P10': p10[0], 'SAIDI P90': p90[0],
                   'ASIDI P10': p10[1], 'ASIDI P90': p90[1]}


def compare_correlations(rho1, n1, rho2, n2):
    """Compare independent Spearman correlations on the Fisher-z scale.

    Variance for each Fisher-transformed correlation (Bonett–Wright):
        v = (1 + rho**2 / 2) / (n - 3)
    Test statistic:
        Z = (atanh(rho1) - atanh(rho2)) / sqrt(v1 + v2)
    """
    if min(n1, n2) <= 3 or not np.isfinite([rho1, rho2]).all():
        raise ValueError('Correlation comparisons require valid rho values and n > 3.')
    if max(abs(rho1), abs(rho2)) >= 1:
        raise ValueError('Fisher z is undefined for perfect correlations; use another method.')
    fisher1, fisher2 = np.arctanh(rho1), np.arctanh(rho2)
    variance1 = (1 + rho1**2 / 2) / (n1 - 3)
    variance2 = (1 + rho2**2 / 2) / (n2 - 3)
    standard_error = np.sqrt(variance1 + variance2)
    z_statistic = (fisher1 - fisher2) / standard_error
    p_value = 2 * norm.sf(abs(z_statistic))
    return float(z_statistic), float(p_value), float(variance1), float(variance2)


def num(value, digits):
    """Fixed decimals with a typographic minus (no '−0.000')."""
    text = f'{value:.{digits}f}'
    if text.lstrip('-').strip('0.') == '':
        text = text.lstrip('-')
    return text.replace('-', MINUS)


def pfmt(value):
    return '<0.0001' if value < 0.0001 else f'{value:.4f}'


# 1. Load the corrected source and retain missing values.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
labels = dict(INDICATOR_METADATA)
source_outcomes = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}
required_columns = {'dso_id', 'dso_name', 'dps_type', *source_outcomes, *labels}
missing_columns = sorted(required_columns - set(source.columns))
if missing_columns:
    raise ValueError(f'Required source columns are missing: {missing_columns}')
source['dso_id'] = source['dso_id'].str.strip()
source['dps_type'] = (source['dps_type'].str.strip()
    .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
if source['dso_id'].isna().any() or source['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')
for column in [*labels, *source_outcomes]:
    source[column] = pd.to_numeric(source[column], errors='raise')
    if not np.isfinite(source[column].dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{column}: infinite values are not supported.')
source = source.rename(columns=source_outcomes)
valid = source['dps_type'].isin(ARCHITECTURES) & source[['SAIDI', 'ASIDI']].notna().all(axis=1)
table_s7_excluded = source.loc[~valid, ['dso_id', 'dso_name', 'dps_type', 'SAIDI', 'ASIDI']].copy()
data = source.loc[valid].copy()

# 2. Recalculate ICPS independently within each architecture.
groups, parameter_rows = {}, []
for architecture in ARCHITECTURES:
    group = data.loc[data['dps_type'].eq(architecture)].copy()
    if len(group) != EXPECTED_GROUP_SIZES[architecture]:
        raise ValueError(
            f'{architecture}: expected {EXPECTED_GROUP_SIZES[architecture]} DSOs, '
            f'found {len(group)}. Verify the corrected CSV and missing outcome values.')
    group, parameters = calculate_icps(group)
    groups[architecture] = group
    parameter_rows.append({'DPS type': architecture, 'N': len(group), **parameters})
icps_parameters = pd.DataFrame(parameter_rows)

# 3. Determine common eligibility without filtering on correlation significance.
common_indicators, eligibility_rows = [], []
for variable in [column for column in source.columns if column in labels]:
    values1 = groups['LV–MV'][variable].dropna()
    values2 = groups['LV–MV–HV'][variable].dropna()
    eligible1 = len(values1) >= MIN_N and values1.nunique() >= 2
    eligible2 = len(values2) >= MIN_N and values2.nunique() >= 2
    common = eligible1 and eligible2
    eligibility_rows.append({
        'Variable': variable, 'Indicator': labels[variable],
        'n LV–MV': len(values1), 'n LV–MV–HV': len(values2),
        'Eligible LV–MV': eligible1, 'Eligible LV–MV–HV': eligible2,
        'Common eligible indicator': common
    })
    if common:
        common_indicators.append(variable)
if len(common_indicators) != EXPECTED_COMMON_INDICATORS:
    raise ValueError(
        f'Expected 64 common indicators, found {len(common_indicators)}. '
        'Check source missing values and indicator eligibility.')
table_s7_eligibility = pd.DataFrame(eligibility_rows)

# 4. Compare each common indicator's correlations with all three outcomes.
# Genuine zeros are retained; missing values are excluded pairwise.
rows, correlation_rows = [], []
for outcome in OUTCOMES:
    for variable in common_indicators:
        correlations = []
        for architecture in ARCHITECTURES:
            pair = groups[architecture][[variable, outcome]].dropna()
            if len(pair) < MIN_N or pair[outcome].nunique() < 2:
                raise ValueError(f'{architecture}: invalid pair {variable} / {outcome}.')
            rho, p_within = spearmanr(pair[variable], pair[outcome])
            correlations.append((float(rho), len(pair)))
            correlation_rows.append({
                'DPS type': architecture, 'Variable': variable, 'Indicator': labels[variable],
                'Outcome': outcome, 'n': len(pair), 'rho': float(rho),
                'Within-architecture p': float(p_within)
            })
        (rho1, n1), (rho2, n2) = correlations
        z_statistic, p_value, variance1, variance2 = compare_correlations(rho1, n1, rho2, n2)
        rows.append({
            'Variable': variable, 'Indicator': labels[variable], 'Outcome': outcome,
            'HV-level indicator': variable in HV_LEVEL_VARIABLES,
            'rho LV–MV': rho1, 'n LV–MV': n1,
            'rho LV–MV–HV': rho2, 'n LV–MV–HV': n2,
            'Delta rho': rho1 - rho2, 'Z': z_statistic, 'p (two-sided)': p_value,
            'BW variance LV–MV': variance1, 'BW variance LV–MV–HV': variance2
        })

table_s7_numeric = pd.DataFrame(rows)
table_s7_correlations = pd.DataFrame(correlation_rows)
if len(table_s7_numeric) != EXPECTED_TOTAL_TESTS:
    raise ValueError('The Table S7 comparison family must contain 192 tests.')

# 5. Apply the two BH definitions separately to the same unrounded p-values.
within_q = 'q (BH within outcome, 64 tests)'
across_q = 'q (BH across 192 tests)'
within_flag = 'Differs (within-outcome q < 0.10)'
across_flag = 'Differs (across-192 q < 0.10)'
for outcome in OUTCOMES:
    mask = table_s7_numeric['Outcome'].eq(outcome)
    if int(mask.sum()) != EXPECTED_COMMON_INDICATORS:
        raise ValueError(f'{outcome}: the BH family must contain 64 tests.')
    table_s7_numeric.loc[mask, within_q] = bh_adjust(
        table_s7_numeric.loc[mask, 'p (two-sided)'].to_numpy())
table_s7_numeric[across_q] = bh_adjust(table_s7_numeric['p (two-sided)'].to_numpy())
table_s7_numeric[within_flag] = table_s7_numeric[within_q] < FDR_THRESHOLD
table_s7_numeric[across_flag] = table_s7_numeric[across_q] < FDR_THRESHOLD

# 6. Sort within each outcome and build the manuscript display table.
table_s7_numeric['Outcome order'] = table_s7_numeric['Outcome'].map(
    {outcome: i for i, outcome in enumerate(OUTCOMES)})
table_s7_numeric = (table_s7_numeric.sort_values(
    ['Outcome order', 'p (two-sided)', 'Variable'], kind='stable')
    .drop(columns='Outcome order').reset_index(drop=True))
table_s7 = pd.DataFrame({
    'Indicator': table_s7_numeric['Indicator']
        + np.where(table_s7_numeric['HV-level indicator'], ' †', ''),
    'Outcome': table_s7_numeric['Outcome'],
    'ρ LV–MV': table_s7_numeric['rho LV–MV'].map(lambda v: num(v, 3)),
    'n LV–MV': table_s7_numeric['n LV–MV'],
    'ρ LV–MV–HV': table_s7_numeric['rho LV–MV–HV'].map(lambda v: num(v, 3)),
    'n LV–MV–HV': table_s7_numeric['n LV–MV–HV'],
    'Δρ': table_s7_numeric['Delta rho'].map(lambda v: num(v, 3)),
    'Z': table_s7_numeric['Z'].map(lambda v: num(v, 3)),
    'p (two-sided)': table_s7_numeric['p (two-sided)'].map(pfmt),
    within_q: table_s7_numeric[within_q].map(pfmt),
    across_q: table_s7_numeric[across_q].map(pfmt),
    within_flag: table_s7_numeric[within_flag].map({True: 'TRUE', False: 'FALSE'}),
})
table_s7_significant = table_s7.loc[table_s7_numeric[within_flag]].reset_index(drop=True)
table_s7_significant_across = table_s7.loc[table_s7_numeric[across_flag]].reset_index(drop=True)

summary_rows = []
for outcome in [*OUTCOMES, 'All outcomes', 'All outcomes without HV-level indicators']:
    if outcome == 'All outcomes':
        subset = table_s7_numeric
    elif outcome == 'All outcomes without HV-level indicators':
        subset = table_s7_numeric.loc[~table_s7_numeric['HV-level indicator']]
    else:
        subset = table_s7_numeric.loc[table_s7_numeric['Outcome'].eq(outcome)]
    summary_rows.append({
        'Outcome': outcome, 'Tests': len(subset),
        'Significant: BH within outcome': int(subset[within_flag].sum()),
        'Significant: BH across 192': int(subset[across_flag].sum())
    })
table_s7_summary = pd.DataFrame(summary_rows)

# 7. Export Table S7, significant differences, full-precision results, summary, input
#    correlations, eligibility audit, ICPS parameters and excluded DSOs.
exports = {
    '': (table_s7, False),
    '_significant_within_outcome': (table_s7_significant, False),
    '_significant_across_192': (table_s7_significant_across, False),
    '_numeric': (table_s7_numeric, True),
    '_summary': (table_s7_summary, False),
    '_correlations': (table_s7_correlations, True),
    '_eligibility': (table_s7_eligibility, False),
    '_ICPS_parameters': (icps_parameters, True),
    '_excluded_DSOs': (table_s7_excluded, True),
}
for suffix, (frame, decimal_comma) in exports.items():
    frame.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}{suffix}.csv', sep=';',
                 decimal=',' if decimal_comma else '.', encoding='utf-8-sig', index=False)

print('Analytical sample: ' + '; '.join(f'{a} = {len(g)} DSOs' for a, g in groups.items()) + '.')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
print(f'Common indicators: {len(common_indicators)}; correlation comparisons: {len(table_s7_numeric)}.')
print('Delta rho and Z are oriented as LV–MV minus LV–MV–HV.')
print('The final Differs flag uses BH within outcome (64 tests), as specified in the Methods.')
print('The across-192 decision is also retained.')
print('† HV-level indicators: zero in LV–MV systems by construction.')
try:
    from IPython.display import display
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.max_colwidth', None):
        display(table_s7_summary)
        display(table_s7)
except ImportError:
    print(table_s7_summary.to_string(index=False))
    print(table_s7.to_string(index=False))

Analytical sample: LV–MV = 669 DSOs; LV–MV–HV = 97 DSOs.
Saved to: ../results/Table_S7.csv
Common indicators: 64; correlation comparisons: 192.
Delta rho and Z are oriented as LV–MV minus LV–MV–HV.
The final Differs flag uses BH within outcome (64 tests), as specified in the Methods.
The across-192 decision is also retained.
† HV-level indicators: zero in LV–MV systems by construction.


,Outcome,Tests,Significant: BH within outcome,Significant: BH across 192
0,ICPS,64,0,0
1,SAIDI,64,0,0
2,ASIDI,64,1,0
3,All outcomes,192,1,0
4,All outcomes without HV-level indicators,177,1,0


,Indicator,Outcome,ρ LV–MV,n LV–MV,ρ LV–MV–HV,n LV–MV–HV,Δρ,Z,p (two-sided),"q (BH within outcome, 64 tests)",q (BH across 192 tests),Differs (within-outcome q < 0.10)
0,Share of wind generation at LV level,ICPS,−0.115,632,−0.429,93,0.314,2.932,0.0034,0.2157,0.2211,FALSE
1,"LV connection applications per 1,000 LV consumers",ICPS,0.108,644,−0.194,91,0.302,2.656,0.0079,0.2534,0.2546,FALSE
2,Share of EV charging stations in peak MV load,ICPS,−0.183,587,0.086,89,−0.270,−2.348,0.0189,0.3140,0.2546,FALSE
3,Share of MV network equipment with centralized control,ICPS,−0.136,664,0.122,96,−0.258,−2.333,0.0196,0.3140,0.2546,FALSE
4,GIS coverage of HV network †,ICPS,0.045,285,−0.220,96,0.265,2.229,0.0258,0.3303,0.2551,FALSE
5,Share of HV lines with centralized online monitoring †,ICPS,−0.038,278,−0.286,96,0.248,2.101,0.0357,0.3806,0.2952,FALSE
6,"New/upgraded MV connection points per 1,000 MV points",ICPS,−0.089,611,0.142,91,−0.231,−2.023,0.0431,0.3938,0.3183,FALSE
7,DG capacity utilization factor (CUF) at LV level,ICPS,0.068,666,−0.152,94,0.220,1.967,0.0492,0.3938,0.3500,FALSE
8,"BESS connection applications per 1,000 LV consumers",ICPS,0.066,627,−0.153,89,0.219,1.905,0.0567,0.4035,0.3632,FALSE
9,DG capacity relative to peak LV load,ICPS,0.172,659,−0.027,96,0.199,1.809,0.0704,0.4504,0.3974,FALSE


In [8]:
# Supplementary Materials, Table S8
# Global permutation test of architecture differences in association profiles.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata

DATA_PATH = Path('../data/DSO_2024_corrected.csv')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S8'
N_PERMUTATIONS = 999          # 9999 gives more stable p-values (≈10× longer run)
SEED = 20261116               # fixed seed for reproducibility
ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
OUTCOMES = ['ICPS', 'SAIDI', 'ASIDI']
EXPECTED_GROUP_SIZES = {'LV–MV': 669, 'LV–MV–HV': 97}
EXPECTED_COMMON_INDICATORS = 64
MIN_N = 10
FDR_THRESHOLD = 0.10
OUTCOME_COLUMNS = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}
NON_INDICATOR_COLUMNS = {'dso_id', 'dso_name', 'dps_type', 'profile_group', *OUTCOME_COLUMNS}
HV_LEVEL_VARIABLES = {'hv_gis_coverage_pct', 'hv_lines_centralized_online_monitoring_share_pct',
                      'hv_mv_transformers_centralized_online_monitoring_share_pct',
                      'hv_equipment_centralized_control_share_pct',
                      'hv_mv_substations_centralized_control_share_pct'}


def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def icps(saidi, asidi):
    """Architecture-specific ICPS (same rules as Tables S2–S5 and Table S7)."""
    x = np.column_stack([saidi, asidi])
    p10, p90 = np.quantile(x, [0.10, 0.90], axis=0, method='linear')
    if np.any(p90 <= p10):
        raise ValueError('P90 must exceed P10 for both continuity components.')
    normalized = (p90 - np.clip(x, p10, p90)) / (p90 - p10)
    shares = normalized / normalized.sum(axis=0)
    terms = np.zeros_like(shares)
    positive = shares > 0
    terms[positive] = shares[positive] * np.log(shares[positive])
    entropy = -terms.sum(axis=0) / np.log(len(x))
    differentiation = np.maximum(1 - entropy, 0)
    weights = differentiation / differentiation.sum()
    return normalized @ weights


def spearman(x, y):
    """Spearman rho with pairwise deletion; returns (rho, n)."""
    mask = ~np.isnan(x) & ~np.isnan(y)
    n = int(mask.sum())
    if n < MIN_N:
        return np.nan, n
    rx, ry = rankdata(x[mask]), rankdata(y[mask])
    if rx.std() == 0 or ry.std() == 0:
        return np.nan, n
    return float(np.corrcoef(rx, ry)[0, 1]), n


def fisher_z(rho1, n1, rho2, n2):
    """Z for two independent Spearman correlations, Bonett–Wright variance."""
    if not np.isfinite([rho1, rho2]).all() or min(n1, n2) <= 3 or max(abs(rho1), abs(rho2)) >= 1:
        return np.nan
    v1 = (1 + rho1**2 / 2) / (n1 - 3)
    v2 = (1 + rho2**2 / 2) / (n2 - 3)
    return (np.arctanh(rho1) - np.arctanh(rho2)) / np.sqrt(v1 + v2)


def statistics(labels, indicators, outcomes_raw, excluded_mask):
    """T = ΣZ² per outcome (all common indicators and without HV-level indicators)."""
    groups = {}
    for architecture in ARCHITECTURES:
        member = labels == architecture
        saidi, asidi = outcomes_raw[member, 0], outcomes_raw[member, 1]
        groups[architecture] = (indicators[member],
                                {'ICPS': icps(saidi, asidi), 'SAIDI': saidi, 'ASIDI': asidi})
    result = {}
    for outcome in OUTCOMES:
        z_values = np.array([
            fisher_z(*spearman(groups['LV–MV'][0][:, j], groups['LV–MV'][1][outcome]),
                     *spearman(groups['LV–MV–HV'][0][:, j], groups['LV–MV–HV'][1][outcome]))
            for j in range(indicators.shape[1])])
        valid = np.isfinite(z_values)
        result[outcome] = (float(np.sum(z_values[valid] ** 2)), int(valid.sum()),
                           float(np.sum(z_values[valid & ~excluded_mask] ** 2)),
                           int((valid & ~excluded_mask).sum()))
    return result


# 1. Load the corrected source (empty cells = missing) and build the analytical sample.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
source['dps_type'] = (source['dps_type'].str.strip()
                      .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
indicator_columns = [c for c in source.columns if c not in NON_INDICATOR_COLUMNS]
for column in [*indicator_columns, *OUTCOME_COLUMNS]:
    source[column] = pd.to_numeric(source[column], errors='raise')
data = source.loc[source['dps_type'].isin(ARCHITECTURES)
                  & source[list(OUTCOME_COLUMNS)].notna().all(axis=1)].reset_index(drop=True)
for architecture, size in EXPECTED_GROUP_SIZES.items():
    if int(data['dps_type'].eq(architecture).sum()) != size:
        raise ValueError(f'{architecture}: expected {size} DSOs.')

# 2. Common indicators: at least MIN_N observations and variation in both architectures.
common = []
for column in indicator_columns:
    ok = True
    for architecture in ARCHITECTURES:
        values = data.loc[data['dps_type'].eq(architecture), column].dropna()
        ok &= len(values) >= MIN_N and values.nunique() >= 2
    if ok:
        common.append(column)
if len(common) != EXPECTED_COMMON_INDICATORS:
    raise ValueError(f'Expected 64 common indicators, found {len(common)}.')

labels = data['dps_type'].to_numpy()
indicators = data[common].to_numpy(dtype=float)
outcomes_raw = data[list(OUTCOME_COLUMNS)].to_numpy(dtype=float)
hv_mask = np.array([c in HV_LEVEL_VARIABLES for c in common])

# 3. Observed statistics and the permutation distribution.
observed = statistics(labels, indicators, outcomes_raw, hv_mask)
rng = np.random.default_rng(SEED)
permuted = {o: [] for o in OUTCOMES}
for b in range(N_PERMUTATIONS):
    result = statistics(rng.permutation(labels), indicators, outcomes_raw, hv_mask)
    for outcome in OUTCOMES:
        permuted[outcome].append(result[outcome])
    if (b + 1) % 100 == 0:
        print(f'{b + 1} of {N_PERMUTATIONS} permutations')

# 4. Permutation p-values and BH across the three outcomes (separately for each variant).
rows = []
for variant, t_index, k_index in [('All common indicators', 0, 1),
                                  ('Without HV-level indicators', 2, 3)]:
    for outcome in OUTCOMES:
        t_obs = observed[outcome][t_index]
        t_perm = np.array([r[t_index] for r in permuted[outcome]])
        rows.append({'Indicator set': variant, 'Outcome': outcome,
                     'Comparisons': observed[outcome][k_index],
                     'T observed': t_obs,
                     'T permuted: mean': float(t_perm.mean()),
                     'T permuted: 95th percentile': float(np.quantile(t_perm, 0.95)),
                     'p (permutation)': (1 + int(np.sum(t_perm >= t_obs))) / (N_PERMUTATIONS + 1)})
table_s8_numeric = pd.DataFrame(rows)
for variant in table_s8_numeric['Indicator set'].unique():
    mask = table_s8_numeric['Indicator set'].eq(variant)
    table_s8_numeric.loc[mask, 'q (BH, 3 tests)'] = bh_adjust(
        table_s8_numeric.loc[mask, 'p (permutation)'])
table_s8_numeric['Profiles differ (q < 0.10)'] = table_s8_numeric['q (BH, 3 tests)'] < FDR_THRESHOLD

# 5. Display table and export.
table_s8 = table_s8_numeric.copy()
for column in ['T observed', 'T permuted: mean', 'T permuted: 95th percentile']:
    table_s8[column] = table_s8_numeric[column].map(lambda v: f'{v:.1f}')
for column in ['p (permutation)', 'q (BH, 3 tests)']:
    table_s8[column] = table_s8_numeric[column].map(lambda v: f'{v:.3f}')
table_s8['Profiles differ (q < 0.10)'] = table_s8_numeric['Profiles differ (q < 0.10)'].map(
    {True: 'TRUE', False: 'FALSE'})

# Table S8, full-precision results and the permutation distributions of T
# (all common indicators and the variant without HV-level indicators).
table_s8.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}.csv', sep=';', encoding='utf-8-sig', index=False)
table_s8_numeric.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_numeric.csv', sep=';', decimal=',',
                        encoding='utf-8-sig', index=False)
table_s8_permutations = pd.DataFrame({
    **{f'{o} T (all common indicators)': [r[0] for r in permuted[o]] for o in OUTCOMES},
    **{f'{o} T (without HV-level indicators)': [r[2] for r in permuted[o]] for o in OUTCOMES},
})
table_s8_permutations.insert(0, 'Permutation', np.arange(1, N_PERMUTATIONS + 1))
table_s8_permutations.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_permutation_distribution.csv', sep=';',
                             decimal=',', encoding='utf-8-sig', index=False)

print(f'Permutations: {N_PERMUTATIONS}; seed: {SEED}; common indicators: {len(common)}.')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
try:
    from IPython.display import display
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        display(table_s8)
except ImportError:
    print(table_s8.to_string(index=False))

100 of 999 permutations


200 of 999 permutations


300 of 999 permutations


400 of 999 permutations


500 of 999 permutations


600 of 999 permutations


700 of 999 permutations


800 of 999 permutations


900 of 999 permutations


Permutations: 999; seed: 20261116; common indicators: 64.
Saved to: ../results/Table_S8.csv


,Indicator set,Outcome,Comparisons,T observed,T permuted: mean,T permuted: 95th percentile,p (permutation),"q (BH, 3 tests)",Profiles differ (q < 0.10)
0,All common indicators,ICPS,64,95.5,62.8,90.8,0.038,0.057,TRUE
1,All common indicators,SAIDI,64,88.6,63.2,93.4,0.082,0.082,TRUE
2,All common indicators,ASIDI,64,133.1,63.4,91.8,0.003,0.009,TRUE
3,Without HV-level indicators,ICPS,59,83.3,58.4,84.1,0.055,0.069,TRUE
4,Without HV-level indicators,SAIDI,59,83.2,58.6,86.8,0.069,0.069,TRUE
5,Without HV-level indicators,ASIDI,59,116.1,58.8,83.7,0.005,0.015,TRUE


In [9]:
# Supplementary Materials, Table S9
# Differences in benchmark values between LV–MV and LV–MV–HV DPSs
# (best ICPS quartile, non-zero values; two-sided Mann–Whitney U test; BH across features).


from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr

DATA_PATH = Path('../data/DSO_2024_corrected.csv')
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S9'
ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
EXPECTED_GROUP_SIZES = {'LV–MV': 669, 'LV–MV–HV': 97}
MIN_N = 10
MIN_NONZERO_N = 5
FDR_THRESHOLD = 0.10
MIN_ABS_RHO = 0.10
MINUS = '\u2212'
OUTCOME_COLUMNS = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}

# Indicator names and units exactly as in Appendix A.
INDICATOR_METADATA = [
    ('lv_network_density_km_per_km2', 'LV network density', 'km/km²'),
    ('mv_network_density_km_per_km2', 'MV network density', 'km/km²'),
    ('lv_to_mv_network_length_ratio', 'Ratio of LV to MV network length', 'km/km'),
    ('network_length_per_consumption_point_m', 'Network length per consumption point', 'm/point'),
    ('network_length_per_generation_point_m', 'Network length per generation point', 'm/point'),
    ('generation_points_per_1000_consumption_points', 'Generation points per 1,000 consumption points', 'units'),
    ('vertical_exchange_intensity_kw_per_kw', 'Vertical exchange intensity', 'kW/kW'),
    ('lv_gis_coverage_pct', 'GIS coverage of LV network', '%'),
    ('mv_gis_coverage_pct', 'GIS coverage of MV network', '%'),
    ('hv_gis_coverage_pct', 'GIS coverage of HV network', '%'),
    ('non_remote_maximum_demand_meters_share_pct', 'Share of non-remote maximum-demand meters', '%'),
    ('local_substations_remote_transformer_measurements_share_pct', 'Share of local substations with remote measurements at the transformer', '%'),
    ('local_substations_remote_lv_board_measurements_share_pct', 'Share of local substations with remote measurements in the LV distribution board', '%'),
    ('lv_lines_centralized_online_monitoring_share_pct', 'Share of LV lines with centralized online monitoring', '%'),
    ('mv_lines_centralized_online_monitoring_share_pct', 'Share of MV lines with centralized online monitoring', '%'),
    ('hv_lines_centralized_online_monitoring_share_pct', 'Share of HV lines with centralized online monitoring', '%'),
    ('mv_lv_transformers_centralized_online_monitoring_share_pct', 'Share of MV/LV transformers with centralized online monitoring', '%'),
    ('hv_mv_transformers_centralized_online_monitoring_share_pct', 'Share of HV/MV transformers with centralized online monitoring', '%'),
    ('lv_equipment_centralized_control_share_pct', 'Share of LV network equipment with centralized control', '%'),
    ('mv_equipment_centralized_control_share_pct', 'Share of MV network equipment with centralized control', '%'),
    ('hv_equipment_centralized_control_share_pct', 'Share of HV network equipment with centralized control', '%'),
    ('mv_lv_substations_centralized_control_share_pct', 'Share of MV/LV substations with centralized control', '%'),
    ('hv_mv_substations_centralized_control_share_pct', 'Share of HV/MV substations with centralized control', '%'),
    ('lv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak LV load', 'kW/kW'),
    ('mv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak MV load', 'kW/kW'),
    ('hv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak HV load', 'kW/kW'),
    ('lv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at LV level', '%'),
    ('mv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at MV level', '%'),
    ('hv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at HV level', '%'),
    ('lv_solar_generation_share_pct', 'Share of solar generation at LV level', '%'),
    ('lv_wind_generation_share_pct', 'Share of wind generation at LV level', '%'),
    ('lv_other_res_generation_share_pct', 'Share of other RES at LV level', '%'),
    ('mv_solar_generation_share_pct', 'Share of solar generation at MV level', '%'),
    ('mv_wind_generation_share_pct', 'Share of wind generation at MV level', '%'),
    ('mv_other_res_generation_share_pct', 'Share of other RES at MV level', '%'),
    ('hv_solar_generation_share_pct', 'Share of solar generation at HV level', '%'),
    ('hv_wind_generation_share_pct', 'Share of wind generation at HV level', '%'),
    ('hv_other_res_generation_share_pct', 'Share of other RES at HV level', '%'),
    ('lv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at LV level', 'kW/point'),
    ('lv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at LV level', 'kW/point'),
    ('lv_other_res_connection_application_avg_capacity_kw', 'Average capacity of other RES connection applications at LV level', 'kW/point'),
    ('mv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at MV level', 'kW/point'),
    ('mv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at MV level', 'kW/point'),
    ('mv_other_res_connection_application_avg_capacity_kw', 'Average capacity of other RES connection applications at MV level', 'kW/point'),
    ('hv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at HV level', 'kW/point'),
    ('hv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at HV level', 'kW/point'),
    ('lv_generation_connection_duration_days', 'Duration of generation connection to LV network', 'days'),
    ('mv_generation_connection_duration_days', 'Duration of generation connection to MV network', 'days'),
    ('hv_generation_connection_duration_days', 'Duration of generation connection to HV network', 'days'),
    ('lv_peak_load_per_consumer_kw', 'Peak LV load per LV consumer', 'kW/consumer'),
    ('mv_peak_load_per_consumer_kw', 'Peak MV load per MV consumer', 'kW/consumer'),
    ('hv_peak_load_per_consumer_kw', 'Peak HV load per HV consumer', 'kW/consumer'),
    ('lv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak LV load', '%'),
    ('mv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak MV load', '%'),
    ('lv_heat_pump_capacity_to_peak_load_pct', 'Share of heat pumps in peak LV load', '%'),
    ('lv_connection_applications_per_1000_consumers', 'LV connection applications per 1,000 LV consumers', 'units'),
    ('lv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded LV connection points per 1,000 LV points', 'units'),
    ('lv_consumer_connection_duration_days', 'Duration of consumer connection to LV network', 'days'),
    ('lv_implemented_consumer_connection_applications_share_pct', 'Share of implemented consumer connection applications at LV level', '%'),
    ('mv_connection_applications_per_1000_consumers', 'MV connection applications per 1,000 MV consumers', 'units'),
    ('mv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded MV connection points per 1,000 MV points', 'units'),
    ('mv_consumer_connection_duration_days', 'Duration of consumer connection to MV network', 'days'),
    ('mv_implemented_consumer_connection_applications_share_pct', 'Share of implemented consumer connection applications at MV level', '%'),
    ('lv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak LV load', 'kW/MW'),
    ('mv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak MV load', 'kW/MW'),
    ('lv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 LV consumers', 'units'),
    ('mv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 MV consumers', 'units'),
    ('lv_bess_application_processing_time_days', 'Processing time of connection requests for consumption devices and storage at LV level', 'days'),
    ('mv_bess_application_processing_time_days', 'Processing time of connection requests for consumption devices and storage at MV level', 'days'),
    ('prosumers_full_market_mode_per_1000_lv_consumers', 'Prosumers in full market mode per 1,000 LV consumers', 'units'),
    ('dynamic_tariff_consumers_per_1000_lv_consumers', 'Consumers with dynamic tariffs per 1,000 LV consumers', 'units'),
    ('prosumer_market_mode_switch_duration_days', 'Time to switch prosumers to full market mode', 'days'),
    ('prosumer_switching_automation_pct', 'Level of automation of prosumer switching', '%')
]
LABELS = {v: (name, unit) for v, name, unit in INDICATOR_METADATA}


def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def add_icps_quartiles(group):
    """Architecture-specific ICPS and quartiles (same rules as Table S2)."""
    group = group.copy()
    x = group[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
    p10, p90 = np.quantile(x, [0.10, 0.90], axis=0, method='linear')
    if np.any(p90 <= p10):
        raise ValueError('P90 must exceed P10 for both continuity components.')
    normalized = (p90 - np.clip(x, p10, p90)) / (p90 - p10)
    shares = normalized / normalized.sum(axis=0)
    terms = np.zeros_like(shares)
    positive = shares > 0
    terms[positive] = shares[positive] * np.log(shares[positive])
    entropy = -terms.sum(axis=0) / np.log(len(group))
    differentiation = np.maximum(1 - entropy, 0)
    weights = differentiation / differentiation.sum()
    group['ICPS'] = normalized @ weights
    p25, p50, p75 = group['ICPS'].quantile([0.25, 0.50, 0.75])
    group['ICPS_quartile'] = np.select(
        [group['ICPS'] <= p25, group['ICPS'] <= p50, group['ICPS'] < p75],
        ['Q1', 'Q2', 'Q3'], default='Q4')
    return group


def eligible(values):
    values = values.dropna()
    return len(values) >= MIN_N and values.nunique() >= 2


def selected_features(group):
    """Features selected as in Tables S4–S5 (BH across all tests of the architecture)."""
    tests = []
    for variable in LABELS:
        if not eligible(group[variable]):
            continue
        for outcome in ['ICPS', 'SAIDI', 'ASIDI']:
            pair = group[[variable, outcome]].dropna()
            rho, p_value = spearmanr(pair[variable], pair[outcome])
            tests.append((variable, outcome, float(rho), float(p_value)))
    tests = pd.DataFrame(tests, columns=['Variable', 'Outcome', 'rho', 'p'])
    tests['q'] = bh_adjust(tests['p'])
    chosen = tests.loc[(tests['q'] < FDR_THRESHOLD) & (tests['rho'].abs() >= MIN_ABS_RHO)]
    return set(chosen['Variable']), len(tests)


def num(value, digits=3):
    text = f'{value:,.{digits}f}'
    if text.lstrip('-').strip('0.,') == '':
        text = text.lstrip('-')
    return text.replace('-', MINUS)


# 1. Load the corrected source (empty cells = missing) and build the analytical sample.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
missing_columns = sorted({'dps_type', *OUTCOME_COLUMNS, *LABELS} - set(source.columns))
if missing_columns:
    raise ValueError(f'Required source columns are missing: {missing_columns}')
source['dps_type'] = (source['dps_type'].str.strip()
                      .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
for column in [*LABELS, *OUTCOME_COLUMNS]:
    source[column] = pd.to_numeric(source[column], errors='raise')
source = source.rename(columns=OUTCOME_COLUMNS)
data = source.loc[source['dps_type'].isin(ARCHITECTURES)
                  & source[['SAIDI', 'ASIDI']].notna().all(axis=1)]
groups = {}
for architecture in ARCHITECTURES:
    group = data.loc[data['dps_type'].eq(architecture)]
    if len(group) != EXPECTED_GROUP_SIZES[architecture]:
        raise ValueError(f'{architecture}: expected {EXPECTED_GROUP_SIZES[architecture]} DSOs, '
                         f'found {len(group)}.')
    groups[architecture] = add_icps_quartiles(group)

# 2. Benchmark features selected within each architecture.
selection, test_counts = {}, {}
for architecture in ARCHITECTURES:
    selection[architecture], test_counts[architecture] = selected_features(groups[architecture])
print(f"Selected features: LV–MV = {len(selection['LV–MV'])} ({test_counts['LV–MV']} tests); "
      f"LV–MV–HV = {len(selection['LV–MV–HV'])} ({test_counts['LV–MV–HV']} tests).")

# 3. Comparable features and best-quartile comparisons.
best = {a: groups[a].loc[groups[a]['ICPS_quartile'].eq('Q4')] for a in ARCHITECTURES}
rows, screening = [], []
for variable in LABELS:
    chosen_in = [a for a in ARCHITECTURES if variable in selection[a]]
    if not chosen_in:
        continue
    common = all(eligible(groups[a][variable]) for a in ARCHITECTURES)
    x = best['LV–MV'][variable].dropna()
    y = best['LV–MV–HV'][variable].dropna()
    x, y = x[x.ne(0)].to_numpy(float), y[y.ne(0)].to_numpy(float)
    comparable = common and len(x) >= MIN_NONZERO_N and len(y) >= MIN_NONZERO_N
    screening.append({'Indicator': LABELS[variable][0], 'Selected in': 'Both' if len(chosen_in) == 2 else chosen_in[0],
                      'Common indicator': common, 'Non-zero n LV–MV': len(x),
                      'Non-zero n LV–MV–HV': len(y), 'Comparable': comparable})
    if not comparable:
        continue
    test = mannwhitneyu(x, y, alternative='two-sided', method='asymptotic', use_continuity=True)
    u_lv_mv = float(test.statistic)
    share_higher_hv = 1 - u_lv_mv / (len(x) * len(y))
    rows.append({'Variable': variable, 'Indicator': LABELS[variable][0], 'Unit': LABELS[variable][1],
                 'Selected in': 'Both' if len(chosen_in) == 2 else chosen_in[0],
                 'n LV–MV': len(x), 'n LV–MV–HV': len(y),
                 'Median LV–MV': float(np.median(x)), 'Median LV–MV–HV': float(np.median(y)),
                 'Difference, %': 100 * (np.median(y) - np.median(x)) / np.median(x),
                 'U (LV–MV)': u_lv_mv, 'P(LV–MV–HV > LV–MV), %': 100 * share_higher_hv,
                 'Rank-biserial r': 2 * share_higher_hv - 1, 'p': float(test.pvalue)})

table_s9_numeric = pd.DataFrame(rows)
table_s9_screening = pd.DataFrame(screening)
table_s9_numeric['q'] = bh_adjust(table_s9_numeric['p'])
table_s9_numeric['Differs (q < 0.10)'] = table_s9_numeric['q'] < FDR_THRESHOLD
table_s9_numeric = table_s9_numeric.sort_values(['q', 'p'], kind='stable').reset_index(drop=True)
print(f'Features selected in at least one architecture: {len(table_s9_screening)}; '
      f'comparable: {len(table_s9_numeric)}; differ: {int(table_s9_numeric["Differs (q < 0.10)"].sum())}.')

# 4. Display table and export.
table_s9 = pd.DataFrame({
    'Indicator': table_s9_numeric['Indicator'],
    'Unit': table_s9_numeric['Unit'],
    'Selected in': table_s9_numeric['Selected in'],
    'n LV–MV': table_s9_numeric['n LV–MV'],
    'n LV–MV–HV': table_s9_numeric['n LV–MV–HV'],
    'Median LV–MV': table_s9_numeric['Median LV–MV'].map(num),
    'Median LV–MV–HV': table_s9_numeric['Median LV–MV–HV'].map(num),
    'Difference, %': table_s9_numeric['Difference, %'].map(lambda v: num(v, 1)),
    'P(LV–MV–HV > LV–MV), %': table_s9_numeric['P(LV–MV–HV > LV–MV), %'].map(lambda v: f'{v:.1f}'),
    'Rank-biserial r': table_s9_numeric['Rank-biserial r'].map(num),
    'p': table_s9_numeric['p'].map(lambda v: '<0.0001' if v < 0.0001 else f'{v:.4f}'),
    'q': table_s9_numeric['q'].map(lambda v: '<0.0001' if v < 0.0001 else f'{v:.4f}'),
    'Differs (q < 0.10)': table_s9_numeric['Differs (q < 0.10)'].map({True: 'TRUE', False: 'FALSE'}),
})
table_s9.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}.csv', sep=';', encoding='utf-8-sig', index=False)
table_s9_numeric.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_numeric.csv', sep=';', decimal=',',
                        encoding='utf-8-sig', index=False)
table_s9_screening.to_csv(OUTPUT_DIRECTORY / f'{TABLE_ID}_screening.csv', sep=';',
                          encoding='utf-8-sig', index=False)
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')

try:
    from IPython.display import display
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.max_colwidth', None):
        display(table_s9)
except ImportError:
    print(table_s9.to_string(index=False))

Selected features: LV–MV = 30 (192 tests); LV–MV–HV = 16 (219 tests).
Features selected in at least one architecture: 38; comparable: 33; differ: 9.
Saved to: ../results/Table_S9.csv


,Indicator,Unit,Selected in,n LV–MV,n LV–MV–HV,Median LV–MV,Median LV–MV–HV,"Difference, %","P(LV–MV–HV > LV–MV), %",Rank-biserial r,p,q,Differs (q < 0.10)
0,"New/upgraded MV connection points per 1,000 MV points",units,LV–MV,51,22,50.000,23.862,−52.3,24.4,−0.512,0.0006,0.0189,TRUE
1,DG capacity relative to peak LV load,kW/kW,LV–MV,164,24,1.526,0.839,−45.0,29.6,−0.409,0.0013,0.0204,TRUE
2,DG capacity utilization factor (CUF) at LV level,%,LV–MV,167,24,6.969,5.920,−15.0,30.7,−0.387,0.0022,0.0204,TRUE
3,Ratio of LV to MV network length,km/km,LV–MV,168,25,2.664,2.123,−20.3,31.2,−0.376,0.0025,0.0204,TRUE
4,Peak LV load per LV consumer,kW/consumer,LV–MV,165,25,0.767,0.672,−12.4,34.3,−0.314,0.0114,0.0755,TRUE
5,"BESS connection applications per 1,000 LV consumers",units,LV–MV–HV,130,17,10.374,7.617,−26.6,32.4,−0.353,0.0183,0.0945,TRUE
6,"LV connection applications per 1,000 LV consumers",units,LV–MV,151,20,14.316,8.916,−37.7,34.4,−0.312,0.0237,0.0945,TRUE
7,Share of solar generation at MV level,%,LV–MV,117,24,80.165,60.626,−24.4,35.7,−0.286,0.0255,0.0945,TRUE
8,Average capacity of solar connection applications at MV level,kW/point,LV–MV–HV,74,19,415.833,"1,033.333",148.5,66.7,0.334,0.0258,0.0945,TRUE
9,GIS coverage of LV network,%,Both,143,23,100.000,100.000,0.0,61.8,0.236,0.0319,0.1054,FALSE


In [10]:
# Supplementary Materials, Table S10: robustness of operator rankings and benchmark values.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

DIRECTORY = Path('../data')
DATA_PATH = DIRECTORY / 'DSO_2024_corrected.csv'
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S10'
MIN_N = 10
FDR_THRESHOLD = 0.10
MIN_ABS_RHO = 0.10
ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
EXPECTED_GROUP_SIZES = {'LV–MV': 669, 'LV–MV–HV': 97}
EXPECTED_BASELINE_FEATURES = {'LV–MV': 30, 'LV–MV–HV': 16}
BASELINE = 'Baseline: capped entropy ICPS'
SCENARIO_NAMES = [BASELINE, 'Equal weights: capped ICPS',
                  'Uncapped entropy ICPS', 'SAIDI only', 'ASIDI only']

# English labels and units (as in Appendix A) for the source CSV variables.
INDICATOR_METADATA = {'lv_network_density_km_per_km2': ('LV network density', 'km/km²'),
 'mv_network_density_km_per_km2': ('MV network density', 'km/km²'),
 'lv_to_mv_network_length_ratio': ('Ratio of LV to MV network length', 'km/km'),
 'network_length_per_consumption_point_m': ('Network length per consumption point', 'm/point'),
 'network_length_per_generation_point_m': ('Network length per generation point', 'm/point'),
 'generation_points_per_1000_consumption_points': ('Generation points per 1,000 consumption points', 'units'),
 'vertical_exchange_intensity_kw_per_kw': ('Vertical exchange intensity', 'kW/kW'),
 'lv_gis_coverage_pct': ('GIS coverage of LV network', '%'),
 'mv_gis_coverage_pct': ('GIS coverage of MV network', '%'),
 'hv_gis_coverage_pct': ('GIS coverage of HV network', '%'),
 'non_remote_maximum_demand_meters_share_pct': ('Share of non-remote maximum-demand meters', '%'),
 'local_substations_remote_transformer_measurements_share_pct': ('Share of local substations with remote measurements at '
                                                                 'the transformer',
                                                                 '%'),
 'local_substations_remote_lv_board_measurements_share_pct': ('Share of local substations with remote measurements in the '
                                                              'LV distribution board',
                                                              '%'),
 'lv_lines_centralized_online_monitoring_share_pct': ('Share of LV lines with centralized online monitoring', '%'),
 'mv_lines_centralized_online_monitoring_share_pct': ('Share of MV lines with centralized online monitoring', '%'),
 'hv_lines_centralized_online_monitoring_share_pct': ('Share of HV lines with centralized online monitoring', '%'),
 'mv_lv_transformers_centralized_online_monitoring_share_pct': ('Share of MV/LV transformers with centralized online '
                                                                'monitoring',
                                                                '%'),
 'hv_mv_transformers_centralized_online_monitoring_share_pct': ('Share of HV/MV transformers with centralized online '
                                                                'monitoring',
                                                                '%'),
 'lv_equipment_centralized_control_share_pct': ('Share of LV network equipment with centralized control', '%'),
 'mv_equipment_centralized_control_share_pct': ('Share of MV network equipment with centralized control', '%'),
 'hv_equipment_centralized_control_share_pct': ('Share of HV network equipment with centralized control', '%'),
 'mv_lv_substations_centralized_control_share_pct': ('Share of MV/LV substations with centralized control', '%'),
 'hv_mv_substations_centralized_control_share_pct': ('Share of HV/MV substations with centralized control', '%'),
 'lv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak LV load', 'kW/kW'),
 'mv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak MV load', 'kW/kW'),
 'hv_dg_capacity_to_peak_load_kw_per_kw': ('DG capacity relative to peak HV load', 'kW/kW'),
 'lv_dg_capacity_utilization_factor_pct': ('DG capacity utilization factor (CUF) at LV level', '%'),
 'mv_dg_capacity_utilization_factor_pct': ('DG capacity utilization factor (CUF) at MV level', '%'),
 'hv_dg_capacity_utilization_factor_pct': ('DG capacity utilization factor (CUF) at HV level', '%'),
 'lv_solar_generation_share_pct': ('Share of solar generation at LV level', '%'),
 'lv_wind_generation_share_pct': ('Share of wind generation at LV level', '%'),
 'lv_other_res_generation_share_pct': ('Share of other RES at LV level', '%'),
 'mv_solar_generation_share_pct': ('Share of solar generation at MV level', '%'),
 'mv_wind_generation_share_pct': ('Share of wind generation at MV level', '%'),
 'mv_other_res_generation_share_pct': ('Share of other RES at MV level', '%'),
 'hv_solar_generation_share_pct': ('Share of solar generation at HV level', '%'),
 'hv_wind_generation_share_pct': ('Share of wind generation at HV level', '%'),
 'hv_other_res_generation_share_pct': ('Share of other RES at HV level', '%'),
 'lv_solar_connection_application_avg_capacity_kw': ('Average capacity of solar connection applications at LV level',
                                                     'kW/point'),
 'lv_wind_connection_application_avg_capacity_kw': ('Average capacity of wind connection applications at LV level',
                                                    'kW/point'),
 'lv_other_res_connection_application_avg_capacity_kw': ('Average capacity of other RES connection applications at LV level',
                                                         'kW/point'),
 'mv_solar_connection_application_avg_capacity_kw': ('Average capacity of solar connection applications at MV level',
                                                     'kW/point'),
 'mv_wind_connection_application_avg_capacity_kw': ('Average capacity of wind connection applications at MV level',
                                                    'kW/point'),
 'mv_other_res_connection_application_avg_capacity_kw': ('Average capacity of other RES connection applications at MV level',
                                                         'kW/point'),
 'hv_solar_connection_application_avg_capacity_kw': ('Average capacity of solar connection applications at HV level',
                                                     'kW/point'),
 'hv_wind_connection_application_avg_capacity_kw': ('Average capacity of wind connection applications at HV level',
                                                    'kW/point'),
 'lv_generation_connection_duration_days': ('Duration of generation connection to LV network', 'days'),
 'mv_generation_connection_duration_days': ('Duration of generation connection to MV network', 'days'),
 'hv_generation_connection_duration_days': ('Duration of generation connection to HV network', 'days'),
 'lv_peak_load_per_consumer_kw': ('Peak LV load per LV consumer', 'kW/consumer'),
 'mv_peak_load_per_consumer_kw': ('Peak MV load per MV consumer', 'kW/consumer'),
 'hv_peak_load_per_consumer_kw': ('Peak HV load per HV consumer', 'kW/consumer'),
 'lv_ev_charging_capacity_to_peak_load_pct': ('Share of EV charging stations in peak LV load', '%'),
 'mv_ev_charging_capacity_to_peak_load_pct': ('Share of EV charging stations in peak MV load', '%'),
 'lv_heat_pump_capacity_to_peak_load_pct': ('Share of heat pumps in peak LV load', '%'),
 'lv_connection_applications_per_1000_consumers': ('LV connection applications per 1,000 LV consumers', 'units'),
 'lv_new_or_upgraded_points_per_1000_existing_points': ('New/upgraded LV connection points per 1,000 LV points', 'units'),
 'lv_consumer_connection_duration_days': ('Duration of consumer connection to LV network', 'days'),
 'lv_implemented_consumer_connection_applications_share_pct': ('Share of implemented consumer connection applications at LV '
                                                               'level',
                                                               '%'),
 'mv_connection_applications_per_1000_consumers': ('MV connection applications per 1,000 MV consumers', 'units'),
 'mv_new_or_upgraded_points_per_1000_existing_points': ('New/upgraded MV connection points per 1,000 MV points', 'units'),
 'mv_consumer_connection_duration_days': ('Duration of consumer connection to MV network', 'days'),
 'mv_implemented_consumer_connection_applications_share_pct': ('Share of implemented consumer connection applications at MV '
                                                               'level',
                                                               '%'),
 'lv_bess_capacity_to_peak_load_kw_per_mw': ('BESS capacity relative to peak LV load', 'kW/MW'),
 'mv_bess_capacity_to_peak_load_kw_per_mw': ('BESS capacity relative to peak MV load', 'kW/MW'),
 'lv_bess_connection_applications_per_1000_consumers': ('BESS connection applications per 1,000 LV consumers', 'units'),
 'mv_bess_connection_applications_per_1000_consumers': ('BESS connection applications per 1,000 MV consumers', 'units'),
 'lv_bess_application_processing_time_days': ('Processing time of connection requests for consumption devices and storage '
                                              'at LV level',
                                              'days'),
 'mv_bess_application_processing_time_days': ('Processing time of connection requests for consumption devices and storage '
                                              'at MV level',
                                              'days'),
 'prosumers_full_market_mode_per_1000_lv_consumers': ('Prosumers in full market mode per 1,000 LV consumers', 'units'),
 'prosumer_market_mode_switch_duration_days': ('Time to switch prosumers to full market mode', 'days'),
 'prosumer_switching_automation_pct': ('Level of automation of prosumer switching', '%'),
 'dynamic_tariff_consumers_per_1000_lv_consumers': ('Consumers with dynamic tariffs per 1,000 LV consumers', 'units')}


def normalize_architecture(series):
    return (series.astype('string').str.strip()
            .str.replace(r'\s*[-−—–]\s*', '–', regex=True))


def normalize_label(value):
    text = str(value).replace('−', '-').replace('—', '–')
    text = text.replace('*', '').replace('¹', '').replace('\u00a0', ' ')
    return ' '.join(text.split()).casefold()


def composite_score(raw, capped, equal_weights):
    """Fit normalization and weights within one architecture and scenario."""
    if capped:
        lower, upper = np.quantile(raw, [0.10, 0.90], axis=0, method='linear')
        prepared = np.clip(raw, lower, upper)
        normalization = 'P10/P90-capped inverse Min–Max'
    else:
        lower, upper = raw.min(axis=0), raw.max(axis=0)
        prepared = raw
        normalization = 'Uncapped inverse Min–Max using observed minimum and maximum'
    if np.any(upper <= lower):
        raise ValueError('Normalization bounds must have positive ranges.')
    z = (upper - prepared) / (upper - lower)
    if not np.isfinite(z).all() or (z < -1e-12).any() or (z > 1 + 1e-12).any():
        raise ValueError('Invalid normalized continuity scores.')
    if equal_weights:
        weights = np.array([0.5, 0.5])
        weighting = 'Equal weights'
    else:
        shares = z / z.sum(axis=0)
        terms = np.zeros_like(shares)
        positive = shares > 0
        terms[positive] = shares[positive] * np.log(shares[positive])
        differentiation = np.maximum(1 + terms.sum(axis=0) / np.log(len(raw)), 0)
        if differentiation.sum() <= np.finfo(float).eps:
            raise ValueError('Entropy weights cannot be determined.')
        weights = differentiation / differentiation.sum()
        weighting = 'Entropy weights fitted within the scenario'
    return z @ weights, {
        'Normalization': normalization, 'Weighting': weighting,
        'SAIDI weight': float(weights[0]), 'ASIDI weight': float(weights[1]),
        'SAIDI lower bound': float(lower[0]), 'SAIDI upper bound': float(upper[0]),
        'ASIDI lower bound': float(lower[1]), 'ASIDI upper bound': float(upper[1])
    }


def scenario_scores(raw):
    specifications = []
    for scenario, capped, equal in [(SCENARIO_NAMES[0], True, False),
                                    (SCENARIO_NAMES[1], True, True),
                                    (SCENARIO_NAMES[2], False, False)]:
        score, parameters = composite_score(raw, capped, equal)
        specifications.append((scenario, score, parameters))
    for scenario, column in [(SCENARIO_NAMES[3], 0), (SCENARIO_NAMES[4], 1)]:
        parameters = {
            'Normalization': 'Raw duration with reversed direction; ranking only',
            'Weighting': 'Single continuity component',
            'SAIDI weight': float(column == 0), 'ASIDI weight': float(column == 1),
            'SAIDI lower bound': np.nan, 'SAIDI upper bound': np.nan,
            'ASIDI lower bound': np.nan, 'ASIDI upper bound': np.nan
        }
        specifications.append((scenario, -raw[:, column], parameters))
    return specifications


def percentile_groups(scores):
    """Retain all ties at P25/P75; never split tied operators by ID."""
    scores = np.asarray(scores, dtype=float)
    if not np.isfinite(scores).all() or np.unique(scores).size < 2:
        raise ValueError('A ranking scenario must contain finite, varying scores.')
    p25, p50, p75 = np.quantile(scores, [0.25, 0.50, 0.75], method='linear')
    return scores >= p75, scores <= p25, (float(p25), float(p50), float(p75))


def nonzero_median(values):
    observed = values.dropna()
    nonzero = observed.loc[observed.ne(0)]
    return (float(nonzero.median()) if len(nonzero) else np.nan,
            len(nonzero), len(observed), int(observed.eq(0).sum()))


def change_statistics(value, reference):
    if not np.isfinite(value) or not np.isfinite(reference):
        return np.nan, np.nan, None
    delta = float(value - reference)
    relative = 100 * delta / abs(reference) if reference != 0 else np.nan
    return delta, relative, bool(np.isclose(value, reference, rtol=1e-12, atol=1e-12))


def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


def select_baseline_features(group, baseline_score):
    """Baseline feature selection as in Tables S4–S5 (baseline ICPS, SAIDI, ASIDI)."""
    outcomes = {'ICPS': baseline_score,
                'SAIDI': group['saidi_lv_min_per_year'].to_numpy(dtype=float),
                'ASIDI': group['asidi_mv_min_per_year'].to_numpy(dtype=float)}
    tests = []
    for variable in [c for c in group.columns if c in INDICATOR_METADATA]:
        observed = group[variable].dropna()
        if len(observed) < MIN_N or observed.nunique() < 2:
            continue
        values = group[variable].to_numpy(dtype=float)
        mask = ~np.isnan(values)
        for outcome, y in outcomes.items():
            rho, p_value = spearmanr(values[mask], y[mask])
            tests.append((variable, outcome, float(rho), float(p_value)))
    tests = pd.DataFrame(tests, columns=['Variable', 'Outcome', 'rho', 'p'])
    tests['q'] = bh_adjust(tests['p'])
    return tests.loc[(tests['q'] < FDR_THRESHOLD) & (tests['rho'].abs() >= MIN_ABS_RHO)]


# 1. Load the corrected individual DSO data.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
outcome_columns = ['saidi_lv_min_per_year', 'asidi_mv_min_per_year']
required = {'dso_id', 'dso_name', 'dps_type', *outcome_columns}
if not required.issubset(source.columns):
    raise ValueError(f'Missing source columns: {sorted(required - set(source.columns))}')
source['dso_id'] = source['dso_id'].str.strip()
source['dps_type'] = normalize_architecture(source['dps_type'])
if source['dso_id'].isna().any() or source['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')
for column in outcome_columns:
    source[column] = pd.to_numeric(source[column], errors='raise')
    if not np.isfinite(source[column].dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{column}: infinite outcome values are not supported.')
if (source[outcome_columns] < 0).any().any():
    raise ValueError('Interruption durations must be non-negative.')
complete = source['dps_type'].isin(ARCHITECTURES) & source[outcome_columns].notna().all(axis=1)
table_s10_excluded = source.loc[~complete, ['dso_id', 'dso_name', 'dps_type', *outcome_columns]].copy()
data = source.loc[complete].copy()

# 2. Baseline feature membership, selected once from the baseline ICPS and then frozen.
for variable in INDICATOR_METADATA:
    if variable not in source.columns:
        raise ValueError(f'Missing source column: {variable}')
    source[variable] = pd.to_numeric(source[variable], errors='raise')
data = source.loc[complete].copy()
baseline_features, feature_rows = {}, []
for architecture in ARCHITECTURES:
    group = data.loc[data['dps_type'].eq(architecture)].copy()
    baseline_score = scenario_scores(group[outcome_columns].to_numpy(dtype=float))[0][1]
    selected_group = select_baseline_features(group, baseline_score)
    features = [column for column in source.columns if column in set(selected_group['Variable'])]
    if len(features) != EXPECTED_BASELINE_FEATURES[architecture]:
        raise ValueError(
            f'{architecture}: expected {EXPECTED_BASELINE_FEATURES[architecture]} baseline features, '
            f'found {len(features)}. Verify the corrected data.')
    baseline_features[architecture] = features
    for variable in features:
        label, unit = INDICATOR_METADATA[variable]
        outcomes = selected_group.loc[selected_group['Variable'].eq(variable), 'Outcome'].tolist()
        feature_rows.append({'DPS type': architecture, 'Variable': variable, 'Indicator': label,
                             'Unit': unit, 'Baseline selected outcomes': ', '.join(outcomes)})
table_s10_baseline_features = pd.DataFrame(feature_rows)

# 3. Compare rankings and non-zero benchmark medians on the same fixed samples.
ranking_rows, parameter_rows, benchmark_rows, operator_rows = [], [], [], []
for architecture in ARCHITECTURES:
    group = data.loc[data['dps_type'].eq(architecture)].copy()
    if len(group) != EXPECTED_GROUP_SIZES[architecture]:
        raise ValueError(f'{architecture}: unexpected sample size; use the corrected source.')
    features = baseline_features[architecture]
    for variable in features:
        group[variable] = pd.to_numeric(group[variable], errors='raise')
        if not np.isfinite(group[variable].dropna().to_numpy(dtype=float)).all():
            raise ValueError(f'{variable}: infinite benchmark values are not supported.')
    raw = group[outcome_columns].to_numpy(dtype=float)
    specifications = scenario_scores(raw)
    baseline_score = specifications[0][1]
    baseline_best, baseline_worst, _ = percentile_groups(baseline_score)
    baseline_ranks = pd.Series(baseline_score).rank(ascending=False, method='average').to_numpy()
    reference_medians = {variable: {
        'Best': nonzero_median(group.loc[baseline_best, variable])[0],
        'All': nonzero_median(group[variable])[0],
        'Worst': nonzero_median(group.loc[baseline_worst, variable])[0]
    } for variable in features}
    for scenario, score, parameters in specifications:
        best, worst, (p25, p50, p75) = percentile_groups(score)
        ranks = pd.Series(score).rank(ascending=False, method='average').to_numpy()
        rho = float(spearmanr(baseline_ranks, ranks).statistic)
        intersection = int((baseline_best & best).sum())
        union = int((baseline_best | best).sum())
        worst_intersection = int((baseline_worst & worst).sum())
        parameter_rows.append({
            'DPS type': architecture, 'Scenario': scenario, 'N': len(group), **parameters,
            'Score P25': p25, 'Score P50': p50, 'Score P75': p75,
            'Best n': int(best.sum()), 'Worst n': int(worst.sum()),
            'Score ties at best cutoff': int(np.count_nonzero(score == p75)),
            'Best/worst overlap n': int((best & worst).sum())
        })
        scenario_benchmarks = []
        for variable in features:
            label, unit = INDICATOR_METADATA[variable]
            row = {'DPS type': architecture, 'Scenario': scenario, 'Variable': variable,
                   'Indicator': label, 'Unit': unit, 'N': len(group)}
            samples = {'Best': group.loc[best, variable], 'All': group[variable],
                       'Worst': group.loc[worst, variable]}
            for name, values in samples.items():
                median, nonzero_n, observed_n, zero_n = nonzero_median(values)
                reference = reference_medians[variable][name]
                delta, percent, unchanged = change_statistics(median, reference)
                row.update({
                    f'{name} median': median, f'Baseline {name.lower()} median': reference,
                    f'{name} median change': delta, f'{name} median change (%)': percent,
                    f'{name} median unchanged': unchanged,
                    f'{name} nonzero n': nonzero_n, f'{name} observed n': observed_n,
                    f'{name} zero n': zero_n
                })
            benchmark_rows.append(row)
            scenario_benchmarks.append(row)
        bench = pd.DataFrame(scenario_benchmarks)
        best_percent = bench['Best median change (%)'].dropna().abs()
        worst_percent = bench['Worst median change (%)'].dropna().abs()
        comparable_best = bench['Best median unchanged'].notna()
        comparable_worst = bench['Worst median unchanged'].notna()
        ranking_rows.append({
            'DPS type': architecture, 'Scenario': scenario, 'N': len(group),
            'Spearman rank rho': rho,
            'Baseline best n': int(baseline_best.sum()), 'Scenario best n': int(best.sum()),
            'Scenario best share (%)': 100 * float(best.mean()),
            'Best intersection n': intersection,
            'Baseline best retained (%)': 100 * intersection / int(baseline_best.sum()),
            'Scenario best shared with baseline (%)': 100 * intersection / int(best.sum()),
            'Best Jaccard (%)': 100 * intersection / union,
            'Left baseline best n': int((baseline_best & ~best).sum()),
            'Entered scenario best n': int((~baseline_best & best).sum()),
            'Baseline worst n': int(baseline_worst.sum()), 'Scenario worst n': int(worst.sum()),
            'Worst intersection n': worst_intersection,
            'Baseline worst retained (%)': 100 * worst_intersection / int(baseline_worst.sum()),
            'Benchmark features': len(features),
            'Comparable best benchmarks': int(comparable_best.sum()),
            'Best medians unchanged': int(bench.loc[comparable_best, 'Best median unchanged'].astype(bool).sum()),
            'Best medians changed': int((~bench.loc[comparable_best, 'Best median unchanged'].astype(bool)).sum()),
            'Comparable worst benchmarks': int(comparable_worst.sum()),
            'Worst medians unchanged': int(bench.loc[comparable_worst, 'Worst median unchanged'].astype(bool).sum()),
            'Worst medians changed': int((~bench.loc[comparable_worst, 'Worst median unchanged'].astype(bool)).sum()),
            'Best relative comparisons': len(best_percent), 'Worst relative comparisons': len(worst_percent),
            'Median abs. best benchmark change (%)': float(best_percent.median()) if len(best_percent) else np.nan,
            'Max abs. best benchmark change (%)': float(best_percent.max()) if len(best_percent) else np.nan,
            'Median abs. worst benchmark change (%)': float(worst_percent.median()) if len(worst_percent) else np.nan,
            'Max abs. worst benchmark change (%)': float(worst_percent.max()) if len(worst_percent) else np.nan,
            'Median absolute rank change': float(np.median(np.abs(ranks - baseline_ranks))),
            'Max absolute rank change': float(np.max(np.abs(ranks - baseline_ranks)))
        })
        membership = np.select(
            [baseline_best & best, baseline_best & ~best, ~baseline_best & best],
            ['Retained in best group', 'Left best group', 'Entered best group'],
            default='Outside both best groups')
        for i, (dso_id, dso_name) in enumerate(zip(group['dso_id'], group['dso_name'])):
            operator_rows.append({
                'dso_id': dso_id, 'dso_name': dso_name, 'DPS type': architecture,
                'Scenario': scenario, 'SAIDI': float(raw[i, 0]), 'ASIDI': float(raw[i, 1]),
                'Ranking score (higher is better)': float(score[i]),
                'Rank (1 = best)': float(ranks[i]), 'Baseline rank': float(baseline_ranks[i]),
                'Rank change (scenario minus baseline)': float(ranks[i] - baseline_ranks[i]),
                'Baseline best group': bool(baseline_best[i]), 'Scenario best group': bool(best[i]),
                'Baseline worst group': bool(baseline_worst[i]), 'Scenario worst group': bool(worst[i]),
                'Best membership change': membership[i]
            })

table_s10_rankings = pd.DataFrame(ranking_rows)
table_s10_parameters = pd.DataFrame(parameter_rows)
table_s10_benchmarks = pd.DataFrame(benchmark_rows)
table_s10_operator_ranks = pd.DataFrame(operator_rows)

# 4. Format a concise main table; full worst-group details remain in table_s10_rankings.
table_s10 = table_s10_rankings[
    ['DPS type', 'Scenario', 'N', 'Spearman rank rho', 'Baseline best n', 'Scenario best n',
     'Best intersection n', 'Baseline best retained (%)', 'Best Jaccard (%)',
     'Benchmark features', 'Best medians changed',
     'Median abs. best benchmark change (%)', 'Max abs. best benchmark change (%)']
].copy()
table_s10['Spearman rank rho'] = table_s10['Spearman rank rho'].map(lambda value: f'{value:.4f}')
for column in ['Baseline best retained (%)', 'Best Jaccard (%)',
               'Median abs. best benchmark change (%)', 'Max abs. best benchmark change (%)']:
    table_s10[column] = table_s10[column].map(
        lambda value: f'{value:.2f}' if np.isfinite(value) else 'NA')

# 5. Export the summary and supporting records.
table_s10.to_csv(OUTPUT_DIRECTORY / 'Table_S10.csv', sep=';', encoding='utf-8-sig', index=False)
for frame, filename in [
    (table_s10_rankings, 'Table_S10_rankings.csv'),
    (table_s10_benchmarks, 'Table_S10_benchmarks.csv'),
    (table_s10_operator_ranks, 'Table_S10_operator_ranks.csv'),
    (table_s10_parameters, 'Table_S10_parameters.csv'),
    (table_s10_baseline_features, 'Table_S10_baseline_features.csv')
]:
    frame.to_csv(OUTPUT_DIRECTORY / filename, sep=';', decimal=',', encoding='utf-8-sig', index=False)

print('Completed Table S10: two architectures, five ranking scenarios, fixed baseline features.')
print('Baseline sample sizes: ' + '; '.join(f'{a} = {int(data["dps_type"].eq(a).sum())}' for a in ARCHITECTURES) + '.')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
print('Boundary ties are retained; actual best-group sizes are reported.')
try:
    from IPython.display import display
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
        display(table_s10)
except ImportError:
    print(table_s10.to_string(index=False))

Completed Table S10: two architectures, five ranking scenarios, fixed baseline features.
Baseline sample sizes: LV–MV = 669; LV–MV–HV = 97.
Saved to: ../results/Table_S10.csv
Boundary ties are retained; actual best-group sizes are reported.


,DPS type,Scenario,N,Spearman rank rho,Baseline best n,Scenario best n,Best intersection n,Baseline best retained (%),Best Jaccard (%),Benchmark features,Best medians changed,Median abs. best benchmark change (%),Max abs. best benchmark change (%)
0,LV–MV,Baseline: capped entropy ICPS,669,1.0000,168,168,168,100.00,100.00,30,0,0.00,0.00
1,LV–MV,Equal weights: capped ICPS,669,0.9984,168,168,166,98.81,97.65,30,11,0.00,13.96
2,LV–MV,Uncapped entropy ICPS,669,0.8727,168,168,132,78.57,64.71,30,22,2.95,136.14
3,LV–MV,SAIDI only,669,0.7954,168,169,123,73.21,57.48,30,21,4.44,200.09
4,LV–MV,ASIDI only,669,0.6841,168,259,133,79.17,45.24,30,20,3.03,313.10
5,LV–MV–HV,Baseline: capped entropy ICPS,97,1.0000,25,25,25,100.00,100.00,16,0,0.00,0.00
6,LV–MV–HV,Equal weights: capped ICPS,97,0.9999,25,25,25,100.00,100.00,16,0,0.00,0.00
7,LV–MV–HV,Uncapped entropy ICPS,97,0.8103,25,25,17,68.00,51.52,16,11,4.52,126.37
8,LV–MV–HV,SAIDI only,97,0.7447,25,25,16,64.00,47.06,16,9,13.18,84.11
9,LV–MV–HV,ASIDI only,97,0.7262,25,25,14,56.00,38.89,16,12,15.55,126.37


In [11]:
#Supplementary Materials, Table S11: feature selection under alternative ICPS specifications.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm

DIRECTORY = Path('../data')
DATA_PATH = DIRECTORY / 'DSO_2024_corrected.csv'
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S11'
TABLE_S3_PATH = DIRECTORY / 'Table_S3.csv'
ARCHITECTURES = ['LV–MV', 'LV–MV–HV']
OUTCOMES = ['ICPS', 'SAIDI', 'ASIDI']
MIN_N = 10
FDR_THRESHOLD = 0.10
MIN_ABS_RHO = 0.10
RETENTION_P_THRESHOLD = 0.10
EXPECTED_GROUP_SIZES = {'LV–MV': 669, 'LV–MV–HV': 97}
EXPECTED_TEST_COUNTS = {'LV–MV': 192, 'LV–MV–HV': 219}
EXPECTED_BASELINE_FEATURES = {'LV–MV': 30, 'LV–MV–HV': 16}
BASELINE = 'Baseline: capped entropy ICPS'
SPECIFICATIONS = [(BASELINE, True, False),
                  ('Equal weights: capped ICPS', True, True),
                  ('Uncapped entropy ICPS', False, False)]

INDICATOR_METADATA = [('lv_network_density_km_per_km2', 'LV network density'),
 ('mv_network_density_km_per_km2', 'MV network density'),
 ('lv_to_mv_network_length_ratio', 'Ratio of LV to MV network length'),
 ('network_length_per_consumption_point_m', 'Network length per consumption point'),
 ('network_length_per_generation_point_m', 'Network length per generation point'),
 ('generation_points_per_1000_consumption_points', 'Generation points per 1,000 consumption points'),
 ('vertical_exchange_intensity_kw_per_kw', 'Vertical exchange intensity'),
 ('lv_gis_coverage_pct', 'GIS coverage of LV network'),
 ('mv_gis_coverage_pct', 'GIS coverage of MV network'),
 ('hv_gis_coverage_pct', 'GIS coverage of HV network'),
 ('non_remote_maximum_demand_meters_share_pct', 'Share of non-remote maximum-demand meters'),
 ('local_substations_remote_transformer_measurements_share_pct',
  'Share of local substations with remote measurements at the transformer'),
 ('local_substations_remote_lv_board_measurements_share_pct',
  'Share of local substations with remote measurements in the LV distribution board'),
 ('lv_lines_centralized_online_monitoring_share_pct', 'Share of LV lines with centralized online monitoring'),
 ('mv_lines_centralized_online_monitoring_share_pct', 'Share of MV lines with centralized online monitoring'),
 ('hv_lines_centralized_online_monitoring_share_pct', 'Share of HV lines with centralized online monitoring'),
 ('mv_lv_transformers_centralized_online_monitoring_share_pct',
  'Share of MV/LV transformers with centralized online monitoring'),
 ('hv_mv_transformers_centralized_online_monitoring_share_pct',
  'Share of HV/MV transformers with centralized online monitoring'),
 ('lv_equipment_centralized_control_share_pct', 'Share of LV network equipment with centralized control'),
 ('mv_equipment_centralized_control_share_pct', 'Share of MV network equipment with centralized control'),
 ('hv_equipment_centralized_control_share_pct', 'Share of HV network equipment with centralized control'),
 ('mv_lv_substations_centralized_control_share_pct', 'Share of MV/LV substations with centralized control'),
 ('hv_mv_substations_centralized_control_share_pct', 'Share of HV/MV substations with centralized control'),
 ('lv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak LV load'),
 ('mv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak MV load'),
 ('hv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak HV load'),
 ('lv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at LV level'),
 ('mv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at MV level'),
 ('hv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at HV level'),
 ('lv_solar_generation_share_pct', 'Share of solar generation at LV level'),
 ('lv_wind_generation_share_pct', 'Share of wind generation at LV level'),
 ('lv_other_res_generation_share_pct', 'Share of other RES at LV level'),
 ('mv_solar_generation_share_pct', 'Share of solar generation at MV level'),
 ('mv_wind_generation_share_pct', 'Share of wind generation at MV level'),
 ('mv_other_res_generation_share_pct', 'Share of other RES at MV level'),
 ('hv_solar_generation_share_pct', 'Share of solar generation at HV level'),
 ('hv_wind_generation_share_pct', 'Share of wind generation at HV level'),
 ('hv_other_res_generation_share_pct', 'Share of other RES at HV level'),
 ('lv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at LV level'),
 ('lv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at LV level'),
 ('lv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at LV level'),
 ('mv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at MV level'),
 ('mv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at MV level'),
 ('mv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at MV level'),
 ('hv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at HV level'),
 ('hv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at HV level'),
 ('lv_generation_connection_duration_days', 'Duration of generation connection to LV network'),
 ('mv_generation_connection_duration_days', 'Duration of generation connection to MV network'),
 ('hv_generation_connection_duration_days', 'Duration of generation connection to HV network'),
 ('lv_peak_load_per_consumer_kw', 'Peak LV load per LV consumer'),
 ('mv_peak_load_per_consumer_kw', 'Peak MV load per MV consumer'),
 ('hv_peak_load_per_consumer_kw', 'Peak HV load per HV consumer'),
 ('lv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak LV load'),
 ('mv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak MV load'),
 ('lv_heat_pump_capacity_to_peak_load_pct', 'Share of heat pumps in peak LV load'),
 ('lv_connection_applications_per_1000_consumers', 'LV connection applications per 1,000 LV consumers'),
 ('lv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded LV connection points per 1,000 LV points'),
 ('lv_consumer_connection_duration_days', 'Duration of consumer connection to LV network'),
 ('lv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at LV level'),
 ('mv_connection_applications_per_1000_consumers', 'MV connection applications per 1,000 MV consumers'),
 ('mv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded MV connection points per 1,000 MV points'),
 ('mv_consumer_connection_duration_days', 'Duration of consumer connection to MV network'),
 ('mv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at MV level'),
 ('lv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak LV load'),
 ('mv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak MV load'),
 ('lv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 LV consumers'),
 ('mv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 MV consumers'),
 ('lv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at LV level'),
 ('mv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at MV level'),
 ('prosumers_full_market_mode_per_1000_lv_consumers', 'Prosumers in full market mode per 1,000 LV consumers'),
 ('prosumer_market_mode_switch_duration_days', 'Time to switch prosumers to full market mode'),
 ('prosumer_switching_automation_pct', 'Level of automation of prosumer switching'),
 ('dynamic_tariff_consumers_per_1000_lv_consumers', 'Consumers with dynamic tariffs per 1,000 LV consumers')]

def bh_adjust(p_values):
    """Benjamini–Hochberg adjustment over one complete architecture test family."""
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted

def spearman_ci(rho, n):
    """Pointwise approximate 95% Fisher-z CI with Bonett–Wright variance.

    The CI is not adjusted for multiple testing; the q-value is BH-adjusted.
    """
    if n <= 3 or not np.isfinite(rho) or abs(rho) > 1:
        raise ValueError('The Spearman CI requires n > 3 and a valid rho.')
    if np.isclose(abs(rho), 1.0, rtol=0, atol=1e-14):
        return float(rho), float(rho)
    z = np.arctanh(rho)
    standard_error = np.sqrt((1 + rho**2 / 2) / (n - 3))
    bounds = np.tanh(z + np.array([-1, 1]) * norm.ppf(0.975) * standard_error)
    return float(bounds[0]), float(bounds[1])

def composite_score(raw, capped, equal_weights):
    """Fit normalization and weights within one architecture and scenario."""
    if capped:
        lower, upper = np.quantile(raw, [0.10, 0.90], axis=0, method='linear')
        prepared = np.clip(raw, lower, upper)
        normalization = 'P10/P90-capped inverse Min–Max'
    else:
        lower, upper = raw.min(axis=0), raw.max(axis=0)
        prepared = raw
        normalization = 'Uncapped inverse Min–Max using observed minimum and maximum'
    if np.any(upper <= lower):
        raise ValueError('Normalization bounds must have positive ranges.')
    z = (upper - prepared) / (upper - lower)
    if not np.isfinite(z).all() or (z < -1e-12).any() or (z > 1 + 1e-12).any():
        raise ValueError('Invalid normalized continuity scores.')
    if equal_weights:
        weights = np.array([0.5, 0.5])
        weighting = 'Equal weights'
    else:
        shares = z / z.sum(axis=0)
        terms = np.zeros_like(shares)
        positive = shares > 0
        terms[positive] = shares[positive] * np.log(shares[positive])
        differentiation = np.maximum(1 + terms.sum(axis=0) / np.log(len(raw)), 0)
        if differentiation.sum() <= np.finfo(float).eps:
            raise ValueError('Entropy weights cannot be determined.')
        weights = differentiation / differentiation.sum()
        weighting = 'Entropy weights fitted within the scenario'
    return z @ weights, {
        'Normalization': normalization, 'Weighting': weighting,
        'SAIDI weight': float(weights[0]), 'ASIDI weight': float(weights[1]),
        'SAIDI lower bound': float(lower[0]), 'SAIDI upper bound': float(upper[0]),
        'ASIDI lower bound': float(lower[1]), 'ASIDI upper bound': float(upper[1])
    }

# 1. Load the corrected source. Keep genuine zeros and preserve missing values.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
source_outcomes = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}
indicator_columns = [variable for variable, _ in INDICATOR_METADATA]
required = {'dso_id', 'dso_name', 'dps_type', *source_outcomes, *indicator_columns}
if not required.issubset(source.columns):
    raise ValueError(f'Missing source columns: {sorted(required - set(source.columns))}')
source['dso_id'] = source['dso_id'].str.strip()
source['dps_type'] = (source['dps_type'].astype('string').str.strip()
                      .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
if source['dso_id'].isna().any() or source['dso_id'].eq('').any() or source['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')
for column in [*indicator_columns, *source_outcomes]:
    source[column] = pd.to_numeric(source[column], errors='raise')
    if not np.isfinite(source[column].dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{column}: infinite values are not supported.')
source = source.rename(columns=source_outcomes)
if (source[['SAIDI', 'ASIDI']] < 0).any().any():
    raise ValueError('Interruption durations must be non-negative.')
valid = source['dps_type'].isin(ARCHITECTURES) & source[['SAIDI', 'ASIDI']].notna().all(axis=1)
table_s11_excluded = source.loc[~valid, ['dso_id', 'dso_name', 'dps_type', 'SAIDI', 'ASIDI']].copy()
data = source.loc[valid].copy()

# 2. Optional: read existing Table S3 flags to verify the recalculated baseline.
# Formatted p/q values are not used for threshold decisions or calculations.
def normalize_label(text):
    text = str(text).replace('−', '-').replace('*', '').replace('†', '').replace('\u00a0', ' ')
    text = ' '.join(text.split()).casefold()
    return text.replace('dg cuf at', 'dg capacity utilization factor (cuf) at')

keys = ['DPS type', 'Variable', 'Outcome']
reference_c = None
if TABLE_S3_PATH.is_file():
    reference_c = pd.read_csv(TABLE_S3_PATH, sep=';', encoding='utf-8-sig', dtype='string')
    required_c = {'DPS type', 'Indicator', 'Outcome', 'Selection'}
    if not required_c.issubset(reference_c.columns):
        raise ValueError(f'Missing Table S3 columns: {sorted(required_c - set(reference_c.columns))}')
    reference_c['DPS type'] = (reference_c['DPS type'].str.strip()
                               .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
    reference_c = reference_c.loc[reference_c['DPS type'].isin(ARCHITECTURES)].copy()
    flags = reference_c['Selection'].str.strip().str.casefold()
    if flags.isna().any() or not flags.isin(['true', 'false', '1', '0']).all():
        raise ValueError('Table S3 Selection must contain TRUE/FALSE values.')
    reference_c['Selection'] = flags.map({'true': True, 'false': False, '1': True, '0': False}).astype(bool)
    lookup = {}
    for variable, label in INDICATOR_METADATA:
        lookup[normalize_label(label)] = variable
        lookup[normalize_label(variable)] = variable
    def to_variable(label):
        key = normalize_label(label)
        if key in lookup:
            return lookup[key]
        base = key.rsplit(', ', 1)[0]          # drop a trailing unit, e.g. ', km/km'
        return lookup.get(base)
    if 'Variable' not in reference_c.columns:
        reference_c['Variable'] = reference_c['Indicator'].map(to_variable)
    else:
        reference_c['Variable'] = reference_c['Variable'].str.strip()
    if reference_c['Variable'].isna().any() or reference_c['Outcome'].isna().any():
        unknown = reference_c.loc[reference_c['Variable'].isna(), 'Indicator'].unique()[:5]
        raise ValueError(f'Table S3 contains unknown indicators, e.g. {list(unknown)}.')
    if reference_c.duplicated(keys).any():
        raise ValueError('Table S3 contains duplicate indicator-outcome pairs.')

# 3. Recompute all eligible tests under each ICPS specification.
# The same predictor family and pairwise observation masks apply in all scenarios.
# BH is applied separately to the complete architecture x specification family.
correlation_frames, eligibility_rows, parameter_rows = [], [], []
for architecture in ARCHITECTURES:
    group = data.loc[data['dps_type'].eq(architecture)].copy()
    if len(group) != EXPECTED_GROUP_SIZES[architecture]:
        raise ValueError(f'{architecture}: unexpected sample size; use the corrected source.')
    eligible_features = []
    for variable, label in INDICATOR_METADATA:
        observed = group[variable].dropna()
        eligible = len(observed) >= MIN_N and observed.nunique() >= 2
        eligibility_rows.append({
            'DPS type': architecture, 'Variable': variable, 'Indicator': label,
            'Observed n': len(observed), 'Missing n': len(group) - len(observed),
            'Zero n': int(observed.eq(0).sum()), 'Unique values': observed.nunique(),
            'Eligible': eligible,
            'Reason': 'Eligible' if eligible else (
                'Fewer than 10 observations' if len(observed) < MIN_N else 'Zero variance')
        })
        if eligible:
            eligible_features.append((variable, label))
    raw = group[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
    for scenario, capped, equal_weights in SPECIFICATIONS:
        score, parameters = composite_score(raw, capped, equal_weights)
        scenario_group = group.assign(ICPS=score)
        parameter_rows.append({'DPS type': architecture, 'Scenario': scenario,
                               'N': len(group), **parameters})
        tests = []
        for variable, label in eligible_features:
            for outcome in OUTCOMES:
                pair = scenario_group[[variable, outcome]].dropna()
                if len(pair) < MIN_N or pair[outcome].nunique() < 2:
                    raise ValueError(f'Invalid pair: {architecture}, {scenario}, {variable}, {outcome}.')
                rho, p_value = spearmanr(pair[variable], pair[outcome])
                lower, upper = spearman_ci(float(rho), len(pair))
                tests.append({
                    'DPS type': architecture, 'Scenario': scenario, 'Variable': variable,
                    'Indicator': label, 'Outcome': outcome, 'n': len(pair),
                    'rho': float(rho), '95% CI lower': lower, '95% CI upper': upper,
                    'p': float(p_value)
                })
        results = pd.DataFrame(tests)
        if len(results) != EXPECTED_TEST_COUNTS[architecture]:
            raise ValueError(f'{architecture}: unexpected test family size.')
        results['q'] = bh_adjust(results['p'].to_numpy())
        results['Selection'] = ((results['q'] < FDR_THRESHOLD)
                                & (results['rho'].abs() >= MIN_ABS_RHO))
        correlation_frames.append(results)
table_s11_correlations = pd.concat(correlation_frames, ignore_index=True)
table_s11_eligibility = pd.DataFrame(eligibility_rows)
table_s11_parameters = pd.DataFrame(parameter_rows)
baseline = table_s11_correlations.loc[table_s11_correlations['Scenario'].eq(BASELINE)].copy()

# Verify every baseline test key and Selection flag against Table S3 (if supplied).
if reference_c is not None:
    verification = baseline[keys + ['Selection']].merge(
        reference_c[keys + ['Selection']], on=keys, how='outer',
        suffixes=('_calculated', '_reference'), indicator=True, validate='one_to_one')
    if not verification['_merge'].eq('both').all() or not (
        verification['Selection_calculated'] == verification['Selection_reference']).all():
        raise ValueError('The recalculated baseline differs from Table S3. Resolve the input mismatch first.')
for architecture in ARCHITECTURES:
    count = baseline.loc[baseline['DPS type'].eq(architecture) & baseline['Selection'], 'Variable'].nunique()
    if count != EXPECTED_BASELINE_FEATURES[architecture]:
        raise ValueError(f'{architecture}: unexpected baseline feature count.')

# 4. Compare each SAME baseline-selected indicator-outcome pair across scenarios.
# Retention follows the manuscript: same sign and unadjusted p < 0.10.
# No additional effect-size threshold is imposed on this retention criterion.
reference = baseline[keys + ['rho', 'p', 'q', 'Selection']].rename(columns={
    'rho': 'Baseline rho', 'p': 'Baseline p', 'q': 'Baseline q',
    'Selection': 'Baseline selection'})
table_s11_associations = table_s11_correlations.merge(reference, on=keys, how='left', validate='many_to_one')
table_s11_associations['Same sign as baseline'] = (
    np.sign(table_s11_associations['rho']) == np.sign(table_s11_associations['Baseline rho']))
table_s11_associations['Retained at p < 0.10'] = (
    table_s11_associations['Baseline selection'] & table_s11_associations['Same sign as baseline']
    & table_s11_associations['p'].lt(RETENTION_P_THRESHOLD))
table_s11_associations['Retained under FDR selection'] = (
    table_s11_associations['Baseline selection'] & table_s11_associations['Same sign as baseline']
    & table_s11_associations['Selection'])
table_s11_associations['New selected association'] = (
    ~table_s11_associations['Baseline selection'] & table_s11_associations['Selection'])
table_s11_associations['Rho change'] = table_s11_associations['rho'] - table_s11_associations['Baseline rho']

feature_rows, summary_rows, outcome_rows = [], [], []
for architecture in ARCHITECTURES:
    baseline_group = baseline.loc[baseline['DPS type'].eq(architecture)]
    baseline_selected = baseline_group.loc[baseline_group['Selection']]
    baseline_set = set(baseline_selected['Variable'])
    baseline_icps_set = set(baseline_selected.loc[baseline_selected['Outcome'].eq('ICPS'), 'Variable'])
    for scenario, _, _ in SPECIFICATIONS:
        tests = table_s11_associations.loc[table_s11_associations['DPS type'].eq(architecture)
                                    & table_s11_associations['Scenario'].eq(scenario)]
        selected_set = set(tests.loc[tests['Selection'], 'Variable'])
        retained_set = set(tests.loc[tests['Retained at p < 0.10'], 'Variable'])
        strict_set = set(tests.loc[tests['Retained under FDR selection'], 'Variable'])
        retained_icps_set = set(tests.loc[tests['Outcome'].eq('ICPS')
                                           & tests['Retained at p < 0.10'], 'Variable'])
        for variable, pairs in tests.groupby('Variable', sort=False):
            base_pairs = pairs.loc[pairs['Baseline selection']]
            retained_pairs = pairs.loc[pairs['Retained at p < 0.10']]
            alternative_pairs = pairs.loc[pairs['Selection']]
            in_baseline = variable in baseline_set
            retained = variable in retained_set
            feature_rows.append({
                'DPS type': architecture, 'Scenario': scenario, 'Variable': variable,
                'Indicator': pairs['Indicator'].iloc[0], 'Baseline selected': in_baseline,
                'Scenario selected under FDR': variable in selected_set,
                'Retained at p < 0.10': retained if in_baseline else None,
                'Retained under FDR selection': variable in strict_set if in_baseline else None,
                'Feature status': ('Retained' if retained else 'Lost') if in_baseline else (
                    'Newly selected under FDR' if variable in selected_set else 'Not selected'),
                'Baseline selected outcomes': ', '.join(base_pairs['Outcome']),
                'Scenario FDR-selected outcomes': ', '.join(alternative_pairs['Outcome']),
                'Retained baseline outcomes at p < 0.10': ', '.join(retained_pairs['Outcome']),
                'Baseline selected pairs': len(base_pairs),
                'Retained baseline pairs at p < 0.10': len(retained_pairs),
                'Baseline selected pairs with reversed sign': int((
                    base_pairs['rho'] * base_pairs['Baseline rho'] < 0).sum())
            })
        summary_rows.append({
            'DPS type': architecture, 'Scenario': scenario,
            'N': int(data['dps_type'].eq(architecture).sum()),
            'Eligible features': tests['Variable'].nunique(), 'Tests in BH family': len(tests),
            'Baseline selected features': len(baseline_set),
            'Retained features (same sign, p < 0.10)': len(retained_set),
            'Lost features (p retention rule)': len(baseline_set - retained_set),
            'Feature retention (%)': 100 * len(retained_set) / len(baseline_set),
            'Scenario FDR-selected features': len(selected_set),
            'Baseline features also FDR-selected': len(baseline_set & selected_set),
            'Baseline features not FDR-selected': len(baseline_set - selected_set),
            'New FDR-selected features': len(selected_set - baseline_set),
            'Features retaining a baseline pair under FDR': len(strict_set),
            'Baseline selected associations': len(baseline_selected),
            'Baseline associations retained at p < 0.10': int(tests['Retained at p < 0.10'].sum()),
            'Baseline associations retained under FDR': int(tests['Retained under FDR selection'].sum()),
            'Baseline associations with reversed sign': int((tests['Baseline selection'] &
                (tests['rho'] * tests['Baseline rho'] < 0)).sum()),
            'Scenario FDR-selected associations': int(tests['Selection'].sum()),
            'New FDR-selected associations': int(tests['New selected association'].sum()),
            'Baseline ICPS-associated features': len(baseline_icps_set),
            'ICPS features retaining a baseline ICPS link at p < 0.10': len(retained_icps_set)
        })
        for outcome in OUTCOMES:
            subset = tests.loc[tests['Outcome'].eq(outcome)]
            selected_pairs = subset.loc[subset['Baseline selection']]
            outcome_rows.append({
                'DPS type': architecture, 'Scenario': scenario, 'Outcome': outcome,
                'Tests': len(subset), 'Baseline selected associations': len(selected_pairs),
                'Retained baseline associations at p < 0.10': int(subset['Retained at p < 0.10'].sum()),
                'Retained baseline associations under FDR': int(subset['Retained under FDR selection'].sum()),
                'Baseline associations with reversed sign': int((
                    selected_pairs['rho'] * selected_pairs['Baseline rho'] < 0).sum()),
                'Scenario FDR-selected associations': int(subset['Selection'].sum()),
                'New FDR-selected associations': int(subset['New selected association'].sum())
            })
table_s11_features = pd.DataFrame(feature_rows)
table_s11_summary = pd.DataFrame(summary_rows)
table_s11_by_outcome = pd.DataFrame(outcome_rows)
table_s11_retention = table_s11_features.loc[table_s11_features['Baseline selected']].copy()
table_s11_new_features = table_s11_features.loc[table_s11_features['Feature status'].eq('Newly selected under FDR')].copy()

# 5. Main manuscript table: retention and full FDR reselection are separate columns.
table_s11 = table_s11_summary[[
    'DPS type', 'Scenario', 'N', 'Baseline selected features',
    'Retained features (same sign, p < 0.10)', 'Lost features (p retention rule)',
    'Feature retention (%)', 'Scenario FDR-selected features',
    'Baseline features not FDR-selected', 'New FDR-selected features',
    'Baseline ICPS-associated features',
    'ICPS features retaining a baseline ICPS link at p < 0.10'
]].copy()
table_s11['Feature retention (%)'] = table_s11['Feature retention (%)'].map(lambda x: f'{x:.1f}')

# 6. Export the main table and full-precision supporting results.
table_s11.to_csv(OUTPUT_DIRECTORY / 'Table_S11.csv', sep=';', encoding='utf-8-sig', index=False)
for frame, filename in [
    (table_s11_summary, 'Table_S11_summary.csv'),
    (table_s11_associations, 'Table_S11_correlations.csv'),
    (table_s11_retention, 'Table_S11_retention.csv'),
    (table_s11_features, 'Table_S11_feature_selection.csv'),
    (table_s11_new_features, 'Table_S11_new_features.csv'),
    (table_s11_by_outcome, 'Table_S11_by_outcome.csv'),
    (table_s11_parameters, 'Table_S11_parameters.csv'),
    (table_s11_eligibility, 'Table_S11_eligibility.csv')
]:
    frame.to_csv(OUTPUT_DIRECTORY / filename, sep=';', decimal=',', encoding='utf-8-sig', index=False)

print('Completed Table S11. ' + ('Baseline matches the Table S3 Selection flags.' if reference_c is not None
      else 'Table_S3.csv not found: baseline selection recomputed from the data only.'))
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
print('Retention: same baseline-selected pair, same sign, unadjusted p < 0.10.')
print('FDR reselection: complete family q < 0.10 and absolute rho >= 0.10.')
try:
    from IPython.display import display
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
        display(table_s11)
        display(table_s11_by_outcome)
except ImportError:
    print(table_s11.to_string(index=False))
    print(table_s11_by_outcome.to_string(index=False))

Completed Table S11. Table_S3.csv not found: baseline selection recomputed from the data only.
Saved to: ../results/Table_S11.csv
Retention: same baseline-selected pair, same sign, unadjusted p < 0.10.
FDR reselection: complete family q < 0.10 and absolute rho >= 0.10.


,DPS type,Scenario,N,Baseline selected features,"Retained features (same sign, p < 0.10)",Lost features (p retention rule),Feature retention (%),Scenario FDR-selected features,Baseline features not FDR-selected,New FDR-selected features,Baseline ICPS-associated features,ICPS features retaining a baseline ICPS link at p < 0.10
0,LV–MV,Baseline: capped entropy ICPS,669,30,30,0,100.0,30,0,0,18,18
1,LV–MV,Equal weights: capped ICPS,669,30,30,0,100.0,30,0,0,18,18
2,LV–MV,Uncapped entropy ICPS,669,30,30,0,100.0,30,0,0,18,17
3,LV–MV–HV,Baseline: capped entropy ICPS,97,16,16,0,100.0,16,0,0,6,6
4,LV–MV–HV,Equal weights: capped ICPS,97,16,16,0,100.0,16,0,0,6,6
5,LV–MV–HV,Uncapped entropy ICPS,97,16,16,0,100.0,17,0,1,6,6


,DPS type,Scenario,Outcome,Tests,Baseline selected associations,Retained baseline associations at p < 0.10,Retained baseline associations under FDR,Baseline associations with reversed sign,Scenario FDR-selected associations,New FDR-selected associations
0,LV–MV,Baseline: capped entropy ICPS,ICPS,64,18,18,18,0,18,0
1,LV–MV,Baseline: capped entropy ICPS,SAIDI,64,16,16,16,0,16,0
2,LV–MV,Baseline: capped entropy ICPS,ASIDI,64,25,25,25,0,25,0
3,LV–MV,Equal weights: capped ICPS,ICPS,64,18,18,17,0,18,1
4,LV–MV,Equal weights: capped ICPS,SAIDI,64,16,16,16,0,16,0
5,LV–MV,Equal weights: capped ICPS,ASIDI,64,25,25,25,0,25,0
6,LV–MV,Uncapped entropy ICPS,ICPS,64,18,17,11,0,14,3
7,LV–MV,Uncapped entropy ICPS,SAIDI,64,16,16,16,0,16,0
8,LV–MV,Uncapped entropy ICPS,ASIDI,64,25,25,25,0,25,0
9,LV–MV–HV,Baseline: capped entropy ICPS,ICPS,73,6,6,6,0,6,0


In [12]:
#Supplementary Materials, Table S12: sensitivity to zero ASIDI observations in LV-MV operators.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata, norm
from scipy.special import stdtr

DIRECTORY = Path('../data')
DATA_PATH = DIRECTORY / 'DSO_2024_corrected.csv'
OUTPUT_DIRECTORY = Path('../results')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_ID = 'Table_S12'
TABLE_S3_PATH = DIRECTORY / 'Table_S3.csv'
ARCHITECTURE = 'LV–MV'
BASELINE = 'Baseline: all complete LV-MV operators'
POSITIVE = 'ASIDI > 0: ICPS refitted'
OUTCOMES = ['ICPS', 'SAIDI', 'ASIDI']
MIN_N = 10
FDR_THRESHOLD = 0.10
MIN_ABS_RHO = 0.10
RETENTION_P_THRESHOLD = 0.10
RANDOM_SUBSAMPLES = 999
RANDOM_SEED = 20261116

INDICATOR_METADATA = [('lv_network_density_km_per_km2', 'LV network density'),
 ('mv_network_density_km_per_km2', 'MV network density'),
 ('lv_to_mv_network_length_ratio', 'Ratio of LV to MV network length'),
 ('network_length_per_consumption_point_m', 'Network length per consumption point'),
 ('network_length_per_generation_point_m', 'Network length per generation point'),
 ('generation_points_per_1000_consumption_points', 'Generation points per 1,000 consumption points'),
 ('vertical_exchange_intensity_kw_per_kw', 'Vertical exchange intensity'),
 ('lv_gis_coverage_pct', 'GIS coverage of LV network'),
 ('mv_gis_coverage_pct', 'GIS coverage of MV network'),
 ('hv_gis_coverage_pct', 'GIS coverage of HV network'),
 ('non_remote_maximum_demand_meters_share_pct', 'Share of non-remote maximum-demand meters'),
 ('local_substations_remote_transformer_measurements_share_pct',
  'Share of local substations with remote measurements at the transformer'),
 ('local_substations_remote_lv_board_measurements_share_pct',
  'Share of local substations with remote measurements in the LV distribution board'),
 ('lv_lines_centralized_online_monitoring_share_pct', 'Share of LV lines with centralized online monitoring'),
 ('mv_lines_centralized_online_monitoring_share_pct', 'Share of MV lines with centralized online monitoring'),
 ('hv_lines_centralized_online_monitoring_share_pct', 'Share of HV lines with centralized online monitoring'),
 ('mv_lv_transformers_centralized_online_monitoring_share_pct',
  'Share of MV/LV transformers with centralized online monitoring'),
 ('hv_mv_transformers_centralized_online_monitoring_share_pct',
  'Share of HV/MV transformers with centralized online monitoring'),
 ('lv_equipment_centralized_control_share_pct', 'Share of LV network equipment with centralized control'),
 ('mv_equipment_centralized_control_share_pct', 'Share of MV network equipment with centralized control'),
 ('hv_equipment_centralized_control_share_pct', 'Share of HV network equipment with centralized control'),
 ('mv_lv_substations_centralized_control_share_pct', 'Share of MV/LV substations with centralized control'),
 ('hv_mv_substations_centralized_control_share_pct', 'Share of HV/MV substations with centralized control'),
 ('lv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak LV load'),
 ('mv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak MV load'),
 ('hv_dg_capacity_to_peak_load_kw_per_kw', 'DG capacity relative to peak HV load'),
 ('lv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at LV level'),
 ('mv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at MV level'),
 ('hv_dg_capacity_utilization_factor_pct', 'DG capacity utilization factor (CUF) at HV level'),
 ('lv_solar_generation_share_pct', 'Share of solar generation at LV level'),
 ('lv_wind_generation_share_pct', 'Share of wind generation at LV level'),
 ('lv_other_res_generation_share_pct', 'Share of other RES at LV level'),
 ('mv_solar_generation_share_pct', 'Share of solar generation at MV level'),
 ('mv_wind_generation_share_pct', 'Share of wind generation at MV level'),
 ('mv_other_res_generation_share_pct', 'Share of other RES at MV level'),
 ('hv_solar_generation_share_pct', 'Share of solar generation at HV level'),
 ('hv_wind_generation_share_pct', 'Share of wind generation at HV level'),
 ('hv_other_res_generation_share_pct', 'Share of other RES at HV level'),
 ('lv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at LV level'),
 ('lv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at LV level'),
 ('lv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at LV level'),
 ('mv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at MV level'),
 ('mv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at MV level'),
 ('mv_other_res_connection_application_avg_capacity_kw',
  'Average capacity of other RES connection applications at MV level'),
 ('hv_solar_connection_application_avg_capacity_kw', 'Average capacity of solar connection applications at HV level'),
 ('hv_wind_connection_application_avg_capacity_kw', 'Average capacity of wind connection applications at HV level'),
 ('lv_generation_connection_duration_days', 'Duration of generation connection to LV network'),
 ('mv_generation_connection_duration_days', 'Duration of generation connection to MV network'),
 ('hv_generation_connection_duration_days', 'Duration of generation connection to HV network'),
 ('lv_peak_load_per_consumer_kw', 'Peak LV load per LV consumer'),
 ('mv_peak_load_per_consumer_kw', 'Peak MV load per MV consumer'),
 ('hv_peak_load_per_consumer_kw', 'Peak HV load per HV consumer'),
 ('lv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak LV load'),
 ('mv_ev_charging_capacity_to_peak_load_pct', 'Share of EV charging stations in peak MV load'),
 ('lv_heat_pump_capacity_to_peak_load_pct', 'Share of heat pumps in peak LV load'),
 ('lv_connection_applications_per_1000_consumers', 'LV connection applications per 1,000 LV consumers'),
 ('lv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded LV connection points per 1,000 LV points'),
 ('lv_consumer_connection_duration_days', 'Duration of consumer connection to LV network'),
 ('lv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at LV level'),
 ('mv_connection_applications_per_1000_consumers', 'MV connection applications per 1,000 MV consumers'),
 ('mv_new_or_upgraded_points_per_1000_existing_points', 'New/upgraded MV connection points per 1,000 MV points'),
 ('mv_consumer_connection_duration_days', 'Duration of consumer connection to MV network'),
 ('mv_implemented_consumer_connection_applications_share_pct',
  'Share of implemented consumer connection applications at MV level'),
 ('lv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak LV load'),
 ('mv_bess_capacity_to_peak_load_kw_per_mw', 'BESS capacity relative to peak MV load'),
 ('lv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 LV consumers'),
 ('mv_bess_connection_applications_per_1000_consumers', 'BESS connection applications per 1,000 MV consumers'),
 ('lv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at LV level'),
 ('mv_bess_application_processing_time_days',
  'Processing time of connection requests for consumption devices and storage at MV level'),
 ('prosumers_full_market_mode_per_1000_lv_consumers', 'Prosumers in full market mode per 1,000 LV consumers'),
 ('prosumer_market_mode_switch_duration_days', 'Time to switch prosumers to full market mode'),
 ('prosumer_switching_automation_pct', 'Level of automation of prosumer switching'),
 ('dynamic_tariff_consumers_per_1000_lv_consumers', 'Consumers with dynamic tariffs per 1,000 LV consumers')]

def bh_adjust(p_values):
    
    p = np.asarray(p_values, dtype=float)
    if not len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('The BH test family must contain valid p-values.')
    order = np.argsort(p, kind='stable')
    adjusted_sorted = p[order] * len(p) / np.arange(1, len(p) + 1)
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted = np.empty_like(p)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted

def spearman_ci(rho, n):

    if n <= 3 or not np.isfinite(rho) or abs(rho) > 1:
        raise ValueError('The Spearman CI requires n > 3 and a valid rho.')
    if np.isclose(abs(rho), 1.0, rtol=0, atol=1e-14):
        return float(rho), float(rho)
    z = np.arctanh(rho)
    standard_error = np.sqrt((1 + rho**2 / 2) / (n - 3))
    bounds = np.tanh(z + np.array([-1, 1]) * norm.ppf(0.975) * standard_error)
    return float(bounds[0]), float(bounds[1])

def composite_score(raw, capped, equal_weights):
    """Fit normalization and weights within one architecture and scenario."""
    if capped:
        lower, upper = np.quantile(raw, [0.10, 0.90], axis=0, method='linear')
        prepared = np.clip(raw, lower, upper)
        normalization = 'P10/P90-capped inverse Min–Max'
    else:
        lower, upper = raw.min(axis=0), raw.max(axis=0)
        prepared = raw
        normalization = 'Uncapped inverse Min–Max using observed minimum and maximum'
    if np.any(upper <= lower):
        raise ValueError('Normalization bounds must have positive ranges.')
    z = (upper - prepared) / (upper - lower)
    if not np.isfinite(z).all() or (z < -1e-12).any() or (z > 1 + 1e-12).any():
        raise ValueError('Invalid normalized continuity scores.')
    if equal_weights:
        weights = np.array([0.5, 0.5])
        weighting = 'Equal weights'
    else:
        shares = z / z.sum(axis=0)
        terms = np.zeros_like(shares)
        positive = shares > 0
        terms[positive] = shares[positive] * np.log(shares[positive])
        differentiation = np.maximum(1 + terms.sum(axis=0) / np.log(len(raw)), 0)
        if differentiation.sum() <= np.finfo(float).eps:
            raise ValueError('Entropy weights cannot be determined.')
        weights = differentiation / differentiation.sum()
        weighting = 'Entropy weights fitted within the scenario'
    return z @ weights, {
        'Normalization': normalization, 'Weighting': weighting,
        'SAIDI weight': float(weights[0]), 'ASIDI weight': float(weights[1]),
        'SAIDI lower bound': float(lower[0]), 'SAIDI upper bound': float(upper[0]),
        'ASIDI lower bound': float(lower[1]), 'ASIDI upper bound': float(upper[1])
    }

def correlations(x, y):

    if x.ndim != 2 or y.ndim != 2 or len(x) != len(y) or not np.isfinite(y).all():
        raise ValueError('Expected two matrices with matching rows and finite outcomes.')
    mask = np.isfinite(x)
    n = mask.sum(axis=0)
    rank_x = np.nan_to_num(rankdata(x, axis=0, nan_policy='omit'), nan=0)
    mean_x = np.divide(rank_x.sum(axis=0), n, out=np.zeros(x.shape[1]), where=n > 0)
    centered_x = np.where(mask, rank_x - mean_x, 0)
    sum_squares_x = (centered_x ** 2).sum(axis=0)
    rho_columns, p_columns, eligible_columns = [], [], []
    for column in range(y.shape[1]):
        masked_y = np.where(mask, y[:, column, None], np.nan)
        rank_y = np.nan_to_num(rankdata(masked_y, axis=0, nan_policy='omit'), nan=0)
        mean_y = np.divide(rank_y.sum(axis=0), n, out=np.zeros(x.shape[1]), where=n > 0)
        centered_y = np.where(mask, rank_y - mean_y, 0)
        sum_squares_y = (centered_y ** 2).sum(axis=0)
        eligible = (n >= MIN_N) & (sum_squares_x > 0) & (sum_squares_y > 0)
        rho = np.divide(
            (centered_x * centered_y).sum(axis=0),
            np.sqrt(sum_squares_x * sum_squares_y),
            out=np.full(x.shape[1], np.nan), where=eligible)
        rho = np.clip(rho, -1, 1)
        with np.errstate(divide='ignore', invalid='ignore'):
            statistic = rho * np.sqrt((n - 2) / ((1 + rho) * (1 - rho)))
        p_value = 2 * stdtr(n - 2, -np.abs(statistic))
        rho_columns.append(rho)
        p_columns.append(p_value)
        eligible_columns.append(eligible)
    return (np.stack(rho_columns, axis=1), np.stack(p_columns, axis=1),
            np.stack(eligible_columns, axis=1), n)

VARIABLES = [variable for variable, _ in INDICATOR_METADATA]
LABELS = dict(INDICATOR_METADATA)
VARIABLE_INDEX = {variable: i for i, variable in enumerate(VARIABLES)}


def test_family(group, target_names, target_values, scenario):
    
    x = group[VARIABLES].to_numpy(dtype=float)
    rho, p_value, valid, n = correlations(x, target_values)
    q = np.full_like(p_value, np.nan)
    if not valid.any():
        raise ValueError(f'{scenario}: no estimable tests in the family.')
    q[valid] = bh_adjust(p_value[valid])
    rows, audit = [], []
    for j, variable in enumerate(VARIABLES):
        for k, outcome in enumerate(target_names):
            estimable = bool(valid[j, k])
            audit.append({
                'DPS type': ARCHITECTURE, 'Scenario': scenario, 'Variable': variable,
                'Indicator': LABELS[variable], 'Outcome': outcome, 'n': int(n[j]),
                'Eligible': estimable,
                'Reason': 'Eligible' if estimable else (
                    f'Fewer than {MIN_N} observations' if n[j] < MIN_N else
                    'Constant predictor or constant outcome in the pair')
            })
            if estimable:
                r = float(rho[j, k])
                lower, upper = spearman_ci(r, int(n[j]))
                rows.append({
                    'DPS type': ARCHITECTURE, 'Scenario': scenario, 'Variable': variable,
                    'Indicator': LABELS[variable], 'Outcome': outcome, 'n': int(n[j]),
                    'rho': r, '95% CI lower': lower, '95% CI upper': upper,
                    'p': float(p_value[j, k]), 'q': float(q[j, k]),
                    'Selection': bool(q[j, k] < FDR_THRESHOLD and abs(r) >= MIN_ABS_RHO)
                })
    return pd.DataFrame(rows), pd.DataFrame(audit)


def retention_flags(reference_rho, alternative_rho, alternative_p):

    return ((reference_rho * alternative_rho > 0)
            & (alternative_p < RETENTION_P_THRESHOLD))


# 1. Load the corrected source. Never replace missing values with zero.
source = pd.read_csv(DATA_PATH, sep=';', decimal=',', encoding='utf-8-sig',
                     dtype={'dso_id': 'string'}, float_precision='round_trip')
source_outcomes = {'saidi_lv_min_per_year': 'SAIDI', 'asidi_mv_min_per_year': 'ASIDI'}
required = {'dso_id', 'dso_name', 'dps_type', *source_outcomes, *VARIABLES}
if not required.issubset(source.columns):
    raise ValueError(f'Missing source columns: {sorted(required - set(source.columns))}')
source['dso_id'] = source['dso_id'].str.strip()
source['dps_type'] = (source['dps_type'].astype('string').str.strip()
                      .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
if source['dso_id'].isna().any() or source['dso_id'].eq('').any() or source['dso_id'].duplicated().any():
    raise ValueError('DSO IDs must be complete and unique.')
for column in [*VARIABLES, *source_outcomes]:
    source[column] = pd.to_numeric(source[column], errors='raise')
    if not np.isfinite(source[column].dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{column}: infinite values are not supported.')
source = source.rename(columns=source_outcomes)
if (source[['SAIDI', 'ASIDI']] < 0).any().any():
    raise ValueError('Interruption durations must be non-negative.')
lv = source.loc[source['dps_type'].eq(ARCHITECTURE)].copy()
complete = lv[['SAIDI', 'ASIDI']].notna().all(axis=1)
table_s12_missing_outcomes = lv.loc[~complete, ['dso_id', 'dso_name', 'SAIDI', 'ASIDI']].copy()
pool = lv.loc[complete].reset_index(drop=True)
positive = pool.loc[pool['ASIDI'].gt(0)].copy()
zero_count = int(pool['ASIDI'].eq(0).sum())
if len(pool) != 669 or len(positive) != 410 or zero_count != 259:
    raise ValueError('Expected 669 complete LV-MV operators, including 259 zero-ASIDI and 410 positive-ASIDI operators.')
if not isinstance(RANDOM_SUBSAMPLES, int) or RANDOM_SUBSAMPLES < 1:
    raise ValueError('RANDOM_SUBSAMPLES must be a positive integer.')

# 2. Refit P10/P90, normalization and entropy weights within each sample.
family_frames, eligibility_frames, parameter_rows = [], [], []
for scenario, group in [(BASELINE, pool), (POSITIVE, positive)]:
    score, parameters = composite_score(group[['SAIDI', 'ASIDI']].to_numpy(dtype=float),
                                         capped=True, equal_weights=False)
    group['ICPS'] = score
    tests, audit = test_family(group, OUTCOMES, group[OUTCOMES].to_numpy(dtype=float), scenario)
    if len(tests) != 192:
        raise ValueError(f'{scenario}: the corrected data should yield 192 eligible tests.')
    family_frames.append(tests)
    eligibility_frames.append(audit)
    parameter_rows.append({'DPS type': ARCHITECTURE, 'Scenario': scenario, 'N': len(group),
                           'Zero ASIDI n': int(group['ASIDI'].eq(0).sum()), **parameters})
table_s12_correlations = pd.concat(family_frames, ignore_index=True)
table_s12_parameters = pd.DataFrame(parameter_rows)
baseline = table_s12_correlations.loc[table_s12_correlations['Scenario'].eq(BASELINE)].copy()
restricted = table_s12_correlations.loc[table_s12_correlations['Scenario'].eq(POSITIVE)].copy()

# 3. Optional: verify every LV-MV baseline key and Selection flag against Table S3.
def normalize_label(text):
    text = str(text).replace('−', '-').replace('*', '').replace('†', '').replace('\u00a0', ' ')
    text = ' '.join(text.split()).casefold()
    return text.replace('dg cuf at', 'dg capacity utilization factor (cuf) at')

keys = ['Variable', 'Outcome']
table_s3_checked = TABLE_S3_PATH.is_file()
if table_s3_checked:
    reference_c = pd.read_csv(TABLE_S3_PATH, sep=';', encoding='utf-8-sig', dtype='string')
    required_c = {'DPS type', 'Indicator', 'Outcome', 'Selection'}
    if not required_c.issubset(reference_c.columns):
        raise ValueError('Table S3 must contain DPS type, Indicator, Outcome and Selection.')
    reference_c['DPS type'] = (reference_c['DPS type'].str.strip()
                               .str.replace(r'\s*[-−—–]\s*', '–', regex=True))
    reference_c = reference_c.loc[reference_c['DPS type'].eq(ARCHITECTURE)].copy()
    flags = reference_c['Selection'].str.strip().str.casefold()
    if flags.isna().any() or not flags.isin(['true', 'false', '1', '0']).all():
        raise ValueError('Table S3 Selection must contain TRUE/FALSE values.')
    reference_c['Selection'] = flags.map({'true': True, 'false': False, '1': True, '0': False}).astype(bool)
    lookup = {normalize_label(label): v for v, label in INDICATOR_METADATA}
    lookup.update({normalize_label(v): v for v, _ in INDICATOR_METADATA})
    def to_variable(label):
        key = normalize_label(label)
        return lookup.get(key, lookup.get(key.rsplit(', ', 1)[0]))
    if 'Variable' not in reference_c.columns:
        reference_c['Variable'] = reference_c['Indicator'].map(to_variable)
    else:
        reference_c['Variable'] = reference_c['Variable'].str.strip()
    if reference_c['Variable'].isna().any() or reference_c.duplicated(keys).any():
        raise ValueError('Table S3 contains unknown indicators or duplicate pairs.')
    verification = baseline[keys + ['Selection']].merge(
        reference_c[keys + ['Selection']], on=keys, how='outer',
        suffixes=('_calculated', '_reference'), indicator=True, validate='one_to_one')
    if not verification['_merge'].eq('both').all() or not (
        verification['Selection_calculated'] == verification['Selection_reference']).all():
        raise ValueError('The recalculated LV-MV baseline differs from Table S3.')
base_selected = baseline.loc[baseline['Selection']].copy().reset_index(drop=True)
base_features = [v for v in VARIABLES if v in set(base_selected['Variable'])]
if len(base_features) != 30:
    raise ValueError('Expected 30 baseline-selected LV-MV features.')
saidi_features = set(base_selected.loc[base_selected['Outcome'].eq('SAIDI'), 'Variable'])
other_features = set(base_features) - saidi_features

# 4. Track the SAME baseline-selected pairs and then aggregate by feature.
ref_columns = ['rho', 'p', 'q', 'n']
table_s12_associations = base_selected[['Variable', 'Indicator', 'Outcome', *ref_columns]].rename(
    columns={column: f'Baseline {column}' for column in ref_columns}).merge(
        restricted[['Variable', 'Outcome', *ref_columns, 'Selection']].rename(
            columns={column: f'Positive ASIDI {column}' for column in ref_columns}),
        on=keys, how='left', validate='one_to_one')
table_s12_associations.insert(0, 'DPS type', ARCHITECTURE)
table_s12_associations['Retained at p < 0.10'] = retention_flags(
    table_s12_associations['Baseline rho'], table_s12_associations['Positive ASIDI rho'],
    table_s12_associations['Positive ASIDI p'])
table_s12_associations['Sign reversed'] = (
    table_s12_associations['Baseline rho'] * table_s12_associations['Positive ASIDI rho'] < 0)
table_s12_associations['Retained under FDR selection'] = (
    table_s12_associations['Retained at p < 0.10'] & table_s12_associations['Selection'].eq(True))
table_s12_associations = table_s12_associations.rename(columns={'Selection': 'Positive ASIDI FDR selection'})
retained_features = set(table_s12_associations.loc[table_s12_associations['Retained at p < 0.10'], 'Variable'])
strict_features = set(table_s12_associations.loc[table_s12_associations['Retained under FDR selection'], 'Variable'])
positive_selected_features = set(restricted.loc[restricted['Selection'], 'Variable'])
feature_rows = []
for variable in base_features:
    pairs = table_s12_associations.loc[table_s12_associations['Variable'].eq(variable)]
    kept = pairs.loc[pairs['Retained at p < 0.10']]
    feature_rows.append({
        'DPS type': ARCHITECTURE, 'Variable': variable, 'Indicator': LABELS[variable],
        'Baseline stratum': 'Selected through SAIDI' if variable in saidi_features else 'Not selected through SAIDI',
        'Baseline selected outcomes': ', '.join(pairs['Outcome']),
        'Retained baseline outcomes at p < 0.10': ', '.join(kept['Outcome']),
        'Retained at p < 0.10': variable in retained_features,
        'Retained under FDR selection': variable in strict_features,
        'Selected under positive-ASIDI FDR': variable in positive_selected_features,
        'Baseline selected associations': len(pairs), 'Retained associations at p < 0.10': len(kept),
        'Associations with reversed sign': int(pairs['Sign reversed'].sum())
    })
table_s12_retention = pd.DataFrame(feature_rows)
new_variables = positive_selected_features - set(base_features)
table_s12_new_features = restricted.loc[restricted['Selection'] & restricted['Variable'].isin(new_variables)].copy()
summary_rows, outcome_rows = [], []
for scenario, group, tests in [(BASELINE, pool, baseline), (POSITIVE, positive, restricted)]:
    is_baseline = scenario == BASELINE
    kept_features = set(base_features) if is_baseline else retained_features
    selected_features = set(tests.loc[tests['Selection'], 'Variable'])
    summary_rows.append({
        'DPS type': ARCHITECTURE, 'Scenario': scenario, 'N': len(group),
        'Zero ASIDI n': int(group['ASIDI'].eq(0).sum()),
        'Excluded zero-ASIDI operators': 0 if is_baseline else zero_count,
        'Baseline zero ASIDI share (%)': 100 * zero_count / len(pool),
        'Tests in BH family': len(tests), 'Baseline selected features': len(base_features),
        'Retained features (same sign, p < 0.10)': len(kept_features),
        'Lost features (p retention rule)': len(base_features) - len(kept_features),
        'Feature retention (%)': 100 * len(kept_features) / len(base_features),
        'Retained SAIDI-selected features': len(kept_features & saidi_features),
        'Baseline SAIDI-selected features': len(saidi_features),
        'Retained other features': len(kept_features & other_features),
        'Baseline other features': len(other_features),
        'Scenario FDR-selected features': len(selected_features),
        'Baseline features also FDR-selected': len(selected_features & set(base_features)),
        'Baseline features not FDR-selected': len(set(base_features) - selected_features),
        'New FDR-selected features': len(selected_features - set(base_features))
    })
    for outcome in OUTCOMES:
        count = int(base_selected['Outcome'].eq(outcome).sum())
        pairs = table_s12_associations.loc[table_s12_associations['Outcome'].eq(outcome)]
        outcome_rows.append({
            'DPS type': ARCHITECTURE, 'Scenario': scenario, 'Outcome': outcome,
            'Baseline selected associations': count,
            'Retained baseline associations at p < 0.10': count if is_baseline else int(pairs['Retained at p < 0.10'].sum()),
            'Retained baseline associations under FDR': count if is_baseline else int(pairs['Retained under FDR selection'].sum()),
            'Scenario FDR-selected associations': int((tests['Outcome'].eq(outcome) & tests['Selection']).sum())
        })
table_s12_summary = pd.DataFrame(summary_rows)
table_s12_by_outcome = pd.DataFrame(outcome_rows)
table_s12 = table_s12_summary.copy()
for column in ['Baseline zero ASIDI share (%)', 'Feature retention (%)']:
    table_s12[column] = table_s12[column].map(lambda x: f'{x:.2f}')

# 5. Separate positive-ASIDI occurrence from its magnitude conditional on ASIDI > 0.

occurrence_values = pool['ASIDI'].gt(0).to_numpy(dtype=float)[:, None]
table_s12_occurrence_tests, occurrence_audit = test_family(
    pool, ['Positive ASIDI indicator'], occurrence_values, 'Occurrence: ASIDI > 0 in full sample')
eligibility_frames.append(occurrence_audit)
table_s12_eligibility = pd.concat(eligibility_frames, ignore_index=True)
asidi_reference = base_selected.loc[base_selected['Outcome'].eq('ASIDI')]
table_s12_asidi_decomposition = asidi_reference[['Variable', 'Indicator', *ref_columns]].rename(
    columns={column: f'Baseline ASIDI {column}' for column in ref_columns})
table_s12_asidi_decomposition = table_s12_asidi_decomposition.merge(
    table_s12_occurrence_tests[['Variable', *ref_columns, 'Selection']].rename(
        columns={**{column: f'Occurrence {column}' for column in ref_columns},
                 'Selection': 'Occurrence FDR selection'}), on='Variable', how='left', validate='one_to_one')
table_s12_asidi_decomposition = table_s12_asidi_decomposition.merge(
    restricted.loc[restricted['Outcome'].eq('ASIDI'), ['Variable', *ref_columns, 'Selection']].rename(
        columns={**{column: f'Positive ASIDI {column}' for column in ref_columns},
                 'Selection': 'Positive ASIDI FDR selection'}), on='Variable', how='left', validate='one_to_one')
table_s12_asidi_decomposition.insert(0, 'DPS type', ARCHITECTURE)
for component, prefix in [('Occurrence', 'Occurrence'), ('Positive ASIDI', 'Positive ASIDI')]:
    table_s12_asidi_decomposition[f'{component} retained at p < 0.10'] = retention_flags(
        table_s12_asidi_decomposition['Baseline ASIDI rho'], table_s12_asidi_decomposition[f'{prefix} rho'],
        table_s12_asidi_decomposition[f'{prefix} p'])
table_s12_asidi_summary = pd.DataFrame([{
    'DPS type': ARCHITECTURE,
    'Analysis': 'Occurrence: indicator ASIDI > 0' if prefix == 'Occurrence' else 'Magnitude: annual ASIDI conditional on ASIDI > 0',
    'Sample N': len(pool) if prefix == 'Occurrence' else len(positive),
    'Baseline selected ASIDI associations': len(table_s12_asidi_decomposition),
    'Estimable baseline associations': int(table_s12_asidi_decomposition[f'{prefix} p'].notna().sum()),
    'Same-sign associations at p < 0.10': int(table_s12_asidi_decomposition[f'{prefix} retained at p < 0.10'].sum()),
    'FDR family tests': len(table_s12_occurrence_tests) if prefix == 'Occurrence' else len(restricted),
    'Same-sign associations also FDR-selected': int((
        table_s12_asidi_decomposition[f'{prefix} retained at p < 0.10']
        & table_s12_asidi_decomposition[f'{prefix} FDR selection'].eq(True)).sum())
} for prefix in ['Occurrence', 'Positive ASIDI']])

# 6. Size-matched random subsets from the full pool, without replacement per draw.

x_pool = pool[VARIABLES].to_numpy(dtype=float)
raw_pool = pool[['SAIDI', 'ASIDI']].to_numpy(dtype=float)
base_vi = np.array([VARIABLE_INDEX[v] for v in base_selected['Variable']])
base_oi = np.array([OUTCOMES.index(o) for o in base_selected['Outcome']])
base_rho = base_selected['rho'].to_numpy(dtype=float)
feature_index = {v: j for j, v in enumerate(base_features)}
pair_to_feature = np.array([feature_index[v] for v in base_selected['Variable']])
feature_vi = np.array([VARIABLE_INDEX[v] for v in base_features])
saidi_mask = np.array([v in saidi_features for v in base_features])
random_pair_retained = np.zeros((RANDOM_SUBSAMPLES, len(base_selected)), dtype=bool)
random_feature_retained = np.zeros((RANDOM_SUBSAMPLES, len(base_features)), dtype=bool)
random_feature_selected = np.zeros_like(random_feature_retained)
rng = np.random.default_rng(RANDOM_SEED)
draw_rows = []
for draw in range(RANDOM_SUBSAMPLES):
    sample_seed = int(rng.integers(0, np.iinfo(np.uint32).max, dtype=np.uint32))
    positions = np.sort(np.random.default_rng(sample_seed).choice(len(pool), len(positive), replace=False))
    raw_sample = raw_pool[positions]
    score, parameters = composite_score(raw_sample, capped=True, equal_weights=False)
    outcomes = np.column_stack([score, raw_sample])
    rho, p_value, valid_pairs, observed_n = correlations(x_pool[positions], outcomes)
    q = np.full_like(p_value, np.nan)
    if not valid_pairs.any():
        raise ValueError(f'Random draw {draw + 1}: no estimable test family.')
    q[valid_pairs] = bh_adjust(p_value[valid_pairs])
    selected_pairs = valid_pairs & (q < FDR_THRESHOLD) & (np.abs(rho) >= MIN_ABS_RHO)
    flags = retention_flags(base_rho, rho[base_vi, base_oi], p_value[base_vi, base_oi])
    feature_flags = np.zeros(len(base_features), dtype=bool)
    np.logical_or.at(feature_flags, pair_to_feature, flags)
    random_pair_retained[draw] = flags
    random_feature_retained[draw] = feature_flags
    random_feature_selected[draw] = selected_pairs[feature_vi].any(axis=1)
    row = {
        'Draw': draw + 1, 'Sample seed': sample_seed, 'N': len(positive),
        'Zero ASIDI n': int((raw_sample[:, 1] == 0).sum()),
        'Tests in BH family': int(valid_pairs.sum()),
        'Retained baseline features at p < 0.10': int(feature_flags.sum()),
        'Retained SAIDI-selected features': int(feature_flags[saidi_mask].sum()),
        'Retained other features': int(feature_flags[~saidi_mask].sum()),
        'Retained baseline associations at p < 0.10': int(flags.sum()),
        'Scenario FDR-selected features': int(selected_pairs.any(axis=1).sum()),
        'Baseline features FDR-selected': int(selected_pairs[feature_vi].any(axis=1).sum()),
        'SAIDI weight': parameters['SAIDI weight'], 'ASIDI weight': parameters['ASIDI weight']
    }
    for outcome in OUTCOMES:
        row[f'Retained {outcome} associations at p < 0.10'] = int(flags[base_selected['Outcome'].eq(outcome)].sum())
    draw_rows.append(row)
table_s12_size_control = pd.DataFrame(draw_rows)
reference_counts = table_s12_size_control['Retained baseline features at p < 0.10'].to_numpy()
observed_count = len(retained_features)
lower_or_equal = int((reference_counts <= observed_count).sum())
reference_quantiles = np.quantile(reference_counts, [0.025, 0.05, 0.50, 0.95, 0.975], method='linear')
table_s12_size_control_summary = pd.DataFrame([{
    'DPS type': ARCHITECTURE, 'Full sample N': len(pool), 'Subset N': len(positive),
    'Random draws': RANDOM_SUBSAMPLES, 'Master seed': RANDOM_SEED,
    'Positive-ASIDI retained baseline features': observed_count,
    'Random retained features mean': float(reference_counts.mean()),
    'Random retained features minimum': int(reference_counts.min()),
    'Random retained features P2.5': float(reference_quantiles[0]),
    'Random retained features P5': float(reference_quantiles[1]),
    'Random retained features median': float(reference_quantiles[2]),
    'Random retained features P95': float(reference_quantiles[3]),
    'Random retained features P97.5': float(reference_quantiles[4]),
    'Random retained features maximum': int(reference_counts.max()),
    'Draws retaining no more features than positive-ASIDI subset': lower_or_equal,
    'Empirical fraction no greater than observed': lower_or_equal / RANDOM_SUBSAMPLES,
    'Smoothed lower-tail reference fraction (+1)': (lower_or_equal + 1) / (RANDOM_SUBSAMPLES + 1)
}])
table_s12_size_control_features = table_s12_retention[['Variable', 'Indicator', 'Baseline stratum', 'Retained at p < 0.10']].copy()
table_s12_size_control_features = table_s12_size_control_features.rename(
    columns={'Retained at p < 0.10': 'Positive-ASIDI subset retained'})
table_s12_size_control_features['Random draws retained'] = random_feature_retained.sum(axis=0)
table_s12_size_control_features['Random retention rate (%)'] = 100 * random_feature_retained.mean(axis=0)
table_s12_size_control_features['Random FDR selection rate (%)'] = 100 * random_feature_selected.mean(axis=0)
table_s12_size_control_associations = table_s12_associations.copy()
table_s12_size_control_associations['Random draws retained'] = random_pair_retained.sum(axis=0)
table_s12_size_control_associations['Random retention rate (%)'] = 100 * random_pair_retained.mean(axis=0)

# Export the ordered sampling pool, so each draw can be reproduced from Sample seed.
table_s12_operators = pool[['dso_id', 'dso_name', 'dps_type', 'SAIDI', 'ASIDI', 'ICPS']].copy()
table_s12_operators.insert(0, 'Pool row (1-based)', np.arange(1, len(pool) + 1))
table_s12_operators = table_s12_operators.rename(columns={'ICPS': 'Baseline ICPS'})
table_s12_operators['Positive-ASIDI subset'] = table_s12_operators['ASIDI'].gt(0)
positive_scores = positive.set_index('dso_id')['ICPS']
table_s12_operators['Positive-subset refitted ICPS'] = table_s12_operators['dso_id'].map(positive_scores)

# 7. Export tables. Numeric files preserve full precision
table_s12.to_csv(OUTPUT_DIRECTORY / 'Table_S12.csv', sep=';', encoding='utf-8-sig', index=False)
for frame, filename in [
    (table_s12_correlations, 'Table_S12_correlations.csv'),
    (table_s12_retention, 'Table_S12_retention.csv'),
    (table_s12_associations, 'Table_S12_associations.csv'),
    (table_s12_new_features, 'Table_S12_new_features.csv'),
    (table_s12_by_outcome, 'Table_S12_by_outcome.csv'),
    (table_s12_asidi_decomposition, 'Table_S12_asidi_decomposition.csv'),
    (table_s12_occurrence_tests, 'Table_S12_occurrence_tests.csv'),
    (table_s12_asidi_summary, 'Table_S12_asidi_summary.csv'),
    (table_s12_size_control, 'Table_S12_size_control.csv'),
    (table_s12_size_control_summary, 'Table_S12_size_control_summary.csv'),
    (table_s12_size_control_features, 'Table_S12_size_control_features.csv'),
    (table_s12_size_control_associations, 'Table_S12_size_control_associations.csv'),
    (table_s12_parameters, 'Table_S12_parameters.csv'),
    (table_s12_eligibility, 'Table_S12_eligibility.csv'),
    (table_s12_operators, 'Table_S12_operators.csv')
]:
    frame.to_csv(OUTPUT_DIRECTORY / filename, sep=';', decimal=',', encoding='utf-8-sig', index=False)


print('Completed Table S12. ' + ('The LV-MV baseline matches Table S3.' if table_s3_checked
      else 'Table_S3.csv not found: baseline selection recomputed from the data only.'))
print(f'ASIDI-zero operators: {zero_count}/{len(pool)} ({100 * zero_count / len(pool):.2f}%).')
print(f'Saved to: {OUTPUT_DIRECTORY / (TABLE_ID + ".csv")}')
print(f'Random size-matched subsets: {RANDOM_SUBSAMPLES}; master seed: {RANDOM_SEED}.')
try:
    from IPython.display import display
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
        display(table_s12)
        display(table_s12_asidi_summary)
        display(table_s12_size_control_summary)
except ImportError:
    print(table_s12.to_string(index=False))
    print(table_s12_asidi_summary.to_string(index=False))
    print(table_s12_size_control_summary.to_string(index=False))

Completed Table S12. Table_S3.csv not found: baseline selection recomputed from the data only.
ASIDI-zero operators: 259/669 (38.71%).
Saved to: ../results/Table_S12.csv
Random size-matched subsets: 999; master seed: 20261116.


,DPS type,Scenario,N,Zero ASIDI n,Excluded zero-ASIDI operators,Baseline zero ASIDI share (%),Tests in BH family,Baseline selected features,"Retained features (same sign, p < 0.10)",Lost features (p retention rule),Feature retention (%),Retained SAIDI-selected features,Baseline SAIDI-selected features,Retained other features,Baseline other features,Scenario FDR-selected features,Baseline features also FDR-selected,Baseline features not FDR-selected,New FDR-selected features
0,LV–MV,Baseline: all complete LV-MV operators,669,259,0,38.71,192,30,30,0,100.00,16,16,14,14,30,30,0,0
1,LV–MV,ASIDI > 0: ICPS refitted,410,0,259,38.71,192,30,16,14,53.33,13,16,3,14,16,12,18,4


,DPS type,Analysis,Sample N,Baseline selected ASIDI associations,Estimable baseline associations,Same-sign associations at p < 0.10,FDR family tests,Same-sign associations also FDR-selected
0,LV–MV,Occurrence: indicator ASIDI > 0,669,25,25,25,64,24
1,LV–MV,Magnitude: annual ASIDI conditional on ASIDI > 0,410,25,25,1,192,0


,DPS type,Full sample N,Subset N,Random draws,Master seed,Positive-ASIDI retained baseline features,Random retained features mean,Random retained features minimum,Random retained features P2.5,Random retained features P5,Random retained features median,Random retained features P95,Random retained features P97.5,Random retained features maximum,Draws retaining no more features than positive-ASIDI subset,Empirical fraction no greater than observed,Smoothed lower-tail reference fraction (+1)
0,LV–MV,669,410,999,20261116,16,27.945946,22,25.0,25.0,28.0,30.0,30.0,30,0,0.0,0.001
